# Code 208 — Modèle de risque d'incendie, forêt de Bouhachem / Jbel Sougna (version autonome)

**Pour l'exécuter** : place ce notebook dans un dossier qui contient le dossier `data/` fourni, ouvre-le dans Jupyter ou VS Code et lance **Exécuter tout**. Une connexion internet est nécessaire (images Sentinel-2 et Landsat sur Microsoft Planetary Computer, météo sur Open-Meteo). Le premier lancement prend environ 1 h 30 à 3 h, car toutes les images sont téléchargées. Les résultats sont écrits dans `resultats/`.

**Méthode**
- Données en Merchich / Nord Maroc (EPSG:26191), une seule grille de 250 m pour toute la forêt (6 071 cellules). Conversion en degrés uniquement pour interroger Open-Meteo et Planetary Computer.
- Labels des zones brûlées par dNBR (seuil 0,27, images juste avant et juste après le feu, masque FIRMS de 2 km, contrôle par la surface officielle).
- Variables : relief, exposition (moyenne circulaire), occupation du sol, population, activité humaine, topoclimat, historique des feux, années depuis le dernier feu, indices satellite d'une image prise entre 1 et 10 jours avant (20 pour Landsat), jamais le jour même.
- Règle du combustible : risque nul si la cellule a brûlé il y a moins d'un an, divisé par 2 entre 1 et 5 ans.
- Évaluation : 12 découpages, chaque incendie évalué par un modèle qui ne l'a jamais vu.
- Danger météo du jour comparé à 976 jours d'été 2017-2024.

La dernière cellule affiche le **bilan final**.


In [ ]:
%pip install -q geopandas rasterio rasterstats shapely pyproj matplotlib pandas numpy requests scikit-learn pystac-client planetary-computer pyarrow joblib xgboost lightgbm

In [ ]:
from pathlib import Path
import os, json, math, hashlib, warnings, random, time
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt, rasterio, joblib, requests
from shapely.geometry import box
from shapely import from_wkb
from rasterstats import zonal_stats
from rasterio.windows import from_bounds as rio_from_bounds
import pystac_client, planetary_computer as pc
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                             recall_score, f1_score, balanced_accuracy_score, confusion_matrix, r2_score)
from IPython.display import display

# ------------------------------------------------------------------
# code208 : VERSION AUTONOME (aucun cache d'un autre notebook).
# Le dossier du projet doit contenir le notebook et un dossier `data/` :
#   data/limites_forets_26191.shp (+ .dbf .shx .prj .cpg)   forêt, 6 parties, Merchich / Nord Maroc
#   data/mnt_bouhachem_26191_30m.tif                          MNT 30 m
#   data/occupation_sol_2021_26191_10m.tif                    occupation du sol 2021, 10 m
#   data/population_2017_26191_100m.tif                       population 2017, 100 m
#   data/feux_FIRMS/*.shp                                      points FIRMS (MODIS et VIIRS)
#   data/osm/osm_code13_2022_distance_*.parquet                routes, pistes, bâtiments, douars, agriculture
# Tous en EPSG:26191. Le reste (Sentinel-2, Landsat, météo) est téléchargé en ligne : connexion internet requise.
# Tous les calculs sont écrits dans `resultats/` (créé automatiquement, à côté du dossier data).
# ------------------------------------------------------------------
def _trouver_racine():
    candidats=[Path(os.environ['PROJECT_ROOT'])] if os.environ.get('PROJECT_ROOT') else []
    candidats+=[Path.cwd(),Path.cwd().parent,Path.cwd().parent.parent]
    for p in candidats:
        if (p/'data'/'limites_forets_26191.shp').exists(): return p
    raise FileNotFoundError("Dossier `data/` introuvable. Place ce notebook dans le dossier du projet, à côté du dossier `data/` "
                            "(ou définis la variable d'environnement PROJECT_ROOT).")
ROOT=_trouver_racine(); print('Dossier du projet :',ROOT)
REPROJ=ROOT/'data'
FOREST=REPROJ/'limites_forets_26191.shp'
DEM=REPROJ/'mnt_bouhachem_26191_30m.tif'
POPULATION=REPROJ/'population_2017_26191_100m.tif'
FIRMS_DIR=REPROJ/'feux_FIRMS'
PREP=ROOT/'resultats'/'calculs'
REPORT=ROOT/'resultats'/'cartes'
for folder in [PREP,REPORT]: folder.mkdir(parents=True,exist_ok=True)
SOURCE_89=PREP/'travail'
WINDOW_CACHE_SOURCE_89=PREP/'journees_sans_feu'     # vide : aucune journée sans feu externe (elles ne servent pas au modèle spatial)
SAT_CACHE_89=PREP/'cache'/'satellite'
PREP_90=PREP
PREP_201=PREP
WINDOW_CACHE_89=PREP/'cache'/'fenetres'

CRS='EPSG:26191'; GRID=250; FIRE_RADIUS=500; SEED=93
TRAIN_START=pd.Timestamp('2017-01-01')   # départ de l'historique causal du PAMF, comme code89/code90

# --- Les événements retenus, corrigés le 22/09/2026 sur la base d'une chronologie précise ---
# (dates de départ/maîtrise officielles, cf. message utilisateur détaillant 5 incendies) :
#   - Bouhachem 2020 : 25/08 -> maîtrisé 01/09/2020 (~1100 ha)              -> RETENU
#   - Bouhachem 2022 : 25/07 -> maîtrisé 02/08/2022 (>7500 ha)              -> RETENU
#   - Sahel El Menzla / Larache 2022 : 13-18/07/2022 (>10500 ha)            -> EXCLU (forêt voisine, PAS Bouhachem)
#   - Derdara / Chefchaouen 2021 : 14-18/08/2021 (>725 ha)                  -> EXCLU pour l'instant (localisation
#     incertaine par rapport au polygone limites_Bouhachem.shp -> voir la cellule de vérification géographique
#     ci-dessous, qui teste automatiquement s'il y a des détections FIRMS locales DANS la forêt sur cette fenêtre)
#   - Jbel Lahbib / Tétouan-Larache 2017 : 30/06-03/07/2017 (>500 ha)       -> EXCLU (forêt voisine, PAS Bouhachem)
FIRE_2020_DATE        = pd.Timestamp('2020-08-25')   # date de départ officielle (corrigée, était 26/08)
FIRE_2020_BEFORE      = pd.Timestamp('2020-08-24')   # code106 : rapprochée du feu (avant : 2020-08-18)   # image Sentinel-2 avant l'incendie 2020 (dNBR)
FIRE_2020_AFTER       = pd.Timestamp('2020-09-02')   # code106 : rapprochée du feu (avant : 2020-09-10)   # image après, largement postérieure à la maîtrise (01/09)
FIRE_2022_BEFORE      = pd.Timestamp('2022-07-24')   # code106 : rapprochée du feu (avant : 2022-07-15)   # image Sentinel-2 avant l'incendie 2022 (marge de sécurité avant le 25/07)
FIRE_2022_AFTER       = pd.Timestamp('2022-08-03')   # code106 : rapprochée du feu (avant : 2022-08-15)   # image après (corrigée : l'ancienne date 27/07 tombait EN PLEIN
                                                      # incendie encore actif, maîtrisé seulement le 02/08 -> sous-estimait la zone brûlée)
FIRE_2022_EVENT_DATE  = pd.Timestamp('2022-07-25')   # date FIRMS réelle confirmée (détections le 25 et le 26/07/2022)

# --- Chronologie précise reçue le 23/09/2026 : Derdara/Sougna 2021 devient un 3e incendie d'entraînement,
# et les 2 dates 2025 (13/08 et 21/08) qui semblaient être DEUX CANDIDATES pour UN SEUL feu incertain sont
# en réalité DEUX INCENDIES DISTINCTS et confirmés (Derdara 12-18/08, puis Bouhachem-près-Derdara 21-24/08)
# -> la cellule qui tranchait entre les deux devient une simple confirmation informative (voir plus bas),
# et le test aveugle est désormais exécuté séparément pour chacun (cellule du test aveugle). ---
DERDARA_2021_DATE     = pd.Timestamp('2021-08-14')   # départ ~14h30, secteur Jbel Sougna / Sougna, entre Derdara et
                                                      # Tanaqoub (confirmé 23/09/2026) - c'est pour couvrir precisement
                                                      # ce secteur que le shapefile a ete etendu aux parties d/e/f (code105)
DERDARA_2021_BEFORE   = pd.Timestamp('2021-08-13')   # code106 : rapprochée du feu (avant : 2021-08-05)   # marge de sécurité avant le départ
DERDARA_2021_AFTER    = pd.Timestamp('2021-08-19')   # code106 : rapprochée du feu (avant : 2021-09-05)   # largement après la maîtrise totale (18/08, ~1100 ha selon
                                                      # la chronologie la plus récente reçue le 23/09/2026, corrige
                                                      # la première estimation à ~725 ha)

# 2 incendies supplémentaires (chronologie élargie reçue le 23/09/2026, secteur Tanaqoub/Bouhachem, partie 'c'
# du shapefile = Tanaqoub/Derdara Sud) : dans la période Sentinel-2 (opérationnel depuis 2017), donc exploitables
# pour le dNBR, contrairement aux événements plus anciens documentés ci-dessous.
FIRE_2019A_DATE    = pd.Timestamp('2019-08-13')   # Tanaqoub-Bouhachem, douar Beni Belal - grand incendie, maîtrisé après >15h
FIRE_2019A_BEFORE  = pd.Timestamp('2019-08-12')   # code106 : rapprochée du feu (avant : 2019-08-01)
FIRE_2019A_AFTER   = pd.Timestamp('2019-08-15')   # code106 : rapprochée du feu (avant : 2019-08-28)   # avant le début de la fenêtre du 2e feu (05/09)
FIRE_2019B_DATE    = pd.Timestamp('2019-09-13')   # Tanaqoub-Bouhachem, nouveau feu, contenu par les secours
FIRE_2019B_BEFORE  = pd.Timestamp('2019-09-12')   # code106 : rapprochée du feu (avant : 2019-09-05)   # après la résolution du feu d'août
FIRE_2019B_AFTER   = pd.Timestamp('2019-09-15')   # code106 : rapprochée du feu (avant : 2019-09-30)

# 4 incendies plus anciens (chronologie élargie reçue le 23/09/2026, étude historique de la province de
# Chefchaouen) : antérieurs à Sentinel-2 (opérationnel depuis 2015). Dans code103 ils étaient EXCLUS
# faute de dNBR possible. Depuis code104 (23/09/2026), le dNBR est calculé pour ces 4 dates via
# l'archive LANDSAT (Landsat 5 TM, opérationnel jusqu'à fin 2011, couvre bien ces 2 dates) au lieu de
# Sentinel-2 - voir _resolve_burn_scene_landsat. Résolution plus grossière (30 m contre 10-20 m pour
# Sentinel-2) mais une vraie vérification spectrale, bien plus fiable qu'un simple point FIRMS/MODIS
# (1 km, souvent aveugle aux feux de cette taille - MODIS avant VIIRS 2012 est particulièrement grossier).
SENTINEL2_AVAILABLE_FROM = pd.Timestamp('2015-06-01')   # bascule automatique Landsat/Sentinel-2 dans compute_dnbr_label
LANDSAT_SR_SCALE  = 0.0000275   # coefficients standards USGS Collection 2 Niveau 2 (réflectance de surface)
LANDSAT_SR_OFFSET = -0.2        # DN -> réflectance : refl = DN * LANDSAT_SR_SCALE + LANDSAT_SR_OFFSET

FIRE_2005_DATE    = pd.Timestamp('2005-09-21')   # Tanaqoub-Bouhachem, 214 ha brûlés
FIRE_2005_BEFORE  = pd.Timestamp('2005-09-20')   # code106 : rapprochée du feu (avant : 2005-08-01)
FIRE_2005_AFTER   = pd.Timestamp('2005-09-23')   # code106 : rapprochée du feu (avant : 2005-11-15)

FIRE_2009A_DATE   = pd.Timestamp('2009-07-26')   # Derdara - Jbel Sougna, 149 ha brûlés, départ signalé à 16h.
                                                  # FIRMS confirme une détection le 28/07/2009 (+2j, dans la
                                                  # fenêtre de tolérance du repli FIRMS - voir label_fire_window)
FIRE_2009A_BEFORE = pd.Timestamp('2009-07-25')   # code106 : rapprochée du feu (avant : 2009-06-01)
FIRE_2009A_AFTER  = pd.Timestamp('2009-07-28')   # code106 : rapprochée du feu (avant : 2009-10-01)

# FUSIONNÉ dans code105 (23/09/2026) : Derdara-Bouhachem (11/08, 168 ha) et Mont Bouhachem (12/08,
# >20 ha) étaient traités comme 2 incendies distincts dans code104. Une détection FIRMS continue sur 2
# jours consécutifs (11-12/08/2009), décrite dans les 2 sources autour de "Bouhachem", est la signature
# typique d'UN SEUL feu qui se propage sur 2 jours, pas de 2 incendies indépendants - on les fusionne
# donc en un seul événement d'entraînement 2009BC plutôt que de les compter deux fois.
# Bilan FIRMS pré-2025 (confirmé 23/09/2026) : 2009a (26/07) a une détection 2 jours après (28/07) ;
# 2009BC (11-12/08) a une détection continue sur les 2 jours ; SEUL 2005 (21/09) n'a aucune détection
# FIRMS confirmée - son label reposera donc entièrement sur le dNBR Landsat, sans repli possible.
FIRE_2009BC_DATE    = pd.Timestamp('2009-08-11')   # Derdara-Bouhachem / Mont Bouhachem, complexe 11-12/08/2009 (168+20 ha)
FIRE_2009BC_BEFORE  = pd.Timestamp('2009-08-10')   # code106 : rapprochée du feu (avant : 2009-06-15)
FIRE_2009BC_AFTER   = pd.Timestamp('2009-08-14')   # code106 : rapprochée du feu (avant : 2009-10-15)

# Historique : ces 4 incendies étaient EXCLUS dans code103 (pas de Sentinel-2 disponible avant 2015).
# Depuis code104, ils sont vérifiés par dNBR via l'archive Landsat et intégrés à EVENTS_ENTRAINEMENT
# (voir cellules suivantes) - ce dict reste disponible, vide pour l'instant, si un futur incendie
# s'avère vraiment invérifiable (ni Landsat ni FIRMS).
EXCLUDED_PRE_SENTINEL2_EVENTS = {}

# ÉVÉNEMENT 2025 A "Derdara" : départ 12/08, 3 des 4 foyers maîtrisés le 13, circonscrit le 14, extinction
# confirmée le 15, reprise isolée le 18 maîtrisée le jour même.
PRE_2025_DERDARA           = pd.Timestamp('2025-08-11')   # code106 : rapprochée du feu (avant : 2025-08-05)
TEST_DATE_2025_DERDARA     = pd.Timestamp('2025-08-12')   # code202 : jour du départ (avant : 13/08), comme pour l'entraînement
AFTER_2025_DERDARA         = pd.Timestamp('2025-08-19')   # après la reprise du 18, avant le départ du 2e feu (21/08)

# ÉVÉNEMENT 2025 B "Bouhachem, près de Derdara" : départ 21/08, foyers actifs 22-23, maîtrise totale le 24.
PRE_2025_BOUHACHEM_PRES_DERDARA        = pd.Timestamp('2025-08-20')   # code106 : rapprochée du feu (avant : 2025-08-19)   # même image que AFTER_2025_DERDARA (juste
                                                                        # après la résolution du feu A, avant le B)
TEST_DATE_2025_BOUHACHEM_PRES_DERDARA  = pd.Timestamp('2025-08-21')
AFTER_2025_BOUHACHEM_PRES_DERDARA      = pd.Timestamp('2025-08-25')   # code106 : rapprochée du feu (avant : 2025-08-29)

# Rétro-compatibilité : alias sur l'événement A (Derdara) pour tout code qui attendrait encore ces noms génériques.
PRE_2025_DATE   = PRE_2025_DERDARA
TEST_DATE_2025  = TEST_DATE_2025_DERDARA
AFTER_2025_DATE = AFTER_2025_DERDARA

# code106 : 'debut'/'fin' = départ et maîtrise (masque FIRMS de la vérité) ; 'sd_avant'/'sd_apres' = fenêtre de
# recherche maximale des images (jours), pour que la vérité d'un feu ne contienne pas l'autre (A : 12-18/08, B : 21-24/08).
EVENEMENTS_2025 = [
    {'nom':'Derdara','slug':'derdara','pre':PRE_2025_DERDARA,'test':TEST_DATE_2025_DERDARA,'after':AFTER_2025_DERDARA,
     'debut':pd.Timestamp('2025-08-12'),'fin':pd.Timestamp('2025-08-18'),'sd_avant':None,'sd_apres':2},
    {'nom':'Bouhachem, près de Derdara','slug':'bouhachem_pres_derdara','pre':PRE_2025_BOUHACHEM_PRES_DERDARA,
     'test':TEST_DATE_2025_BOUHACHEM_PRES_DERDARA,'after':AFTER_2025_BOUHACHEM_PRES_DERDARA,
     'debut':pd.Timestamp('2025-08-21'),'fin':pd.Timestamp('2025-08-24'),'sd_avant':6,'sd_apres':None},
]

# Fenêtres approximatives des incendies voisins non retenus (pour la vérification géographique automatique) :
EXCLUDED_NEIGHBOR_EVENTS = {
    'Larache/Sahel_El_Menzla_2022': (pd.Timestamp('2022-07-13'), pd.Timestamp('2022-07-18')),
    'Jbel_Lahbib_Tetouan_2017':     (pd.Timestamp('2017-06-30'), pd.Timestamp('2017-07-03')),
}

BURN_DNBR_THRESHOLD = 0.27   # code106 (avant : 0.10) : début de la sévérité "modérée" USGS ; 0.10 = limite du "non brûlé",
                             # dépassée par le simple dessèchement d'été -> labels 10 à 90 fois trop grands dans code105
N_NEG_DAYS = 20               # nombre de journées "sans feu" additionnelles tirées du cache code89
NEG_EXCLUSION_DAYS = 30       # marge en jours autour de chaque incendie pour exclure des journées "négatives" (fumée, stress post-feu...)
FIRE_2020_WINDOW_DAYS = 2      # tolérance (en jours) autour de FIRE_2020_DATE si aucune détection FIRMS ne tombe exactement ce jour-là

# --- NOUVEAU (code106) ---
AGE_MAX_SENTINEL2 = 10   # code202 : âge maximal (jours) de l'image Sentinel-2 utilisée pour le NBR/NDVI/NDMI du jour
AGE_MAX_LANDSAT   = 20   # code202 : idem pour Landsat (2005/2009, revisite 16 jours)
# Dans les deux cas, l'image doit être prise au plus tard la VEILLE du jour étudié (J-1) : jamais le jour même.
CELL_HA = (GRID/100)**2          # surface d'une cellule en hectares (6.25 ha pour 250 m)
FIRMS_MASK_BUFFER_M = 2000       # une cellule n'est 'brûlée' que si elle est à moins de 2 km d'une détection FIRMS du feu
FACTEUR_SURFACE_MAX = 2.0        # si le label dépasse 2 x la surface officielle -> on garde les cellules au dNBR le plus fort
FRACTIONS_ALERTE_CANDIDATES = [0.05,0.10,0.15,0.20]   # part de cellules en alerte chaque jour (choisie en validation ; 20 % max pour rester utile sur le terrain)
MOIS_ETE = [6,7,8,9]             # mois de référence pour l'indice de danger météo du jour
DANGER_VARS = {'temperature_2m_max':+1,'vpd_max':+1,'wind_speed_10m_max':+1,'relative_humidity_2m_min':-1}  # +1 : plus haut = plus dangereux

print('Cache code89 :',WINDOW_CACHE_89)
print('Incendies entraînement (Sentinel-2) : 2020 ({}), 2022 ({}), Derdara 2021 ({}), Tanaqoub 2019a ({}), Tanaqoub 2019b ({})'.format(
      FIRE_2020_DATE.date(),FIRE_2022_EVENT_DATE.date(),DERDARA_2021_DATE.date(),FIRE_2019A_DATE.date(),FIRE_2019B_DATE.date()))
print('Incendies entraînement (Landsat, pré-2015) : 2005 ({}), 2009a ({}), 2009bc ({}, fusion 11-12/08 - seule date '
      'confirmée par FIRMS avant 2025)'.format(FIRE_2005_DATE.date(),FIRE_2009A_DATE.date(),FIRE_2009BC_DATE.date()))
print('Tests aveugles 2025 :',', '.join('{} ({})'.format(e['nom'],e['test'].date()) for e in EVENEMENTS_2025))
if EXCLUDED_PRE_SENTINEL2_EVENTS:
    print('Incendies documentés mais exclus (invérifiables) :',', '.join(EXCLUDED_PRE_SENTINEL2_EVENTS))


# ------------------------------------------------------------------
# code201 (correctif) : test de connexion internet AVANT tout calcul. Sentinel-2/Landsat (Planetary Computer)
# et la météo (Open-Meteo) sont lus en ligne : sans connexion, le notebook s'arrête ici avec un message
# clair, au lieu de remplacer silencieusement tous les labels dNBR par FIRMS plus loin.
# ------------------------------------------------------------------
import socket
def verifier_connexion(hotes=('planetarycomputer.microsoft.com','archive-api.open-meteo.com')):
    for hote in hotes:
        try:
            socket.create_connection((hote,443),timeout=10).close()
        except OSError as exc:
            raise ConnectionError("Pas d'accès à {} ({}). Vérifie la connexion internet (Wi-Fi, VPN, proxy), "
                                  "puis relance le notebook depuis le début.".format(hote,exc)) from None
    print('Connexion internet OK :',', '.join(hotes))
verifier_connexion()


# ------------------------------------------------------------------
# code202 : contrôle des inputs reprojetés - chaque fichier doit exister et être en EPSG:26191.
# ------------------------------------------------------------------
OSM_DIR_REPROJ=REPROJ/'osm'
LULC_REPROJ=REPROJ/'occupation_sol_2021_26191_10m.tif'
def controler_inputs_reprojetes():
    from pyproj import CRS as _CRS
    attendu=_CRS.from_user_input(CRS)
    fichiers=[FOREST,DEM,POPULATION,LULC_REPROJ]+sorted(FIRMS_DIR.glob('fire_*.shp'))+sorted(OSM_DIR_REPROJ.glob('osm_code13_2022_*.parquet'))
    if not REPROJ.exists() or len(fichiers)<4+3+5:
        raise FileNotFoundError("Inputs reprojetés incomplets dans {} : vérifie le contenu du dossier data/ (voir la liste en haut de cette cellule).".format(REPROJ))
    for p in fichiers:
        if not p.exists(): raise FileNotFoundError('Fichier reprojeté manquant : '+str(p))
        if p.suffix=='.tif':
            with rasterio.open(p) as s: crs=s.crs
        elif p.suffix=='.shp': crs=gpd.read_file(p,rows=1).crs
        else: crs=gpd.read_parquet(p).crs
        if crs is None or not _CRS.from_user_input(crs).equals(attendu):
            raise ValueError('{} n\'est pas en {} ({}) : fichier à remplacer dans data/.'.format(p.name,CRS,crs))
        print('  {:<50} {}'.format(str(p.relative_to(REPROJ)),CRS))
    print('Tous les inputs locaux sont en',CRS)
controler_inputs_reprojetes()


# ------------------------------------------------------------------
# code201 (correctif) : Open-Meteo répond parfois très lentement (ReadTimeout après 120 s). Chaque requête
# est maintenant réessayée jusqu'à 5 fois, avec une attente croissante (10, 20, 40, 80 s) entre les essais.
# ------------------------------------------------------------------
def requete_avec_reessais(url,params=None,timeout=120,essais=5,attente=10):
    for k in range(essais):
        try:
            r=requests.get(url,params=params,timeout=timeout)
            if r.status_code in (429,500,502,503,504): raise requests.HTTPError('statut {}'.format(r.status_code))
            return r
        except (requests.Timeout,requests.ConnectionError,requests.HTTPError) as exc:
            if k==essais-1: raise
            print('  {} : essai {}/{} échoué ({}) - nouvel essai dans {} s.'.format(url.split('/')[2],k+1,essais,type(exc).__name__,attente*2**k))
            time.sleep(attente*2**k)


In [ ]:
# Même grille et mêmes FIRMS que code89/code90
# ------------------------------------------------------------------
# code202 : contrôle de projection de la forêt. Le .prj est un "Nord_Maroc" ESRI (Merchich, Lambert conique
# conforme) aux mêmes paramètres que EPSG:26191 : la reprojection ne doit donc pas déplacer le polygone.
# ------------------------------------------------------------------
forest_brut=gpd.read_file(FOREST)
print('Projection du shapefile de la forêt :',forest_brut.crs.name if forest_brut.crs else 'AUCUNE')
forest=forest_brut.to_crs(CRS)
_ecart=np.abs(np.asarray(forest.total_bounds)-np.asarray(forest_brut.total_bounds)).max()
if _ecart>1:
    print('ATTENTION : la reprojection vers {} déplace la forêt de {:.1f} m alors que les paramètres sont identiques - '
          'on garde les coordonnées d\'origine (set_crs) pour ne pas décaler la forêt.'.format(CRS,_ecart))
    forest=forest_brut.set_crs(CRS,allow_override=True)
else:
    print('Reprojection de la forêt vers {} : déplacement {:.2f} m (attendu ~0) - OK.'.format(CRS,_ecart))
forest['partie']=forest['nom'].astype(str).str.lower() if 'nom' in forest else [chr(97+k) for k in range(len(forest))]

# ------------------------------------------------------------------
# code202 : UNE SEULE grille sur toute la forêt (les 6 parties réunies). Une cellule de 250 m n'existe qu'une
# fois, même si elle est à cheval sur deux parties. `partie` = partie qui occupe la plus grande surface de la
# cellule, gardée seulement pour l'information et les diagnostics (jamais utilisée par le modèle).
# ------------------------------------------------------------------
foret_unique=forest.geometry.union_all()
minx,miny,maxx,maxy=foret_unique.bounds
_sindex=forest.sindex
rows=[]
for x in np.arange(math.floor(minx/GRID)*GRID,maxx,GRID):
    for y in np.arange(math.floor(miny/GRID)*GRID,maxy,GRID):
        carre=box(x,y,x+GRID,y+GRID)
        geom=carre.intersection(foret_unique)
        if geom.is_empty or geom.area<100: continue
        _cand=forest.iloc[list(_sindex.query(carre,predicate='intersects'))]
        _partie=_cand.partie.iloc[int(np.argmax(_cand.geometry.intersection(carre).area.to_numpy()))] if len(_cand) else '?'
        rows.append({'id_cell':f'foret_{len(rows):05d}','partie':_partie,'geometry':geom})
grid=gpd.GeoDataFrame(rows,crs=CRS)

# ------------------------------------------------------------------
# code203 : CENTRE et RECTANGLE de la forêt calculés en Merchich (mètres), puis convertis en degrés WGS 84
# UNIQUEMENT pour les deux services en ligne qui l'exigent (Open-Meteo : un point ; Planetary Computer :
# un rectangle de recherche). Conversion par PROJ avec changement de datum Merchich -> WGS 84 (~335 m ici).
# Le rectangle est densifié (21 points par côté) : un rectangle droit en Merchich est légèrement courbé en
# degrés, ses 4 coins seuls ne suffiraient pas à l'englober.
# ------------------------------------------------------------------
from pyproj import Transformer
from shapely.geometry import Point
_vers_wgs84=Transformer.from_crs(CRS,'EPSG:4326',always_xy=True)
CENTRE_MERCHICH=foret_unique.centroid
BBOX_MERCHICH=foret_unique.bounds
_lon,_lat=_vers_wgs84.transform(CENTRE_MERCHICH.x,CENTRE_MERCHICH.y)
CENTRE_WGS84=Point(_lon,_lat)
BBOX_WGS84=list(_vers_wgs84.transform_bounds(*BBOX_MERCHICH,densify_pts=21))
print('Centre de la forêt   : Merchich X = {:.1f} m, Y = {:.1f} m  ->  WGS 84 lat = {:.6f}°, lon = {:.6f}°'.format(
      CENTRE_MERCHICH.x,CENTRE_MERCHICH.y,CENTRE_WGS84.y,CENTRE_WGS84.x))
print('Rectangle de la forêt: Merchich X {:.0f} -> {:.0f} m, Y {:.0f} -> {:.0f} m'.format(
      BBOX_MERCHICH[0],BBOX_MERCHICH[2],BBOX_MERCHICH[1],BBOX_MERCHICH[3]))
print('                       WGS 84 lon {:.5f} -> {:.5f}°, lat {:.5f} -> {:.5f}°'.format(
      BBOX_WGS84[0],BBOX_WGS84[2],BBOX_WGS84[1],BBOX_WGS84[3]))
print('(Ces valeurs en degrés servent seulement à interroger Open-Meteo et Planetary Computer ; tous les calculs restent en Merchich.)')
grid['surface_foret_km2']=grid.area/1_000_000
print('Grille unique : {} cellules sur toute la forêt ({:.0f} ha). Répartition indicative par partie :'.format(
      len(grid),grid.area.sum()/1e4),grid.partie.value_counts().sort_index().to_dict())

fire_frames=[]
for path in FIRMS_DIR.rglob('fire_*.shp'):
    f=gpd.read_file(path)
    print('  {} : projection {}'.format(path.name,f.crs.name if f.crs else 'AUCUNE'))
    if f.crs is None: raise ValueError('Fichier FIRMS sans projection (.prj manquant) : '+str(path))
    f=f.to_crs(CRS)
    f['date']=pd.to_datetime(f['ACQ_DATE'],errors='coerce').dt.normalize()
    f['frp']=pd.to_numeric(f['FRP'],errors='coerce').fillna(0)
    fire_frames.append(f[['date','frp','geometry']])
firms=gpd.GeoDataFrame(pd.concat(fire_frames,ignore_index=True),crs=CRS).dropna(subset=['date'])
firms=firms[firms.geometry.intersects(forest.geometry.union_all().buffer(5000))]
firms=firms.drop_duplicates(subset=['date','geometry']).sort_values('date').reset_index(drop=True)
print(len(grid),'cellules | FIRMS',firms.date.min().date(),'→',firms.date.max().date())
print('Contrôle projection : {} point(s) FIRMS dans la forêt elle-même, {} à moins de 5 km.'.format(
      int(firms.geometry.within(foret_unique).sum()),len(firms)))

In [ ]:
# ------------------------------------------------------------------
# code202 : contrôle de la forêt UNIQUE, en Merchich / Nord Maroc (EPSG:26191), coordonnées en mètres.
# (L'ancienne cellule qui affichait le centre de chaque partie en degrés a été supprimée : elle ne servait
# qu'à l'affichage et ne participait à aucun calcul.)
# ------------------------------------------------------------------
_centre=foret_unique.centroid
print('Forêt unique : {} polygone(s) réunis en une seule géométrie, surface {:.0f} ha, système {}.'.format(
      len(forest),foret_unique.area/1e4,grid.crs.to_string()))
print('Centre de la forêt : X = {:.0f} m, Y = {:.0f} m (Merchich / Nord Maroc).'.format(_centre.x,_centre.y))
print('Emprise : X {:.0f} -> {:.0f} m, Y {:.0f} -> {:.0f} m.'.format(*foret_unique.bounds[0::2],*foret_unique.bounds[1::2]))


## Inventaire de tous les episodes de feu disponibles (au-dela de 2020 et 2022)

Avant de se limiter a 2 evenements d'entrainement, autant verifier combien d'autres episodes de detection FIRMS existent reellement dans/pres de la foret sur toute la periode couverte par les fichiers locaux. Les detections brutes sont regroupees en episodes (jours proches dans le temps) plutot que listees jour par jour, pour donner une vraie liste de candidats a evenements supplementaires — qui pourraient reduire le probleme de generalisation observe en leave-one-fire-out.

Le tableau complet est aussi exporte en CSV (`inventaire_episodes_firms_bouhachem.csv`) pour pouvoir l'inspecter ou le partager en dehors de Jupyter sans risquer une troncature d'affichage.

In [ ]:
# ------------------------------------------------------------------
# Inventaire de TOUS les episodes de detection FIRMS dans/pres de la foret, sur toute la periode
# disponible dans le shapefile local - pas seulement 2020 et 2022. Sert a reperer d'autres incendies
# exploitables comme evenements d'entrainement supplementaires (voir la discussion sur le manque de
# donnees avec seulement 2 evenements en leave-one-fire-out).
#
# Les detections brutes sont regroupees en "episodes" (des jours consecutifs ou proches, tolerance de
# 5 jours entre deux detections, pour tenir compte des trous dus aux nuages/passages du satellite) car
# un seul incendie s'etale generalement sur plusieurs jours - regarder jour par jour sous-estimerait
# fortement chaque evenement et melangerait un vrai gros feu avec du bruit isole.
# ------------------------------------------------------------------
forest_only=forest.geometry.union_all()
firms_in_forest=firms[firms.geometry.within(forest_only.buffer(GRID))]  # tolerance d'une cellule de grille
detections_par_jour=(firms_in_forest.assign(jour=firms_in_forest.date.dt.date)
                      .groupby('jour')
                      .agg(nb_detections=('frp','size'),frp_total=('frp','sum'))
                      .sort_index())

jours_tries=list(detections_par_jour.index)
episodes=[]
if jours_tries:
    courant=[jours_tries[0]]
    for jour in jours_tries[1:]:
        if (jour-courant[-1]).days<=5:
            courant.append(jour)
        else:
            episodes.append(courant); courant=[jour]
    episodes.append(courant)

evenements_connus=[(FIRE_2020_DATE.date(),'incendie_2020_deja_utilise'),
                    (FIRE_2022_EVENT_DATE.date(),'incendie_2022_deja_utilise')]

rows=[]
for ep in episodes:
    debut,fin=ep[0],ep[-1]
    sous=detections_par_jour.loc[ep]
    chevauche=[nom for date_connue,nom in evenements_connus
               if (debut-pd.Timedelta(days=5))<=date_connue<=(fin+pd.Timedelta(days=5))]
    rows.append({'debut':debut,'fin':fin,'duree_jours':(fin-debut).days+1,
                 'nb_jours_avec_detection':len(ep),'nb_detections_total':int(sous.nb_detections.sum()),
                 'frp_total':float(sous.frp_total.sum()),
                 'deja_utilise_comme':', '.join(chevauche) if chevauche else ''})

inventaire_episodes=pd.DataFrame(rows).sort_values('frp_total',ascending=False).reset_index(drop=True)

print('{} jour(s) distinct(s) avec detection FIRMS dans/pres de la foret, regroupes en {} episode(s), '
      'entre {} et {} (donnees locales).'.format(
      len(detections_par_jour),len(inventaire_episodes),firms.date.min().date(),firms.date.max().date()))
print('Les episodes deja utilises en entrainement (2020, 2022) sont marques dans la colonne "deja_utilise_comme".')
print('Chaque AUTRE episode est un candidat potentiel a verifier (geographie, dates precises, disponibilite')
print('Sentinel-2) avant de l\'ajouter comme evenement d\'entrainement supplementaire.')

pd.set_option('display.max_rows',None); pd.set_option('display.width',200)
display(inventaire_episodes)
pd.reset_option('display.max_rows'); pd.reset_option('display.width')

chemin_inventaire=REPORT/'inventaire_episodes_firms_bouhachem.csv'
inventaire_episodes.to_csv(chemin_inventaire,index=False)
print('Inventaire complet exporte (pour partage/inspection hors Jupyter) :',chemin_inventaire)

## Vérification géographique des incendies voisins (Larache 2022, Derdara 2021, Jbel Lahbib 2017)

Ces 3 incendies sont documentés dans des sources externes mais décrits comme situés dans des forêts
**voisines**, pas dans le massif de Bouhachem lui-même (`limites_Bouhachem.shp`). Plutôt que de faire
confiance aux toponymes des articles de presse, on vérifie ici directement s'il existe des détections
FIRMS **locales** (le shapefile déjà chargé dans ce notebook) qui tombent réellement à l'intérieur du
polygone de la forêt pendant ces fenêtres de dates. Si oui, l'incendie a débordé sur Bouhachem et
pourrait être ajouté à l'entraînement ; si non, l'exclusion est confirmée.

In [ ]:
# ------------------------------------------------------------------
# Pour chaque incendie voisin non retenu, on regarde s'il y a des détections FIRMS *dans* la forêt
# (firms_in_forest, tolérance 1 cellule de grille) pendant sa fenêtre de dates. `firms` contient déjà
# un buffer plus large (5 km autour de la forêt) donc les détections de ces feux voisins doivent
# normalement y apparaître, mais PAS dans firms_in_forest si le feu est réellement à l'extérieur.
# ------------------------------------------------------------------
for nom,(debut,fin) in EXCLUDED_NEIGHBOR_EVENTS.items():
    proche=firms[(firms.date>=debut-pd.Timedelta(days=1))&(firms.date<=fin+pd.Timedelta(days=1))]
    dedans=firms_in_forest[(firms_in_forest.date>=debut-pd.Timedelta(days=1))&(firms_in_forest.date<=fin+pd.Timedelta(days=1))]
    print('{:<32} {} -> {} : {} détection(s) à proximité (buffer 5km), dont {} RÉELLEMENT dans la forêt.'.format(
          nom,debut.date(),fin.date(),len(proche),len(dedans)))
    if len(dedans):
        print('   -> ATTENTION : ce feu semble avoir touché la forêt de Bouhachem elle-même. Dates concernées :',
              sorted(dedans.date.dt.date.unique()),'. À ré-évaluer comme événement d\'entraînement potentiel.')
    else:
        print('   -> aucune détection dans la forêt : exclusion confirmée, cet événement reste hors apprentissage.')


In [ ]:
# Facteurs structurels : on réutilise le cache déjà calculé par code90 s'il existe (même grille, mêmes id_cell),
# sinon on les recalcule une seule fois avec exactement la même méthode que code90.
#
# CORRECTIF (code105, 23/09/2026) : ce cache a été calculé à l'origine sur l'ANCIENNE grille (limites_Bouhachem.shp,
# sans Jbel Sougna). Avant de lui faire confiance, on vérifie que TOUS les id_cell de la grille ACTUELLE (avec
# Jbel Sougna = parties d/e/f) sont bien présents dedans - sinon un simple merge 'left' laisserait silencieusement
# altitude_m/pente_pct/exposition_deg/... à NaN pour toute la zone Jbel Sougna (imputés ensuite à une médiane
# globale non pertinente, sans aucun avertissement). Si des id_cell manquent, on recalcule tout depuis le DEM/OSM
# au lieu de réutiliser un cache partiel.
cache_structural = PREP_90/'facteurs_structurels_code90.parquet'
_reuse_cache_structural = False
if False and cache_structural.exists():   # code202 : jamais réutilisé, les facteurs sont recalculés depuis les inputs reprojetés
    structural_flat = pd.read_parquet(cache_structural)
    _manquants = set(grid.id_cell) - set(structural_flat.id_cell)
    if _manquants:
        print("ATTENTION : le cache structurel de code90 ({}) ne couvre pas {} cellule(s) de la grille actuelle "
              "(probablement Jbel Sougna, ajouté le 23/09/2026 - vérifie aussi que mnt_bouhachem.tif et les fichiers "
              "OSM couvrent bien géographiquement cette zone) -> recalcul complet depuis le DEM/OSM.".format(
              cache_structural,len(_manquants)))
    else:
        _reuse_cache_structural = True

if _reuse_cache_structural:
    # On ne prend que id_cell + geometry de `grid` : si on fusionnait `grid` en entier, les colonnes
    # communes (surface_foret_km2, partie...) seraient dupliquées en _x/_y par le merge et cassent
    # `causal_pamf` plus loin (frame.surface_foret_km2 n'existerait plus sous ce nom).
    structural = grid[['id_cell','geometry']].merge(structural_flat, on='id_cell', how='left', validate='one_to_one')
    structural = gpd.GeoDataFrame(structural, geometry='geometry', crs=CRS)
    print('Facteurs structurels réutilisés depuis code90 :', cache_structural)
else:
    OSM_DIR=OSM_DIR_REPROJ   # code202 : OSM reprojeté
    OSM_FILES={
     'distance_route_m':OSM_DIR/'osm_code13_2022_distance_route_m.parquet',
     'distance_piste_m':OSM_DIR/'osm_code13_2022_distance_piste_m.parquet',
     'distance_batiment_m':OSM_DIR/'osm_code13_2022_distance_batiment_m.parquet',
     'distance_douar_m':OSM_DIR/'osm_code13_2022_distance_douar_m.parquet',
     'distance_agriculture_m':OSM_DIR/'osm_code13_2022_distance_agriculture_m.parquet'}
    LULC=LULC_REPROJ         # code202 : occupation du sol reprojetée (10 m, EPSG:26191)

    structural=grid.copy(); centroids=structural.geometry.centroid
    for name,path in OSM_FILES.items():
        if not path.exists(): raise FileNotFoundError(path)
        raw=pd.read_parquet(path)
        objects=gpd.GeoSeries(from_wkb(raw['geom'].to_numpy()),crs=CRS)
        structural[name]=centroids.distance(objects.union_all()).astype(float)

    pressure=[]
    for name in OSM_FILES:
        scale=max(float(structural[name].quantile(.95)),1.)
        pressure.append(1-(structural[name]/scale).clip(0,1))
    structural['pression_humaine_01']=pd.concat(pressure,axis=1).mean(axis=1)

    with rasterio.open(LULC) as src:
        cats=zonal_stats(structural.to_crs(src.crs),str(LULC),categorical=True,nodata=src.nodata,all_touched=False)
    structural['occupation_sol_2021']=[max(row,key=row.get) if row else np.nan for row in cats]
    fuel_lookup={1:0.,2:.85,4:.55,5:.65,7:.10,8:.15,9:0.,10:np.nan,11:.75}
    structural['combustibilite_01']=structural.occupation_sol_2021.map(fuel_lookup).astype(float)

    with rasterio.open(DEM) as src:
        dem=src.read(1).astype('float32'); invalid=(dem==src.nodata) if src.nodata is not None else ~np.isfinite(dem)
        dem[invalid]=np.nan
        yres=abs(src.transform.e); xres=abs(src.transform.a)
        dzdy,dzdx=np.gradient(dem,yres,xres)
        slope=np.sqrt(dzdx**2+dzdy**2)*100
        aspect=(np.degrees(np.arctan2(-dzdx,dzdy))+360)%360
        cells=structural.to_crs(src.crs)
        zs=zonal_stats(cells,dem,affine=src.transform,stats=['mean'],nodata=np.nan,all_touched=True)
        ps=zonal_stats(cells,slope,affine=src.transform,stats=['mean'],nodata=np.nan,all_touched=True)
        # code204 : exposition par MOYENNE CIRCULAIRE (moyenne des sinus et des cosinus). Une moyenne simple des
        # degrés donnait ~180° (sud) pour un versant nord dont les pixels valent ~355° et ~5°.
        az_sin=zonal_stats(cells,np.sin(np.radians(aspect)),affine=src.transform,stats=['mean'],nodata=np.nan,all_touched=True)
        az_cos=zonal_stats(cells,np.cos(np.radians(aspect)),affine=src.transform,stats=['mean'],nodata=np.nan,all_touched=True)
    structural['altitude_m']=[z['mean'] for z in zs]
    structural['pente_pct']=[z['mean'] for z in ps]
    _s=np.array([np.nan if z['mean'] is None else z['mean'] for z in az_sin],float)
    _c=np.array([np.nan if z['mean'] is None else z['mean'] for z in az_cos],float)
    structural['exposition_deg']=(np.degrees(np.arctan2(_s,_c))+360)%360
    # code202 : population (habitants par cellule) recalculée sur la nouvelle grille
    if POPULATION.exists():
        with rasterio.open(POPULATION) as src:
            _pop=zonal_stats(structural.to_crs(src.crs),str(POPULATION),stats=['sum'],nodata=src.nodata,all_touched=True)
        structural['population']=[z['sum'] if z['sum'] is not None else 0. for z in _pop]
    else:
        print('ATTENTION : raster de population introuvable ({}) - variable population absente.'.format(POPULATION))
    structural.drop(columns='geometry').to_parquet(PREP/'facteurs_structurels_code208.parquet',index=False)
    print('Facteurs structurels recalculés depuis les inputs reprojetés en {}.'.format(CRS))

display(structural.drop(columns='geometry').describe().T)

In [ ]:
# ------------------------------------------------------------------
# Vérification de couverture par partie de forêt (Bouhachem / Derdara / Tanaqoub / Jbel Sougna x3) : le
# DEM local (mnt_bouhachem.tif) et les fichiers OSM ont pu être calculés à l'origine seulement sur
# l'emprise de Bouhachem - si c'est le cas, altitude_m/pente_pct/exposition_deg (et les distances OSM)
# seraient NaN pour toutes les cellules des parties ajoutées ensuite, ce qui casserait SILENCIEUSEMENT
# l'entraînement/test sur ces zones (features manquantes -> imputées à une valeur médiane globale non
# pertinente localement). La cellule précédente (code105) recalcule déjà automatiquement depuis le DEM/
# OSM si le cache code90 ne couvre pas tous les id_cell de la grille actuelle - mais ça ne sert à rien si
# le DEM/les fichiers OSM eux-mêmes ne couvrent pas géographiquement Jbel Sougna (d/e/f) : c'est ce que
# cette cellule vérifie. Si une partie a une couverture très faible ci-dessous, il faut un DEM/extraction
# OSM regénéré sur toute l'emprise des 6 parties, pas seulement Bouhachem.
# ------------------------------------------------------------------
colonnes_a_verifier=[c for c in ['altitude_m','pente_pct','exposition_deg','distance_route_m','distance_piste_m',
                                  'distance_batiment_m','distance_douar_m','distance_agriculture_m',
                                  'occupation_sol_2021','combustibilite_01'] if c in structural.columns]
_base_couverture=structural.drop(columns=['partie'],errors='ignore')[['id_cell']+colonnes_a_verifier]
couverture_par_partie=_base_couverture.merge(grid[['id_cell','partie']],on='id_cell',how='left')
couverture_par_partie=couverture_par_partie.groupby('partie')[colonnes_a_verifier].apply(lambda d:d.notna().mean())
print('Couverture (proportion de cellules NON manquantes) par partie de forêt et par facteur structurel :')
display(couverture_par_partie)
for _partie in couverture_par_partie.index:
    _pires=couverture_par_partie.loc[_partie][couverture_par_partie.loc[_partie]<0.5]
    if len(_pires):
        print('  ATTENTION : partie "{}" a une couverture < 50% pour : {} - le DEM/cache structurel ne couvre '
              'probablement pas cette zone.'.format(_partie,', '.join(_pires.index)))


In [ ]:
def label_fire_today(frame,date):
    date=pd.Timestamp(date).normalize(); active=firms[firms.date==date]
    if active.empty:return np.zeros(len(frame),dtype='uint8')
    return frame.geometry.intersects(active.geometry.union_all().buffer(FIRE_RADIUS)).astype('uint8').to_numpy()

def causal_pamf(frame,date):
    # Uniquement les feux de 2017 jusqu'à J-1 : aucune fuite du jour cible.
    date=pd.Timestamp(date).normalize()
    prior=firms[(firms.date>=TRAIN_START)&(firms.date<date)]
    counts=pd.Series(0.,index=frame.id_cell)
    frp=pd.Series(0.,index=frame.id_cell)
    if not prior.empty:
        joined=gpd.sjoin(prior,frame[['id_cell','geometry']],predicate='within',how='inner')
        if len(joined):
            counts=joined.groupby('id_cell').size().reindex(frame.id_cell,fill_value=0).astype(float)
            frp=joined.groupby('id_cell').frp.sum().reindex(frame.id_cell,fill_value=0).astype(float)
    area=frame.surface_foret_km2.clip(lower=1e-6).to_numpy()
    raw=counts.to_numpy()/area
    out=pd.DataFrame({'pamf_feux_km2':raw,'pamf_log1p':np.log1p(raw),'frp_historique_km2':frp.to_numpy()/area},index=frame.index)
    out['pamf_classe_1_5']=1
    positive=out.pamf_feux_km2>0
    if positive.sum()>=5:
        out.loc[positive,'pamf_classe_1_5']=pd.qcut(out.loc[positive,'pamf_feux_km2'],5,labels=False,duplicates='drop').astype(int)+1
    return out

# ------------------------------------------------------------------
# Topoclimat (methode Chefchaouen-Ouazzane, Assali et al. 2016 ; cf. workflow SIG/IA fourni) :
# combine l'exposition de chaque cellule (statique, deja calculee par les facteurs structurels) et la
# direction moyenne du vent du jour (dynamique) pour reperer les versants exposes au vent sec/chaud.
# Formule publiee : indice = 2 + cos(exposition - direction_vent) + cos(exposition - 180), angles en
# radians. Valeurs elevees = versant plus sec/ventile, donc plus propice a l'eclosion.
# ------------------------------------------------------------------
WIND_DIR_CACHE = PREP/'direction_vent_cache.parquet'

def _load_wind_dir_cache():
    if WIND_DIR_CACHE.exists():
        return pd.read_parquet(WIND_DIR_CACHE).set_index('date')['wind_direction_deg'].to_dict()
    return {}

def _save_wind_dir_cache(cache_dict):
    WIND_DIR_CACHE.parent.mkdir(parents=True,exist_ok=True)
    pd.DataFrame({'date':list(cache_dict.keys()),'wind_direction_deg':list(cache_dict.values())}).to_parquet(WIND_DIR_CACHE,index=False)

_WIND_DIR_MEMO=_load_wind_dir_cache()

def get_wind_direction_deg(date,operational=False):
    """Direction moyenne (circulaire, ponderee par la vitesse) du vent du jour au centroide de la foret.
    Mise en cache a part (fichier dedie), sans jamais toucher aux caches existants de code89/code90,
    pour eviter un appel reseau a chaque relecture d'une meme date deja vue."""
    date=pd.Timestamp(date).normalize(); key=date.date().isoformat()
    if key in _WIND_DIR_MEMO and not (operational and date.normalize()==pd.Timestamp.now().normalize()):
        return _WIND_DIR_MEMO[key]
    center=CENTRE_WGS84   # code203 : centre calculé en Merchich puis converti (voir cellule de la grille)
    url='https://api.open-meteo.com/v1/forecast' if operational else 'https://archive-api.open-meteo.com/v1/archive'
    response=requete_avec_reessais(url,params={'latitude':center.y,'longitude':center.x,
      'start_date':key,'end_date':key,
      'hourly':'wind_speed_10m,wind_direction_10m','timezone':'Africa/Casablanca'},timeout=120)
    response.raise_for_status(); hourly=response.json()['hourly']
    spd=np.asarray([np.nan if x is None else x for x in hourly.get('wind_speed_10m',[])],float)
    wdir=np.asarray([np.nan if x is None else x for x in hourly.get('wind_direction_10m',[])],float)
    valid=np.isfinite(spd)&np.isfinite(wdir)&(spd>0)
    if valid.sum():
        rad=np.radians(wdir[valid])
        u=np.sum(spd[valid]*np.sin(rad)); v=np.sum(spd[valid]*np.cos(rad))
        value=float((np.degrees(np.arctan2(u,v))+360)%360)
    else:
        value=np.nan
    _WIND_DIR_MEMO[key]=value; _save_wind_dir_cache(_WIND_DIR_MEMO)
    return value

def compute_topoclimat(exposition_deg,wind_direction_deg,n_classes=5):
    exp_rad=np.radians(exposition_deg); wind_rad=np.radians(wind_direction_deg)
    brut=2+np.cos(exp_rad-wind_rad)+np.cos(exp_rad-np.radians(180))
    return brut

def classify_topoclimat(brut,n_classes=5):
    values=pd.Series(brut); unique=values[np.isfinite(values)].unique()
    if len(unique)<n_classes: return pd.Series(3,index=values.index,dtype=int)
    classes=pd.qcut(values,n_classes,labels=False,duplicates='drop').astype('Int64')+1
    return classes.fillna(3).astype(int)

def _band_means(href,frame):
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR',GDAL_HTTP_MULTIRANGE='YES',VSI_CACHE='TRUE'):
        with rasterio.open(href) as src:
            cells=frame.to_crs(src.crs); bounds=cells.total_bounds
            window=rio_from_bounds(*bounds,transform=src.transform).round_offsets().round_lengths()
            array=src.read(1,window=window,boundless=True,fill_value=src.nodata or 0).astype('float32')
            transform=src.window_transform(window); nodata=src.nodata or 0
    stats=zonal_stats(cells,array,affine=transform,stats=['mean'],nodata=nodata,all_touched=True)
    return np.asarray([x['mean'] if x['mean'] is not None else np.nan for x in stats])

def _sentinel_indices(date,lookback=30,min_coverage=0.90,max_candidates=6,max_tuiles_par_date=3):
    # Appelé seulement quand la fenêtre du jour n'existe pas. Les bandes déjà
    # présentes dans le cache satellite de code89 sont réutilisées.
    #
    # CORRECTIF (23/09/2026) : depuis que la forêt couvre 3 parties (Bouhachem/Derdara/Tanaqoub), son
    # rectangle englobant peut déborder sur plusieurs dalles/tuiles Sentinel-2 - une seule tuile ne suffit
    # plus a couvrir 90% de l'AOI (plafond observé ~83% quelle que soit la date, confirmé en pratique :
    # même nombre de cellules valides sur des dates totalement différentes). On recherche maintenant, pour
    # chaque DATE d'acquisition candidate, TOUTES les tuiles de ce passage satellite (mêmes minutes
    # d'acquisition, tuiles complémentaires et non redondantes) et on les FUSIONNE en une mosaïque avant
    # de calculer la couverture, au lieu de se limiter à une seule tuile par date.
    date=pd.Timestamp(date).normalize(); aoi=list(BBOX_WGS84)   # code203 : rectangle calculé en Merchich puis converti
    stac=pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1',modifier=pc.sign_inplace)
    search=stac.search(collections=['sentinel-2-l2a'],bbox=aoi,
      datetime=f'{(date-pd.Timedelta(days=lookback)).date()}/{date.date()}',
      query={'eo:cloud_cover':{'lt':50}})
    items=list(search.items())
    if not items: raise RuntimeError(f'Aucune scène Sentinel-2 antérieure à {date.date()}')

    groupes={}
    for raw in items:
        d=pd.Timestamp(raw.datetime).tz_localize(None).normalize()
        groupes.setdefault(d,[]).append(raw)
    dates_triees=sorted(groupes,reverse=True)  # les plus récentes d'abord, comme avant

    tried=[]; meilleur=None  # (coverage, result, scene_date, items_utilises) du meilleur candidat LISIBLE
    for scene_date in dates_triees[:max_candidates]:
        candidats=sorted(groupes[scene_date],key=lambda it:it.properties.get('eo:cloud_cover',100))[:max_tuiles_par_date]
        result={'ndvi':np.full(len(grid),np.nan),'ndmi':np.full(len(grid),np.nan),'nbr':np.full(len(grid),np.nan)}
        items_utilises=[]
        for raw in candidats:
            item=pc.sign(raw); cache=SAT_CACHE_89/f'{item.id}.npz'
            try:
                # CORRECTIF (23/09/2026) : SAT_CACHE_89 est PARTAGE entre toutes les versions du notebook
                # (code103/104/105...) et cle uniquement par item.id, sans jamais verifier que le tableau
                # mis en cache correspond a la grille ACTUELLE - or limites_forets.shp (6 parties, Jbel
                # Sougna) n'a pas le meme nombre de cellules que l'ancien limites_Bouhachem.shp (3 parties).
                # Une meme scene Sentinel-2 deja vue sous l'ancienne grille renvoyait donc un tableau plus
                # court, provoquant "boolean index did not match indexed array" plus bas des que ce resultat
                # etait combine a une mosaique dimensionnee pour la nouvelle grille. On ignore maintenant tout
                # cache dont la longueur ne correspond pas a len(grid), et on le recalcule.
                if cache.exists() and len(np.load(cache)['nbr'])==len(grid):
                    saved=np.load(cache); tuile={k:saved[k] for k in ['ndvi','ndmi','nbr']}
                else:
                    # Une erreur reseau/lecture transitoire (throttling Planetary Computer, timeout /vsicurl,
                    # etc.) sur UNE tuile ne doit pas faire echouer toute la recherche : on l'enregistre et on
                    # passe a la tuile suivante du meme passage, au lieu de laisser l'exception remonter.
                    b04=_band_means(item.assets['B04'].href,grid); b08=_band_means(item.assets['B08'].href,grid)
                    b11=_band_means(item.assets['B11'].href,grid); b12=_band_means(item.assets['B12'].href,grid)
                    if str(raw.properties.get('s2:processing_baseline','00.00'))>='04.00': b04-=1000;b08-=1000;b11-=1000;b12-=1000
                    with np.errstate(divide='ignore',invalid='ignore'):
                        tuile={'ndvi':(b08-b04)/(b08+b04),'ndmi':(b08-b11)/(b08+b11),'nbr':(b08-b12)/(b08+b12)}
                    SAT_CACHE_89.mkdir(parents=True,exist_ok=True); np.savez_compressed(cache,**tuile)
            except Exception as exc:
                tried.append((item.id,'erreur : {}'.format(exc)))
                time.sleep(3)  # pause courte : les echecs en rafale ressemblent a du throttling reseau/PC
                continue
            manquant=np.isnan(result['nbr'])
            for k in result: result[k][manquant]=tuile[k][manquant]
            items_utilises.append(item.id)
            if np.isfinite(result['nbr']).mean()>=min_coverage: break
        coverage=float(np.isfinite(result['nbr']).mean())
        tried.append((scene_date.date().isoformat(),items_utilises,round(coverage,3)))
        if not items_utilises: continue
        if coverage>=min_coverage:
            return result,scene_date
        if meilleur is None or coverage>meilleur[0]:
            meilleur=(coverage,result,scene_date,items_utilises)
    # Aucune date n'atteint le seuil ideal (90%), meme en mosaiquant plusieurs tuiles : contrairement au
    # dNBR (qui a son propre repli sur FIRMS et peut se permettre d'etre strict), nbr_last/nbr_trend_day ne
    # sont que des variables d'appoint du modele de risque - les bloquer completement casserait la
    # construction des evenements d'entrainement et des journees negatives. On se rabat donc sur la
    # MEILLEURE mosaique reellement lisible trouvee, meme sous le seuil, plutot que d'echouer.
    if meilleur is not None:
        coverage,result,scene_date,items_utilises=meilleur
        print('  ATTENTION {} : aucune date ne permet une mosaique >= {:.0%} de la foret parmi {} date(s) candidate(s) ; '
              'repli sur la meilleure trouvee ({}, couverture {:.0%}).'.format(
              date.date(),min_coverage,len(tried),items_utilises,coverage))
        return result,scene_date
    raise RuntimeError('Aucune tuile Sentinel-2 lisible antérieure à {} parmi {} date(s) candidate(s) essayée(s) (toutes en échec de lecture) : {}.'.format(
        date.date(),len(tried),tried))

def _meteo_spatialisee(date,frame,operational=False):
    """Recupere la meteo du jour (un seul appel Open-Meteo, au centroide de la foret).
    HISTORIQUE : une tentative de spatialiser temperature/VPD par cellule via un gradient thermique
    (-6.5°C/km sur altitude_m) a ete ANNULEE ici. Diagnostic : delta_temp=-0.0065*(altitude_m-altitude
    moyenne) ne depend QUE de altitude_m (deja une colonne du modele) et est IDENTIQUE chaque jour - donc
    temperature_2m/_max/_min/vpd_mean/vpd_max devenaient chacune une simple transformation monotone de
    altitude_m (correlation quasi -1, confirmee dans reparer_caches_essentiels). Avec seulement 2 incendies
    d'entrainement independants, ça a multiplie par 6 (altitude_m + 5 quasi-doublons) le poids donne a
    "l'altitude ou se trouvaient CES incendies-la" au lieu d'apporter une vraie information meteo distincte
    par cellule - ROC-AUC sur le test aveugle 2025 (vs verite dNBR) : 0.80 -> 0.12. On revient donc a UNE
    SEULE valeur meteo par jour, dupliquee sur toutes les cellules (comme avant) ; le relief est deja
    represente par altitude_m seul, sans le dupliquer sous 5 formes deguisees.
    Isolee dans sa propre fonction pour pouvoir re-appliquer ce correctif meteo sur un cache deja
    existant (nbr_last/spectral deja bon) sans refaire toute la recherche Sentinel-2, couteuse en reseau -
    voir repatcher_meteo_seule et reparer_caches_essentiels."""
    date=pd.Timestamp(date).normalize()
    center=CENTRE_WGS84   # code203 : centre calculé en Merchich puis converti (voir cellule de la grille)
    url='https://api.open-meteo.com/v1/forecast' if operational else 'https://archive-api.open-meteo.com/v1/archive'
    variables=['temperature_2m','relative_humidity_2m','wind_speed_10m','precipitation']
    response=requete_avec_reessais(url,params={'latitude':center.y,'longitude':center.x,
      'start_date':date.date().isoformat(),'end_date':date.date().isoformat(),
      'hourly':','.join(variables+['wind_direction_10m']),'timezone':'Africa/Casablanca'},timeout=120)
    response.raise_for_status(); hourly=response.json()['hourly']
    out={}
    for variable in variables:
        values=np.asarray([np.nan if x is None else x for x in hourly[variable]],float)
        out[variable]=float(np.nansum(values) if variable=='precipitation' else np.nanmean(values))
        out[variable+'_max']=float(np.nanmax(values)); out[variable+'_min']=float(np.nanmin(values))

    # Direction de vent : moyenne circulaire ponderee par la vitesse (une simple moyenne des degres
    # serait fausse pour une grandeur angulaire, ex: 350 deg et 10 deg donneraient a tort 180 deg).
    wdir=np.asarray([np.nan if x is None else x for x in hourly.get('wind_direction_10m',[])],float)
    wspd=np.asarray([np.nan if x is None else x for x in hourly['wind_speed_10m']],float)
    valid_wd=np.isfinite(wdir)&np.isfinite(wspd)&(wspd>0)
    if valid_wd.sum():
        rad=np.radians(wdir[valid_wd])
        u=np.sum(wspd[valid_wd]*np.sin(rad)); v=np.sum(wspd[valid_wd]*np.cos(rad))
        out['wind_direction_deg']=float((np.degrees(np.arctan2(u,v))+360)%360)
    else:
        out['wind_direction_deg']=np.nan

    # VPD : calcule une fois a partir de la temperature/humidite horaires du centroide (une seule valeur,
    # dupliquee sur toutes les cellules - meme logique que temperature/humidite/precipitation ci-dessus).
    temp_hourly=np.asarray([np.nan if x is None else x for x in hourly['temperature_2m']],float)
    rh_hourly=np.asarray([np.nan if x is None else x for x in hourly['relative_humidity_2m']],float)
    with np.errstate(invalid='ignore'):
        vpd_hourly=.6108*np.exp(17.27*temp_hourly/(temp_hourly+237.3))*(1-rh_hourly/100)
    out['vpd_mean']=float(np.nanmean(vpd_hourly)); out['vpd_max']=float(np.nanmax(vpd_hourly))
    return out

def repatcher_meteo_seule(date):
    """Reapplique UNIQUEMENT le correctif meteo (temperature spatialisee + VPD) sur un cache DEJA
    EXISTANT dont le spectral (nbr_last...) est deja bon - beaucoup plus rapide qu'une regeneration
    complete via _create_missing_cache (qui refait inutilement toute la recherche Sentinel-2, la partie
    couteuse en reseau, alors que seule la meteo doit changer)."""
    date=pd.Timestamp(date).normalize(); cache_path=WINDOW_CACHE_89/f'{date.date()}.parquet'
    frame=gpd.read_parquet(cache_path)
    for col,values in _meteo_spatialisee(date,frame).items():
        frame[col]=values
    frame.to_parquet(cache_path,index=False)
    return cache_path

def _create_missing_cache(date,operational=False):
    date=pd.Timestamp(date).normalize(); static89=gpd.read_parquet(SOURCE_89/'cache/static.parquet') if (SOURCE_89/'cache/static.parquet').exists() else grid[['id_cell','geometry']].copy()   # code208
    # CORRECTIF (23/09/2026) : meme classe de bug que SAT_CACHE_89/DNBR_CACHE_DIR/le cache structurel de code90
    # (cache PARTAGE entre versions du notebook, jamais revalide contre la grille actuelle) - mais repere ici
    # seulement maintenant : SOURCE_89/'cache/static.parquet' vient de code89 et n'a jamais ete regenere pour
    # Jbel Sougna, donc reste a l'ancienne taille (5542 cellules). Sans ce controle, `frame=static89.copy()`
    # ci-dessous recevait ensuite des colonnes ndvi_last/nbr_last... de longueur 6098 (calculees sur `grid`,
    # a jour) assignees dans un DataFrame de 5542 lignes -> "ValueError: Length of values (6098) does not
    # match length of index (5542)" (vu en construisant le cache journalier de l'evenement 2005). On repart
    # de `grid` (toujours a jour) si la couverture ne correspond pas, en ne recuperant de static89 que ses
    # colonnes propres (absentes de grid) - celles-ci valent NaN pour Jbel Sougna, sans consequence : elles
    # sont soit remplacees juste apres par le calcul spectral, soit par le merge avec `structural` plus tard
    # dans load_code89_day.
    if len(static89)!=len(grid) or set(grid.id_cell)-set(static89.id_cell):
        manquantes=set(grid.id_cell)-set(static89.id_cell)
        print("Information : base de la grille  ({}) ne couvre pas toute la grille actuelle ({} cellule(s) "
              "manquante(s), probablement Jbel Sougna) - reconstruction a partir de `grid`.".format(
              SOURCE_89/'cache/static.parquet',len(manquantes)))
        extra=[c for c in static89.columns if c not in grid.columns]
        static89=grid.merge(static89[['id_cell']+extra],on='id_cell',how='left',validate='one_to_one')
        static89=gpd.GeoDataFrame(static89,geometry='geometry',crs=CRS)
    # CORRECTIF (code105, 23/09/2026) : _sentinel_indices interroge uniquement Sentinel-2, qui n'existe pas
    # avant ~2015 - appelé tel quel sur une date comme 2005-09-21, il levait RuntimeError('Aucune scène
    # Sentinel-2 antérieure à 2005-09-21') avant même d'atteindre la logique dNBR/Landsat de la cellule
    # suivante. On bascule ici vers _landsat_indices (archive Landsat via Planetary Computer) pour toute date antérieure
    # à SENTINEL2_AVAILABLE_FROM, exactement comme compute_dnbr_label le fait déjà pour le label dNBR.
    _indices_fn=_landsat_indices if date<SENTINEL2_AVAILABLE_FROM else _sentinel_indices
    # NOUVEAU (code201) : l'image doit être prise entre J-AGE_MAX et J-1 (jamais le jour même, jamais une
    # image plus vieille que AGE_MAX jours). _indices_fn(date_fin, lookback=n) cherche dans [date_fin-n, date_fin].
    _age_max=AGE_MAX_LANDSAT if date<SENTINEL2_AVAILABLE_FROM else AGE_MAX_SENTINEL2
    _veille=date-pd.Timedelta(days=1)
    try:
        recent,recent_date=_indices_fn(_veille,lookback=_age_max-1)
    except RuntimeError as exc:
        print("  {} : aucune image valide entre {} et {} ({}) - variables satellite du jour laissées vides.".format(
              date.date(),(date-pd.Timedelta(days=_age_max)).date(),_veille.date(),exc))
        recent,recent_date=None,None
    older=None
    if recent is not None:
        try:
            older,older_date=_indices_fn(recent_date-pd.Timedelta(days=20),lookback=(60 if date<SENTINEL2_AVAILABLE_FROM else 40))
        except RuntimeError as exc:
            print("  {} : pas d'image plus ancienne pour la tendance ({}) - tendance laissée vide.".format(date.date(),exc))
    frame=static89.copy()
    _vide=np.full(len(frame),np.nan)
    for index in ['ndvi','ndmi','nbr']:
        last=recent[index] if recent is not None else _vide
        if older is not None:
            trend=(recent[index]-older[index])/max((recent_date-older_date).days,1)
        else:
            trend=_vide
        frame[index+'_last']=last; frame[index+'_trend_day']=trend
        frame[index+'_today_est']=last   # code202 : plus d'extrapolation (variable retirée du modèle)
    age=(date-recent_date).days if recent_date is not None else np.nan
    frame['image_age_days']=age; frame['spectral_observed_today']=0   # toujours 0 : jamais d'image le jour même

    for col,values in _meteo_spatialisee(date,frame,operational=operational).items():
        frame[col]=values

    frame['doy_sin']=np.sin(2*np.pi*date.dayofyear/365.25);frame['doy_cos']=np.cos(2*np.pi*date.dayofyear/365.25)
    frame['date']=date.date().isoformat();frame['satellite_date']=recent_date.date().isoformat() if recent_date is not None else None
    frame['cache_version']='code201'
    WINDOW_CACHE_89.mkdir(parents=True,exist_ok=True); path=WINDOW_CACHE_89/f'{date.date()}.parquet'
    frame.to_parquet(path,index=False); print('Nouveau cache créé une seule fois :',path)
    return path

def load_code89_day(date,include_target=True,allow_missing_download=False,operational=False,cache_dir=None):
    # code202 : cache_dir permet de lire un autre dossier (ex. WINDOW_CACHE_SOURCE_89 pour les journées sans feu)
    date=pd.Timestamp(date).normalize(); path=(cache_dir or WINDOW_CACHE_89)/f'{date.date()}.parquet'
    if not path.exists():
        if allow_missing_download: path=_create_missing_cache(date,operational=operational)
        else: raise FileNotFoundError(f'Cache code89 absent pour {date.date()}.')
    cached=gpd.read_parquet(path)
    keep=[c for c in cached.columns if c not in structural.columns or c=='id_cell']
    frame=structural.merge(cached[keep],on='id_cell',how='left',validate='one_to_one')
    frame=gpd.GeoDataFrame(frame,geometry='geometry',crs=CRS)
    pamf=causal_pamf(frame,date)
    for col in pamf: frame[col]=pamf[col].to_numpy()

    # code202 : NDVI/NDMI de la dernière image (avant J), plus de valeur extrapolée
    ndvi=((frame['ndvi_last']+1)/2).clip(0,1)
    dryness=(1-(frame['ndmi_last']+1)/2).clip(0,1)
    frame['inflammabilite_01']=(.45*frame.combustibilite_01+.20*ndvi+.35*dryness).clip(0,1)

    # Topoclimat : si le cache journalier date d'avant l'ajout de cette variable, on va chercher (et on
    # met en cache a part) la direction de vent du jour plutot que de recalculer tout le cache existant.
    if 'wind_direction_deg' not in frame.columns or frame['wind_direction_deg'].isna().all():
        frame['wind_direction_deg']=get_wind_direction_deg(date,operational=operational)
    frame['topoclimat_brut']=compute_topoclimat(frame['exposition_deg'].to_numpy(),frame['wind_direction_deg'].to_numpy())
    frame['topoclimat_classe_1_5']=classify_topoclimat(frame['topoclimat_brut']).to_numpy()

    if include_target: frame['fire_today']=label_fire_today(frame,date)
    return frame

In [ ]:
# ------------------------------------------------------------------
# Label affiné de l'incendie 2022 par différence de NBR (dNBR), à partir de scènes Sentinel-2
# avant/après choisies pour RÉELLEMENT couvrir la forêt (les scènes Sentinel-2 ont des bords en
# diagonale sans données selon l'orbite du satellite, qui peuvent tomber pile sur une petite zone
# d'étude comme Bouhachem selon la date). Si aucune scène valide n'est trouvée, on retombe
# automatiquement sur le label FIRMS (avec tolérance de quelques jours) pour que le notebook
# aille jusqu'au bout sans intervention manuelle.
# Les dates avant/après de 2020 et 2022 sont fixes -> le RESULTAT (label + dNBR, pas juste les bandes
# brutes) est mis en cache par paire de dates (voir DNBR_CACHE_DIR ci-dessous) : une fois résolu une
# première fois, un nouveau lancement du notebook saute entièrement la recherche Sentinel-2 pour ces
# dates-là (plusieurs minutes économisées à chaque exécution).
# ------------------------------------------------------------------
warnings.filterwarnings('ignore', message='.*DoesNotConformTo.*')

def _band_pair_cached(item,frame):
    cache=SAT_CACHE_89/f'{item.id}_b08b12.npz'
    if cache.exists():
        # CORRECTIF (23/09/2026, meme bug que _sentinel_indices) : ce cache est partage entre toutes les
        # versions du notebook et ne verifiait pas que la longueur correspondait a la grille actuelle - une
        # scene deja traitee sous l'ancien limites_Bouhachem.shp (5542 cellules) renvoyait un tableau trop
        # court des que la grille est passee a limites_forets.shp (6098 cellules avec Jbel Sougna), d'ou
        # l'erreur "boolean index did not match indexed array ... size 5542 ... 6098" observee a l'execution
        # sur 2020/2022/Derdara 2021/Tanaqoub 2019a/b. On ignore le cache si sa longueur ne correspond pas.
        saved=np.load(cache)
        if len(saved['b08'])==len(frame):
            return saved['b08'],saved['b12']
    b08=_band_means(item.assets['B08'].href,frame); b12=_band_means(item.assets['B12'].href,frame)
    SAT_CACHE_89.mkdir(parents=True,exist_ok=True); np.savez_compressed(cache,b08=b08,b12=b12)
    return b08,b12

def _resolve_burn_scene(date,direction,frame=None,search_days=15,min_coverage=0.90,max_candidates=6,max_tuiles_par_date=3):
    """Cherche, parmi plusieurs DATES candidates proches de `date`, celle qui permet de reconstituer la
    meilleure MOSAÏQUE de l'emprise de la forêt (plusieurs tuiles Sentinel-2 fusionnées, pas une seule
    scène). Retourne (liste d'ids de tuiles utilisées, nbr_array, couverture).
    CORRECTIF (23/09/2026, comme _sentinel_indices) : depuis que la forêt couvre 3 parties (Bouhachem/
    Derdara/Tanaqoub), son rectangle englobant déborde sur plusieurs dalles/tuiles Sentinel-2 - une seule
    tuile plafonnait la couverture à ~83% quelle que soit la date (confirmé en pratique : même nombre de
    cellules valides sur 2020/2022/Derdara 2021, des dates totalement différentes). On fusionne maintenant
    toutes les tuiles d'un même passage satellite (mêmes minutes d'acquisition, tuiles complémentaires)
    avant de mesurer la couverture, et on ne bloque toujours pas le calcul si aucune date n'atteint le
    seuil idéal (90%) - repli sur la meilleure mosaïque réellement lisible trouvée (avec un avertissement),
    RuntimeError seulement si vraiment aucune tuile n'a pu être lue du tout."""
    if frame is None: frame=grid
    date=pd.Timestamp(date).normalize(); aoi=list(BBOX_WGS84)   # code203 : rectangle calculé en Merchich puis converti
    stac=pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1',modifier=pc.sign_inplace)
    if direction=='before':
        window=f'{(date-pd.Timedelta(days=search_days)).date()}/{date.date()}'
    else:
        window=f'{date.date()}/{(date+pd.Timedelta(days=search_days)).date()}'
    search=stac.search(collections=['sentinel-2-l2a'],bbox=aoi,datetime=window,query={'eo:cloud_cover':{'lt':60}})
    items=list(search.items())
    if not items: raise RuntimeError('Aucune scène Sentinel-2 {} le {} (fenêtre de {} jours).'.format(direction,date.date(),search_days))

    groupes={}
    for raw in items:
        d=pd.Timestamp(raw.datetime).tz_localize(None).normalize()
        groupes.setdefault(d,[]).append(raw)
    dates_triees=sorted(groupes,key=lambda d:abs((d-date).total_seconds()))

    tried=[]; meilleur=None  # (coverage, items_utilises, nbr) de la meilleure mosaïque LISIBLE, même sous le seuil idéal
    for group_date in dates_triees[:max_candidates]:
        candidats=sorted(groupes[group_date],key=lambda it:it.properties.get('eo:cloud_cover',100))[:max_tuiles_par_date]
        b08_mosaic=np.full(len(frame),np.nan); b12_mosaic=np.full(len(frame),np.nan); items_utilises=[]
        for raw in candidats:
            item=pc.sign(raw)
            try:
                # Meme protection que _sentinel_indices : une erreur de lecture transitoire (throttling,
                # timeout /vsicurl) sur une tuile ne doit pas interrompre toute la recherche.
                b08,b12=_band_pair_cached(item,frame)
            except Exception as exc:
                tried.append((item.id,'erreur : {}'.format(exc)))
                time.sleep(3)  # pause courte : les echecs en rafale ressemblent a du throttling reseau/PC
                continue
            if str(item.properties.get('s2:processing_baseline','00.00'))>='04.00': b08=b08-1000;b12=b12-1000
            manquant=np.isnan(b08_mosaic)
            b08_mosaic[manquant]=b08[manquant]; b12_mosaic[manquant]=b12[manquant]
            items_utilises.append(item.id)
            if np.isfinite(b08_mosaic).mean()>=min_coverage: break
        coverage=float(np.isfinite(b08_mosaic).mean())
        tried.append((group_date.date().isoformat(),items_utilises,round(coverage,3)))
        if not items_utilises: continue
        with np.errstate(divide='ignore',invalid='ignore'):
            nbr=(b08_mosaic-b12_mosaic)/(b08_mosaic+b12_mosaic)
        if coverage>=min_coverage:
            return items_utilises,nbr,coverage
        if meilleur is None or coverage>meilleur[0]:
            meilleur=(coverage,items_utilises,nbr)
    if meilleur is not None:
        coverage,items_utilises,nbr=meilleur
        print('  ATTENTION : aucune date Sentinel-2 {} le {} ne permet une mosaïque >= {:.0%} de la forêt parmi {} date(s) candidate(s) ; '
              'repli sur la meilleure trouvée ({}, couverture {:.0%}).'.format(
              direction,date.date(),min_coverage,len(tried),items_utilises,coverage))
        return items_utilises,nbr,coverage
    raise RuntimeError('Aucune tuile Sentinel-2 {} le {} lisible parmi {} date(s) candidate(s) essayée(s) (toutes en échec de lecture) : {}.'.format(
        direction,date.date(),len(tried),tried))

# ------------------------------------------------------------------
# LANDSAT VIA PLANETARY COMPUTER, code105 (23/09/2026, v2) : remplace le client M2M (USGS EarthExplorer)
# utilise dans une premiere version de ce notebook. Le M2M avait ete choisi a la demande explicite de
# l'utilisateur pour interroger directement earthexplorer.usgs.gov, mais impose des identifiants USGS EROS
# ET depend de la disponibilite du service M2M lui-meme (ex. maintenance planifiee du 23/09/2026, qui a
# bloque scene-search puis login-token pendant plusieurs heures - voir _m2m_call dans l'historique de ce
# notebook si besoin de comparer). Microsoft Planetary Computer heberge la meme donnee officielle USGS -
# Landsat Collection 2 Niveau 2 (reflectance de surface), collection "landsat-c2-l2", couverture
# 1982-aujourd'hui (Landsat 4/5 TM, 7 ETM+, 8/9 OLI-TIRS) - en lecture directe des bandes via /vsicurl
# (comme Sentinel-2 dans _sentinel_indices/_resolve_burn_scene ci-dessus), sans aucun identifiant ni
# telechargement d'archive .tar : plus simple, plus rapide, et pas de fenetre de maintenance a attendre.
# Memes coefficients de conversion DN -> reflectance (LANDSAT_SR_SCALE/OFFSET, cellule de config) car il
# s'agit du meme produit Collection 2 Niveau 2, quelle que soit la voie d'acces. Noms des bandes utiles
# dans cette collection : red, nir08, swir16, swir22 (au lieu de SR_B3/4/5/7 cote M2M/EarthExplorer - meme
# contenu, nom d'actif STAC different).
# ------------------------------------------------------------------

def _band_pair_cached_landsat(item,frame):
    cache=SAT_CACHE_89/f'{item.id}_nirswir22.npz'
    if cache.exists():
        # Meme correctif que _band_pair_cached (cache partage entre versions du notebook, verification de
        # longueur contre la grille actuelle).
        saved=np.load(cache)
        if len(saved['nir'])==len(frame):
            return saved['nir'],saved['swir22']
    nir=_band_means(item.assets['nir08'].href,frame); swir22=_band_means(item.assets['swir22'].href,frame)
    SAT_CACHE_89.mkdir(parents=True,exist_ok=True); np.savez_compressed(cache,nir=nir,swir22=swir22)
    return nir,swir22

def _resolve_burn_scene_landsat(date,direction,frame=None,search_days=45,min_coverage=0.90,max_candidates=10,max_tuiles_par_date=2):
    """Equivalent de _resolve_burn_scene (Sentinel-2), mais sur l'archive Landsat via Planetary Computer -
    seul moyen de calculer un dNBR reel pour les incendies documentes ANTERIEURS a Sentinel-2 (2015) :
    Tanaqoub-Bouhachem 2005, Derdara-Jbel Sougna 2009, complexe Derdara-Bouhachem/Mont Bouhachem 2009.
    Landsat 5 TM (jusqu'a fin 2011) et Landsat 7 ETM+ couvrent ces dates ; resolution 30 m (contre 10-20 m
    pour Sentinel-2), donc verification un peu moins fine mais reelle - bien mieux qu'un simple point
    FIRMS/MODIS (1 km, souvent aveugle aux petits feux de cette epoque).
    Reflectance de surface obtenue via les coefficients standards USGS Collection 2 Niveau 2
    (LANDSAT_SR_SCALE/OFFSET) : necessaire ici car, contrairement au NBR Sentinel-2 (ou seul le RAPPORT des
    comptes numeriques bruts compte, l'offset de calibration s'annulant a peu pres dans le rapport), l'offset
    Landsat (-0.2) ne s'annule PAS dans le rapport NBR et fausserait le resultat s'il etait applique sur des
    comptes numeriques bruts non convertis.
    Fenetre de recherche plus large par defaut (45 jours contre 15 pour Sentinel-2) : le revisit Landsat
    (16 jours par satellite) et la disponibilite/qualite des archives sur le Maroc dans les annees 2000 sont
    plus rares. Retourne (liste d'ids de scenes utilisees, nbr_array, couverture), meme signature que
    _resolve_burn_scene, pour que compute_dnbr_label puisse appeler l'un ou l'autre indifferemment."""
    if frame is None: frame=grid
    date=pd.Timestamp(date).normalize(); aoi=list(BBOX_WGS84)   # code203 : rectangle calculé en Merchich puis converti
    stac=pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1',modifier=pc.sign_inplace)
    if direction=='before':
        window=f'{(date-pd.Timedelta(days=search_days)).date()}/{date.date()}'
    else:
        window=f'{date.date()}/{(date+pd.Timedelta(days=search_days)).date()}'
    search=stac.search(collections=['landsat-c2-l2'],bbox=aoi,datetime=window,query={'eo:cloud_cover':{'lt':70}})
    items=list(search.items())
    if not items: raise RuntimeError('Aucune scène Landsat (Planetary Computer) {} le {} (fenêtre de {} jours).'.format(direction,date.date(),search_days))

    groupes={}
    for raw in items:
        d=pd.Timestamp(raw.datetime).tz_localize(None).normalize()
        groupes.setdefault(d,[]).append(raw)
    dates_triees=sorted(groupes,key=lambda d:abs((d-date).total_seconds()))

    tried=[]; meilleur=None  # (coverage, items_utilises, nbr) de la meilleure mosaïque LISIBLE, même sous le seuil idéal
    for group_date in dates_triees[:max_candidates]:
        candidats=sorted(groupes[group_date],key=lambda it:it.properties.get('eo:cloud_cover',100))[:max_tuiles_par_date]
        nir_mosaic=np.full(len(frame),np.nan); swir_mosaic=np.full(len(frame),np.nan); items_utilises=[]
        for raw in candidats:
            item=pc.sign(raw)
            try:
                # Meme protection que _resolve_burn_scene : une erreur de lecture transitoire (throttling,
                # timeout /vsicurl) sur une tuile ne doit pas interrompre toute la recherche.
                nir_dn,swir_dn=_band_pair_cached_landsat(item,frame)
            except Exception as exc:
                tried.append((item.id,'erreur : {}'.format(exc)))
                time.sleep(3)
                continue
            nir=nir_dn*LANDSAT_SR_SCALE+LANDSAT_SR_OFFSET; swir=swir_dn*LANDSAT_SR_SCALE+LANDSAT_SR_OFFSET
            manquant=np.isnan(nir_mosaic)
            nir_mosaic[manquant]=nir[manquant]; swir_mosaic[manquant]=swir[manquant]
            items_utilises.append(item.id)
            if np.isfinite(nir_mosaic).mean()>=min_coverage: break
        coverage=float(np.isfinite(nir_mosaic).mean())
        tried.append((group_date.date().isoformat(),items_utilises,round(coverage,3)))
        if not items_utilises: continue
        with np.errstate(divide='ignore',invalid='ignore'):
            nbr=(nir_mosaic-swir_mosaic)/(nir_mosaic+swir_mosaic)
        if coverage>=min_coverage:
            return items_utilises,nbr,coverage
        if meilleur is None or coverage>meilleur[0]:
            meilleur=(coverage,items_utilises,nbr)
    if meilleur is not None:
        coverage,items_utilises,nbr=meilleur
        print('  ATTENTION (Landsat/Planetary Computer) : aucune date {} le {} ne permet une mosaïque >= {:.0%} de la forêt parmi {} date(s) candidate(s) ; '
              'repli sur la meilleure trouvée ({}, couverture {:.0%}).'.format(
              direction,date.date(),min_coverage,len(tried),items_utilises,coverage))
        return items_utilises,nbr,coverage
    raise RuntimeError('Aucune scène Landsat {} le {} lisible parmi {} date(s) candidate(s) essayée(s) (toutes en échec) : {}.'.format(
        direction,date.date(),len(tried),tried))

def _band_quad_cached_landsat(item,frame):
    cache=SAT_CACHE_89/f'{item.id}_redswir_nirswir22.npz'
    if cache.exists():
        # Meme correctif que _band_pair_cached (verification de longueur contre la grille actuelle).
        saved=np.load(cache)
        if len(saved['red'])==len(frame):
            return saved['red'],saved['nir'],saved['swir16'],saved['swir22']
    red=_band_means(item.assets['red'].href,frame); nir=_band_means(item.assets['nir08'].href,frame)
    swir16=_band_means(item.assets['swir16'].href,frame); swir22=_band_means(item.assets['swir22'].href,frame)
    SAT_CACHE_89.mkdir(parents=True,exist_ok=True)
    np.savez_compressed(cache,red=red,nir=nir,swir16=swir16,swir22=swir22)
    return red,nir,swir16,swir22

def _landsat_indices(date,lookback=60,min_coverage=0.90,max_candidates=10,max_tuiles_par_date=2):
    """Equivalent de _sentinel_indices (construction du cache journalier ndvi/ndmi/nbr utilise comme
    variables du modele de risque, PAS le dNBR de verite terrain), mais sur l'archive Landsat via Planetary
    Computer - utilise pour toute date anterieure a SENTINEL2_AVAILABLE_FROM (voir _create_missing_cache).
    lookback plus large par defaut (60 jours) que pour Sentinel-2 (30 jours) : archives Landsat des annees
    2000 plus rares sur le Maroc."""
    date=pd.Timestamp(date).normalize(); aoi=list(BBOX_WGS84)   # code203 : rectangle calculé en Merchich puis converti
    stac=pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1',modifier=pc.sign_inplace)
    search=stac.search(collections=['landsat-c2-l2'],bbox=aoi,
      datetime=f'{(date-pd.Timedelta(days=lookback)).date()}/{date.date()}',
      query={'eo:cloud_cover':{'lt':70}})
    items=list(search.items())
    if not items: raise RuntimeError(f'Aucune scène Landsat (Planetary Computer) antérieure à {date.date()}')

    groupes={}
    for raw in items:
        d=pd.Timestamp(raw.datetime).tz_localize(None).normalize()
        groupes.setdefault(d,[]).append(raw)
    dates_triees=sorted(groupes,reverse=True)  # les plus récentes d'abord, comme _sentinel_indices

    tried=[]; meilleur=None
    for scene_date in dates_triees[:max_candidates]:
        candidats=sorted(groupes[scene_date],key=lambda it:it.properties.get('eo:cloud_cover',100))[:max_tuiles_par_date]
        result={'ndvi':np.full(len(grid),np.nan),'ndmi':np.full(len(grid),np.nan),'nbr':np.full(len(grid),np.nan)}
        items_utilises=[]
        for raw in candidats:
            item=pc.sign(raw)
            try:
                red_dn,nir_dn,swir16_dn,swir22_dn=_band_quad_cached_landsat(item,grid)
            except Exception as exc:
                tried.append((item.id,'erreur : {}'.format(exc))); time.sleep(3); continue
            red=red_dn*LANDSAT_SR_SCALE+LANDSAT_SR_OFFSET; nir=nir_dn*LANDSAT_SR_SCALE+LANDSAT_SR_OFFSET
            swir16=swir16_dn*LANDSAT_SR_SCALE+LANDSAT_SR_OFFSET; swir22=swir22_dn*LANDSAT_SR_SCALE+LANDSAT_SR_OFFSET
            with np.errstate(divide='ignore',invalid='ignore'):
                tuile={'ndvi':(nir-red)/(nir+red),'ndmi':(nir-swir16)/(nir+swir16),'nbr':(nir-swir22)/(nir+swir22)}
            manquant=np.isnan(result['nbr'])
            for k in result: result[k][manquant]=tuile[k][manquant]
            items_utilises.append(item.id)
            if np.isfinite(result['nbr']).mean()>=min_coverage: break
        coverage=float(np.isfinite(result['nbr']).mean())
        tried.append((scene_date.date().isoformat(),items_utilises,round(coverage,3)))
        if not items_utilises: continue
        if coverage>=min_coverage:
            return result,scene_date
        if meilleur is None or coverage>meilleur[0]:
            meilleur=(coverage,result,scene_date,items_utilises)
    if meilleur is not None:
        coverage,result,scene_date,items_utilises=meilleur
        print('  ATTENTION (Landsat/Planetary Computer) {} : aucune date ne permet une mosaique >= {:.0%} de la foret parmi {} date(s) candidate(s) ; '
              'repli sur la meilleure trouvee ({}, couverture {:.0%}).'.format(
              date.date(),min_coverage,len(tried),items_utilises,coverage))
        return result,scene_date
    raise RuntimeError('Aucune scène Landsat lisible antérieure à {} parmi {} date(s) candidate(s) essayée(s) (toutes en échec) : {}.'.format(
        date.date(),len(tried),tried))

DNBR_CACHE_DIR=PREP/'cache_dnbr'   # code202 : dNBR déjà calculés par code201   # cache du RESULTAT final (label + dNBR), pas juste des bandes brutes :
                                    # les dates avant/après de 2020 et 2022 ne changent jamais, donc une fois la
                                    # meilleure scène Sentinel-2 identifiée il est inutile de refaire toute la
                                    # recherche STAC (plusieurs minutes) à chaque exécution du notebook.

def compute_dnbr_label(date_before,date_after,threshold=BURN_DNBR_THRESHOLD,min_positive_cells=5,use_cache=True,
                       search_days_before=None,search_days_after=None):
    """dNBR entre l'image la plus proche AVANT date_before et l'image la plus proche APRÈS date_after.
    NOUVEAU (code106) : search_days_before / search_days_after limitent la fenêtre de recherche des images,
    pour qu'un feu voisin dans le temps (ex. 2009a / 2009bc, ou les 2 feux de 2025) ne se retrouve pas
    dans le label de l'autre. None = fenêtre par défaut du capteur (15 jours Sentinel-2, 45 jours Landsat)."""
    date_before=pd.Timestamp(date_before).normalize(); date_after=pd.Timestamp(date_after).normalize()
    capteur='landsat' if date_after<SENTINEL2_AVAILABLE_FROM else 'sentinel2'
    resolve_scene=_resolve_burn_scene_landsat if capteur=='landsat' else _resolve_burn_scene
    suffixe='_landsat' if capteur=='landsat' else ''
    if search_days_before or search_days_after:
        suffixe+='_sd{}-{}'.format(search_days_before or 'def',search_days_after or 'def')
    cache_path=DNBR_CACHE_DIR/f'{date_before.date()}_{date_after.date()}{suffixe}.npz'
    if use_cache and cache_path.exists():
        saved=np.load(cache_path)
        if len(saved['label'])!=len(grid):
            print("  Cache dNBR trouvé pour ces dates mais avec une grille différente ({} cellules au lieu de {}) - recalcul.".format(len(saved['label']),len(grid)))
        elif float(saved['threshold_demande'])==float(threshold):
            label=saved['label'].astype('uint8'); dnbr=saved['dnbr']
            print("dNBR {} -> {} ({}) : résultat repris du cache (tuiles avant {}, couverture {:.0%} ; tuiles après {}, "
                  "couverture {:.0%}) - aucune recherche relancée.".format(
                  date_before.date(),date_after.date(),capteur,np.atleast_1d(saved['item_before_id']).tolist(),float(saved['cov_before']),
                  np.atleast_1d(saved['item_after_id']).tolist(),float(saved['cov_after'])))
            print("  cellules valides : {} / {}".format(int(saved['n_valid']),int(saved['n_total'])))
            print("  -> {} cellules au-dessus du seuil dNBR (seuil utilisé {:.3f}), avant affinage".format(int(label.sum()),float(saved['threshold_utilise'])))
            print("  (pour forcer un nouveau calcul : supprime {} ou appelle compute_dnbr_label(...,use_cache=False))".format(cache_path))
            return label,dnbr
        else:
            print("  Cache dNBR trouvé pour ces dates mais avec un autre seuil demandé ({} au lieu de {}) - recalcul.".format(threshold,float(saved['threshold_demande'])))
    kw_before={'search_days':search_days_before} if search_days_before else {}
    kw_after={'search_days':search_days_after} if search_days_after else {}
    items_before,nbr_before,cov_before=resolve_scene(date_before,'before',**kw_before)
    items_after,nbr_after,cov_after=resolve_scene(date_after,'after',**kw_after)
    print("dNBR {} -> {} ({}) : {} tuile(s) avant {} (couverture {:.0%}), {} tuile(s) après {} (couverture {:.0%})".format(
          date_before.date(),date_after.date(),capteur,len(items_before),items_before,cov_before,len(items_after),items_after,cov_after))
    dnbr=nbr_before-nbr_after
    valid=np.isfinite(dnbr)
    print("  cellules valides : {} / {}".format(int(valid.sum()),len(dnbr)))
    if valid.sum()<100:
        raise RuntimeError("dNBR quasiment entièrement invalide ({} cellule(s) valide(s) sur {}) malgré la vérification de couverture.".format(int(valid.sum()),len(dnbr)))
    print("  statistiques dNBR : min={:.3f} p50={:.3f} p90={:.3f} p99={:.3f} max={:.3f}".format(
          np.nanmin(dnbr),np.nanpercentile(dnbr,50),np.nanpercentile(dnbr,90),
          np.nanpercentile(dnbr,99),np.nanmax(dnbr)))
    used_threshold=threshold
    label=(dnbr>=used_threshold).astype('uint8')
    if label.sum()<min_positive_cells:
        used_threshold=float(np.nanpercentile(dnbr,99))
        print("  ATTENTION : le seuil fixe {} ne détecte que {} cellule(s) brûlée(s).".format(threshold,int(label.sum())))
        print("  Repli sur un seuil adaptatif (percentile 99 = {:.3f}) - vérifie visuellement la carte du dNBR, ce repli n'est pas fiable en soi.".format(used_threshold))
        label=(dnbr>=used_threshold).astype('uint8')
    print("  -> {} cellules au-dessus du seuil dNBR (seuil utilisé {:.3f}), avant affinage".format(int(label.sum()),used_threshold))
    if use_cache:
        DNBR_CACHE_DIR.mkdir(parents=True,exist_ok=True)
        np.savez_compressed(cache_path,label=label,dnbr=dnbr,threshold_demande=threshold,threshold_utilise=used_threshold,
            item_before_id=np.array(items_before),item_after_id=np.array(items_after),cov_before=cov_before,cov_after=cov_after,
            n_valid=int(valid.sum()),n_total=len(dnbr))
        print("  (résultat mis en cache dans {} - les prochains lancements sauteront la recherche pour ces dates)".format(cache_path))
    return label,dnbr

def affiner_label_dnbr(label,dnbr,debut,fin,surface_officielle_ha=None,nom=''):
    """NOUVEAU (code106) : affine un label dNBR brut en 2 étapes.
    (a) Masque FIRMS : une cellule ne peut être brûlée que si elle est à moins de FIRMS_MASK_BUFFER_M d'une
        détection FIRMS du feu (fenêtre debut-1 jour -> fin+2 jours). Sans détection FIRMS (ex. 2005), pas de
        masque : on passe directement à l'étape (b).
    (b) Contrôle de surface : si le label dépasse FACTEUR_SURFACE_MAX x la surface officielle, on garde
        seulement les cellules au dNBR le plus fort jusqu'à atteindre cette surface officielle.
    Le dessèchement saisonnier donne des dNBR faibles et étalés ; une vraie brûlure donne des dNBR forts et
    regroupés près des détections thermiques : ces 2 étapes retirent surtout le premier."""
    label=np.asarray(label,dtype='uint8').copy(); dnbr=np.asarray(dnbr,dtype=float)
    debut=pd.Timestamp(debut).normalize(); fin=pd.Timestamp(fin).normalize()
    n_brut=int(label.sum())
    fenetre=firms[(firms.date>=debut-pd.Timedelta(days=1))&(firms.date<=fin+pd.Timedelta(days=2))]
    if len(fenetre):
        zone=fenetre.geometry.union_all().buffer(FIRMS_MASK_BUFFER_M)
        proche=grid.geometry.intersects(zone).to_numpy()
        label=(label.astype(bool)&proche).astype('uint8')
        info_masque='{} détection(s) FIRMS {} -> {}, rayon {} m'.format(len(fenetre),(debut-pd.Timedelta(days=1)).date(),
                                                                      (fin+pd.Timedelta(days=2)).date(),FIRMS_MASK_BUFFER_M)
    else:
        info_masque='aucune détection FIRMS -> pas de masque'
    n_masque=int(label.sum())
    info_surface='surface officielle inconnue -> pas de contrôle'
    if surface_officielle_ha:
        n_attendu=max(int(round(surface_officielle_ha/CELL_HA)),1)
        if label.sum()>FACTEUR_SURFACE_MAX*n_attendu:
            candidats=np.where(label==1)[0]
            garder=candidats[np.argsort(-dnbr[candidats])][:n_attendu]
            label=np.zeros_like(label); label[garder]=1
            info_surface='label > {:.0f} x surface officielle ({} ha = {} cellules) -> on garde les {} cellules au dNBR le plus fort'.format(
                FACTEUR_SURFACE_MAX,surface_officielle_ha,n_attendu,n_attendu)
        else:
            info_surface='cohérent avec la surface officielle ({} ha = {} cellules)'.format(surface_officielle_ha,n_attendu)
    n_final=int(label.sum())
    print('Label affiné {} : {} cellules (dNBR brut, ~{:.0f} ha) -> {} après masque FIRMS [{}] -> {} au final (~{:.0f} ha) [{}].'.format(
          nom,n_brut,n_brut*CELL_HA,n_masque,info_masque,n_final,n_final*CELL_HA,info_surface))
    return label

def label_fire_window(frame,center_date,window_days=FIRE_2020_WINDOW_DAYS):
    """Comme label_fire_today mais tolère un décalage de quelques jours autour de la date officielle de
    l'incendie - utile quand la détection FIRMS ne tombe pas exactement le jour renseigné."""
    center_date=pd.Timestamp(center_date).normalize()
    window=firms[(firms.date>=center_date-pd.Timedelta(days=window_days))&(firms.date<=center_date+pd.Timedelta(days=window_days))]
    if window.empty: return np.zeros(len(frame),dtype='uint8')
    return frame.geometry.intersects(window.geometry.union_all().buffer(FIRE_RADIUS)).astype('uint8').to_numpy()

# ------------------------------------------------------------------
# NOUVEAU (code106) : les 8 labels dNBR sont calculés dans une seule boucle à partir de ce tableau.
#   avant / apres : dates de référence des images (on cherche l'image valide la plus proche AVANT `avant`
#                   et la plus proche APRÈS `apres`) - voir cellule 2, rapprochées du feu dans code106 ;
#   debut / fin   : départ et maîtrise du feu (fenêtre du masque FIRMS) ;
#   surface_ha    : surface officielle connue (None si inconnue) ;
#   sd_avant/sd_apres : fenêtre de recherche maximale des images (None = défaut du capteur).
# Les noms de variables de code105 (dnbr_label_2020, LABEL_2020_SOURCE, ...) sont conservés pour que les
# cellules suivantes fonctionnent sans changement.
# ------------------------------------------------------------------
EVENEMENTS_DNBR={
 '2020':          dict(avant=FIRE_2020_BEFORE,   apres=FIRE_2020_AFTER,   debut=FIRE_2020_DATE,      fin='2020-09-01',surface_ha=1100,sd_avant=None,sd_apres=None,
                       var='2020',      source='LABEL_2020_SOURCE',        capteur='dNBR'),
 '2022':          dict(avant=FIRE_2022_BEFORE,   apres=FIRE_2022_AFTER,   debut=FIRE_2022_EVENT_DATE,fin='2022-08-02',surface_ha=7500,sd_avant=None,sd_apres=None,
                       var='2022',      source='LABEL_2022_SOURCE',        capteur='dNBR'),
 'derdara_2021':  dict(avant=DERDARA_2021_BEFORE,apres=DERDARA_2021_AFTER,debut=DERDARA_2021_DATE,   fin='2021-08-18',surface_ha=1100,sd_avant=None,sd_apres=None,
                       var='derdara2021',source='LABEL_DERDARA2021_SOURCE',capteur='dNBR'),
 'tanaqoub_2019a':dict(avant=FIRE_2019A_BEFORE,  apres=FIRE_2019A_AFTER,  debut=FIRE_2019A_DATE,     fin='2019-08-14',surface_ha=None,sd_avant=None,sd_apres=None,
                       var='2019a',     source='LABEL_2019A_SOURCE',       capteur='dNBR'),
 'tanaqoub_2019b':dict(avant=FIRE_2019B_BEFORE,  apres=FIRE_2019B_AFTER,  debut=FIRE_2019B_DATE,     fin='2019-09-14',surface_ha=None,sd_avant=None,sd_apres=None,
                       var='2019b',     source='LABEL_2019B_SOURCE',       capteur='dNBR'),
 '2005':          dict(avant=FIRE_2005_BEFORE,   apres=FIRE_2005_AFTER,   debut=FIRE_2005_DATE,      fin='2005-09-22',surface_ha=214, sd_avant=None,sd_apres=None,
                       var='2005',      source='LABEL_2005_SOURCE',        capteur='dNBR (Landsat)'),
 # 2009a et 2009bc ne sont séparés que de 16 jours : on limite les fenêtres à 14 jours pour que l'image
 # "après" de 2009a ne soit pas postérieure au 11/08, et que l'image "avant" de 2009bc ne soit pas antérieure au 27/07.
 '2009a':         dict(avant=FIRE_2009A_BEFORE,  apres=FIRE_2009A_AFTER,  debut=FIRE_2009A_DATE,     fin='2009-07-27',surface_ha=149, sd_avant=None,sd_apres=14,
                       var='2009a',     source='LABEL_2009A_SOURCE',       capteur='dNBR (Landsat)'),
 '2009bc':        dict(avant=FIRE_2009BC_BEFORE, apres=FIRE_2009BC_AFTER, debut=FIRE_2009BC_DATE,    fin='2009-08-12',surface_ha=188, sd_avant=14,  sd_apres=None,
                       var='2009bc',    source='LABEL_2009BC_SOURCE',      capteur='dNBR (Landsat)'),
}

# Validité du dNBR par incendie (True = la cellule a une valeur dNBR exploitable). Sert à retirer de
# l'entraînement et de l'évaluation les cellules sans vérité (nuages, bord d'image) - cellule 21.
VALIDITE_DNBR={}
for _nom,_cfg in EVENEMENTS_DNBR.items():
    _fenetre=firms[(firms.date>=_cfg['avant'])&(firms.date<=_cfg['apres'])]
    print('-'*100)
    print('Incendie {} : images {} -> {} | dates FIRMS dans la fenêtre : {}'.format(
          _nom,_cfg['avant'].date(),_cfg['apres'].date(),sorted(_fenetre.date.dt.date.unique()) if len(_fenetre) else 'aucune'))
    try:
        _label_brut,_valeurs=compute_dnbr_label(_cfg['avant'],_cfg['apres'],
                                                 search_days_before=_cfg['sd_avant'],search_days_after=_cfg['sd_apres'])
        _label=affiner_label_dnbr(_label_brut,_valeurs,_cfg['debut'],_cfg['fin'],_cfg['surface_ha'],nom=_nom)
        globals()['dnbr_label_'+_cfg['var']]=_label
        globals()['dnbr_values_'+_cfg['var']]=_valeurs
        globals()[_cfg['source']]=_cfg['capteur']+' affiné'
        VALIDITE_DNBR[_nom]=np.isfinite(_valeurs)
    except Exception as exc:
        if isinstance(exc,(ConnectionError,OSError)) or 'Max retries exceeded' in str(exc) or 'NameResolution' in str(exc):
            # code201 (correctif) : une coupure réseau n'est pas une absence d'image - on arrête au lieu de
            # remplacer le label dNBR par FIRMS (ce qui fausserait tout l'entraînement sans prévenir).
            raise ConnectionError("Connexion perdue pendant le calcul du dNBR de {} : {}. Rétablis la connexion "
                                  "et relance cette cellule.".format(_nom,exc)) from None
        print('dNBR indisponible pour {} ({}).'.format(_nom,exc))
        print("-> repli automatique sur le label FIRMS (avec tolérance de quelques jours) pour ne pas bloquer le notebook.")
        globals()['dnbr_label_'+_cfg['var']]=None
        globals()[_cfg['source']]='FIRMS (repli)'
        VALIDITE_DNBR[_nom]=None

print('='*100)
print('Bilan des labels affinés (surface estimée = cellules x {:.2f} ha) :'.format(CELL_HA))
display(pd.DataFrame([{'incendie':_nom,'source':globals()[_cfg['source']],
     'cellules_brulees':(int(globals()['dnbr_label_'+_cfg['var']].sum()) if globals()['dnbr_label_'+_cfg['var']] is not None else np.nan),
     'surface_estimee_ha':(globals()['dnbr_label_'+_cfg['var']].sum()*CELL_HA if globals()['dnbr_label_'+_cfg['var']] is not None else np.nan),
     'surface_officielle_ha':_cfg['surface_ha'],
     'cellules_valides':(int(VALIDITE_DNBR[_nom].sum()) if VALIDITE_DNBR[_nom] is not None else np.nan)}
     for _nom,_cfg in EVENEMENTS_DNBR.items()]))


## Modele de repli climat -> indices spectraux (pour les dates sans Sentinel-2, ex. incendie 2009)

Pour une date sans aucune scene Sentinel-2 disponible (le satellite n'existe que depuis 2015 ; toute
date anterieure, comme l'incendie de 2009, n'a donc pas de NBR/NDVI/NDMI observe), l'extrapolation par
tendance utilisee ailleurs dans ce notebook (`_sentinel_indices`, qui projette depuis les 2 scenes les
plus proches) n'a pas de sens sur un ecart de plusieurs annees.

A la place, on calibre ici une relation statistique entre le climat (pluie cumulee sur 30/60/90 jours,
temperature) et l'indice spectral observe, en utilisant uniquement les jours ou une vraie observation
Sentinel-2 existe deja dans le cache local (2017-2025) - donc sans aucune extrapolation temporelle. On
applique ensuite cette relation a une date cible sans satellite, en ne lui donnant que sa meteo
(Open-Meteo, qui remonte tres loin dans le passe).

Cette approche est etayee par la litterature sur des ecosystemes comparables :
- **Tahiri et al. (2014)**, *bassin du Tahaddart (Rif occidental, tres proche de Bouhachem)* : relation
  lineaire NDVI ~ pluviometrie par station (ex. NDVI = 0,0027 x P + 0,2177), r ~ 0,73 en moyenne
  (0,61 a 0,85 selon les stations).
- **Soares et al. (2022)**, *chenaie-liege mediterraneenne (Portugal)* : regressions par type de
  vegetation (chene-liege, matorral/Cistus, strate herbacee) utilisant des cumuls de pluie et de
  temperature a 30/60/90 jours - structure de lag directement reprise ci-dessous. R2 variable selon le
  type (0,33 pour le chene-liege adulte, jusqu'a 0,68 pour la strate herbacee, plus sensible au climat
  court terme).

Ces deux references servent de point de comparaison (ordre de grandeur du coefficient, R2 attendu) pour
juger si le modele calibre ci-dessous sur les donnees de Bouhachem est plausible - pas comme formule a
appliquer telle quelle, un cumul de pluie a Tahaddart ou au Portugal n'ayant pas la meme echelle qu'a
Bouhachem.

**Important** : ce modele reste une estimation statistique, pas une observation. Toute date qui s'appuie
dessus doit etre clairement marquee (`spectral_source='proxy_climatique'`) et son R2 de validation doit
etre regarde avant de faire confiance a un evenement d'entrainement construit ainsi.

In [ ]:
# ------------------------------------------------------------------
# Covariables climatiques a 30/60/90 jours (cumul de pluie, temperature), mises en cache a part
# (comme la direction de vent) pour ne jamais toucher aux caches existants de code89/code90 et eviter
# de re-interroger Open-Meteo a chaque relecture d'une meme date.
# ------------------------------------------------------------------
CLIMATE_LAG_CACHE = PREP/'covariables_climat_lag_cache.parquet'

def _load_climate_lag_cache():
    if CLIMATE_LAG_CACHE.exists():
        return pd.read_parquet(CLIMATE_LAG_CACHE).set_index('date').to_dict('index')
    return {}

def _save_climate_lag_cache(cache_dict):
    CLIMATE_LAG_CACHE.parent.mkdir(parents=True,exist_ok=True)
    rows=[{'date':k,**v} for k,v in cache_dict.items()]
    pd.DataFrame(rows).to_parquet(CLIMATE_LAG_CACHE,index=False)

_CLIMATE_LAG_MEMO=_load_climate_lag_cache()

def climate_lag_features(date):
    """Cumuls de precipitation et moyennes de temperature sur les 30/60/90 jours precedant `date`,
    selon la structure de lag de Soares et al. (2022). Fonctionne pour n'importe quelle date passee,
    y compris tres anciennes (Open-Meteo archive remonte a 1940)."""
    date=pd.Timestamp(date).normalize(); key=date.date().isoformat()
    if key in _CLIMATE_LAG_MEMO: return _CLIMATE_LAG_MEMO[key]
    start=date-pd.Timedelta(days=91); end=date-pd.Timedelta(days=1)
    center=CENTRE_WGS84   # code203 : centre calculé en Merchich puis converti (voir cellule de la grille)
    response=requete_avec_reessais('https://archive-api.open-meteo.com/v1/archive',params={
      'latitude':center.y,'longitude':center.x,
      'start_date':start.date().isoformat(),'end_date':end.date().isoformat(),
      'daily':'precipitation_sum,temperature_2m_mean,temperature_2m_min,temperature_2m_max',
      'timezone':'Africa/Casablanca'},timeout=120)
    response.raise_for_status(); daily=response.json()['daily']
    dates=pd.to_datetime(daily['time'])
    precip=np.asarray([np.nan if x is None else x for x in daily['precipitation_sum']],float)
    tmean=np.asarray([np.nan if x is None else x for x in daily['temperature_2m_mean']],float)
    tmin=np.asarray([np.nan if x is None else x for x in daily['temperature_2m_min']],float)
    tmax=np.asarray([np.nan if x is None else x for x in daily['temperature_2m_max']],float)
    def cumul(days):
        mask=dates>=(date-pd.Timedelta(days=days)); return float(np.nansum(precip[mask])) if mask.any() else np.nan
    def moy(arr,days):
        mask=dates>=(date-pd.Timedelta(days=days)); return float(np.nanmean(arr[mask])) if mask.any() else np.nan
    feats={'precip_cum_30j':cumul(30),'precip_cum_60j':cumul(60),'precip_cum_90j':cumul(90),
           'temp_moy_30j':moy(tmean,30),'temp_min_moy_30j':moy(tmin,30),'temp_max_moy_30j':moy(tmax,30),
           'doy_sin':float(np.sin(2*np.pi*date.dayofyear/365.25)),'doy_cos':float(np.cos(2*np.pi*date.dayofyear/365.25))}
    _CLIMATE_LAG_MEMO[key]=feats; _save_climate_lag_cache(_CLIMATE_LAG_MEMO)
    return feats

In [ ]:
# ------------------------------------------------------------------
# Construction du jeu d'entrainement (climat -> indice spectral observe) a partir de TOUTES les
# journees deja en cache ou une vraie scene Sentinel-2 a ete lue (spectral_observed_today == 1) - donc
# sans aucune extrapolation, uniquement des observations reelles couplees a leur propre climat.
# ------------------------------------------------------------------
PROXY_DATASET_CACHE = PREP/'jeu_entrainement_proxy_spectral.parquet'
PROXY_MODEL_PATH = PREP/'modele_proxy_spectral.joblib'
PROXY_STATIC_FEATURES = ['altitude_m','pente_pct','exposition_deg','combustibilite_01']
PROXY_CLIMATE_FEATURES = ['precip_cum_30j','precip_cum_60j','precip_cum_90j',
                          'temp_moy_30j','temp_min_moy_30j','temp_max_moy_30j','doy_sin','doy_cos']

def build_spectral_proxy_dataset(force_rebuild=False,repair_stale=False,seuil_cache_casse=0.05):
    # repair_stale=False par defaut : regenerer un jour casse implique de re-parcourir jusqu'a 6
    # scenes candidates x plusieurs bandes x 2 appels (recent/older) par jour - sur 86 jours en
    # cache, ca peut prendre plus d'une heure et beaucoup de dates anciennes (2017-2018) echouent
    # de toute facon car aucune scene disponible ne couvre suffisamment la foret (limite reelle
    # de disponibilite, pas un bug). Utiliser reparer_caches_essentiels() ci-dessous pour ne
    # reparer que les quelques jours reellement utilises par le modele (rapide et cible).
    if PROXY_DATASET_CACHE.exists() and not force_rebuild:
        return pd.read_parquet(PROXY_DATASET_CACHE)
    rows=[]
    cached_paths=sorted(WINDOW_CACHE_SOURCE_89.glob('*.parquet'))   # code202 : ancien cache code89
    print('Examen de {} jour(s) en cache pour trouver des observations Sentinel-2 reelles...'.format(len(cached_paths)))
    for cache_path in cached_paths:
        try:
            date=pd.Timestamp(cache_path.stem)
        except ValueError:
            continue
        try:
            raw=gpd.read_parquet(cache_path)
        except Exception:
            continue
        if 'spectral_observed_today' not in raw.columns or not (raw['spectral_observed_today']==1).any():
            continue
        clim=climate_lag_features(date)
        frame=structural[['id_cell']+PROXY_STATIC_FEATURES].merge(
            raw[['id_cell','nbr_last','ndvi_today_est','ndmi_today_est']],on='id_cell',how='inner')
        # Diagnostic : verifier explicitement, jour par jour, combien de cellules ont une valeur nbr_last
        # exploitable - un jour marque "observation reelle" (spectral_observed_today==1 quelque part) peut
        # tres bien n'avoir aucune valeur nbr_last valide. Le diagnostic precedent a montre que c'etait le
        # cas pour la grande majorite des jours en cache (0/5542 cellules), a cause d'un bug dans
        # _sentinel_indices (scene la plus recente prise sans jamais verifier sa couverture reelle de la
        # foret - deja corrige juste au-dessus, avec la meme logique que _resolve_burn_scene pour le dNBR).
        # Ces caches journaliers ont ete ecrits AVANT le correctif : on les regenere ici automatiquement au
        # lieu de simplement les ignorer, pour beneficier du correctif sans devoir tout relancer a la main.
        n_total=len(frame); n_valide=int(frame['nbr_last'].notna().sum())
        if repair_stale and (n_valide/max(n_total,1))<seuil_cache_casse:
            print('  {} : {}/{} cellules valides - cache genere avant le correctif de _sentinel_indices, regeneration...'.format(date.date(),n_valide,n_total))
            try:
                cache_path.unlink()
                new_path=_create_missing_cache(date)
                raw=gpd.read_parquet(new_path)
                frame=structural[['id_cell']+PROXY_STATIC_FEATURES].merge(
                    raw[['id_cell','nbr_last','ndvi_today_est','ndmi_today_est']],on='id_cell',how='inner')
                n_total=len(frame); n_valide=int(frame['nbr_last'].notna().sum())
                print('    -> apres regeneration : {}/{} cellules valides.'.format(n_valide,n_total))
            except Exception as exc:
                print('    -> echec de la regeneration ({}) - jour ignore.'.format(exc))
                continue
        if n_valide==0:
            print('  {} : 0/{} cellules avec nbr_last valide - jour ignore.'.format(date.date(),n_total))
            continue
        elif n_valide<n_total:
            print('  {} : {}/{} cellules avec nbr_last valide.'.format(date.date(),n_valide,n_total))
        for feat,val in clim.items(): frame[feat]=val
        frame['date']=date.date().isoformat()
        rows.append(frame)
    if not rows:
        print('Aucun jour avec observation Sentinel-2 reelle trouve dans le cache : impossible de calibrer le modele proxy pour le moment.')
        return pd.DataFrame()
    dataset=pd.concat(rows,ignore_index=True)
    dataset.to_parquet(PROXY_DATASET_CACHE,index=False)
    print('{} lignes (cellule x jour) constituees a partir de {} jour(s) avec observation reelle.'.format(len(dataset),dataset.date.nunique()))
    return dataset

def fit_spectral_proxy_model(target='nbr_last',dataset=None):
    """Regression Ridge (climat + facteurs structurels statiques) -> indice spectral observe.
    Validation par GroupKFold sur la DATE (jamais melanger les cellules d'un meme jour entre train et
    test, sinon le R2 serait artificiellement gonfle par la correlation spatiale intra-journee)."""
    if dataset is None: dataset=build_spectral_proxy_dataset()
    if dataset.empty or dataset.date.nunique()<5:
        print("Pas assez de journees distinctes avec observation reelle ({}) pour calibrer/valider serieusement ce modele (minimum recommande : 5).".format(
              dataset.date.nunique() if not dataset.empty else 0))
        return None,np.nan
    features=PROXY_STATIC_FEATURES+PROXY_CLIMATE_FEATURES
    X=dataset[features]; y=dataset[target]; groups=dataset['date']
    # SimpleImputer ci-dessous impute uniquement les features (X) - jamais la cible (y) : sklearn refuse
    # d'entrainer un modele si y contient des NaN. On retire donc les lignes sans valeur cible observee
    # (ex. cellules hors emprise de la scene Sentinel-2 ce jour-la) avant tout split/entrainement.
    masque_cible_valide=y.notna()
    if not masque_cible_valide.all():
        print("{} ligne(s) sur {} sans valeur observee pour '{}' (hors emprise image, nuage, etc.) - retirees avant calibration.".format(
              (~masque_cible_valide).sum(),len(dataset),target))
    X=X.loc[masque_cible_valide]; y=y.loc[masque_cible_valide]; groups=groups.loc[masque_cible_valide]
    if groups.nunique()<5:
        print("Apres retrait des valeurs manquantes, il ne reste que {} journee(s) distincte(s) avec '{}' observe (minimum recommande : 5) - modele proxy non calibre.".format(
              groups.nunique(),target))
        return None,np.nan
    n_splits=min(5,groups.nunique())
    gkf=GroupKFold(n_splits=n_splits)
    oof_pred=np.full(len(X),np.nan)
    for train_idx,test_idx in gkf.split(X,y,groups):
        fold=Pipeline([('impute',SimpleImputer(strategy='median',keep_empty_features=True)),
                       ('scale',StandardScaler()),('ridge',Ridge(alpha=5.0,random_state=SEED))])
        fold.fit(X.iloc[train_idx],y.iloc[train_idx])
        oof_pred[test_idx]=fold.predict(X.iloc[test_idx])
    r2=r2_score(y,oof_pred)
    print("Modele proxy pour '{}' : R2 (validation par jour exclu, {} plis) = {:.3f} sur {} jour(s) / {} lignes.".format(
          target,n_splits,r2,groups.nunique(),len(X)))
    print("Pour comparaison, la litterature rapporte R2 ~ 0.33 (chene-liege adulte, Soares et al. 2022) a ~ 0.73-0.85 (bassin du Tahaddart, Tahiri et al. 2014, climat plus proche de Bouhachem).")
    model=Pipeline([('impute',SimpleImputer(strategy='median',keep_empty_features=True)),
                    ('scale',StandardScaler()),('ridge',Ridge(alpha=5.0,random_state=SEED))])
    model.fit(X,y)
    joblib.dump({'model':model,'features':features,'target':target,'r2_validation':r2},PROXY_MODEL_PATH)
    return model,r2

# Cause identifiee : _sentinel_indices choisissait la scene Sentinel-2 la plus RECENTE sans jamais
# verifier qu'elle couvre reellement la foret (corrige plus haut, avec la meme logique de couverture
# >= 90% que _resolve_burn_scene pour le dNBR). force_rebuild=True pour ne pas relire l'ancien cache
# casse, mais SANS repair_stale (voir reparer_caches_essentiels() plus bas pour une reparation ciblee et
# rapide des seuls jours utilises par le modele, plutot que les 86 jours en cache).
# code201 (correctif) : modèle proxy climat -> NBR DÉSACTIVÉ. Il n'est utilisé nulle part par le modèle de
# risque, il lit les anciens caches de code89 (construits sur l'ancienne grille, donc 0 cellule commune avec la
# grille unique : "0/0 cellules avec nbr_last valide") et il interroge Open-Meteo pour des dizaines de jours,
# ce qui provoquait le ReadTimeout. Mettre UTILISER_MODELE_PROXY=True seulement pour l'étudier à part.
UTILISER_MODELE_PROXY=False
if UTILISER_MODELE_PROXY:
    _proxy_dataset=build_spectral_proxy_dataset(force_rebuild=True)
    _proxy_model,_proxy_r2=fit_spectral_proxy_model('nbr_last',_proxy_dataset)
    if _proxy_dataset is not None and not _proxy_dataset.empty:
        _tahiri_predit=0.0027*_proxy_dataset['precip_cum_30j']+0.2177
        print("Repere externe (formule Tahiri et al. 2014, bassin du Tahaddart) appliquee a nos propres pluies : "
              "NDVI predit moyen = {:.3f} (a comparer a l'ordre de grandeur du NBR/NDVI reellement observe ici, "
              "les deux indices ne sont pas identiques mais du meme ordre de grandeur en foret mediterraneenne).".format(
              _tahiri_predit.mean()))
else:
    print('Modèle proxy climat -> NBR désactivé (annexe, non utilisé par le modèle de risque).')


In [ ]:
def predict_spectral_proxy(date,target='nbr_last'):
    """Estime l'indice spectral (par defaut nbr_last) pour une date SANS observation Sentinel-2,
    a partir du modele climat->spectral calibre ci-dessus. A utiliser uniquement en dernier recours
    (aucune scene Sentinel-2 dans une fenetre de recherche raisonnable), jamais a la place d'une vraie
    observation quand elle existe. Retourne (valeurs_par_cellule, r2_validation_du_modele) pour que
    l'appelant puisse juger de la fiabilite avant d'utiliser le resultat."""
    if not PROXY_MODEL_PATH.exists():
        raise RuntimeError("Modele proxy climat->spectral non calibre (cellule precedente) - impossible d'estimer {} pour {}.".format(target,pd.Timestamp(date).date()))
    saved=joblib.load(PROXY_MODEL_PATH)
    if saved['target']!=target:
        raise RuntimeError("Le modele proxy sauvegarde predit '{}', pas '{}'. Relance fit_spectral_proxy_model('{}').".format(saved['target'],target,target))
    clim=climate_lag_features(date)
    frame=structural[['id_cell']+PROXY_STATIC_FEATURES].copy()
    for feat,val in clim.items(): frame[feat]=val
    valeurs=saved['model'].predict(frame[saved['features']])
    print("Estimation proxy climatique pour {} ({}) : R2 de validation du modele = {:.3f} - a utiliser avec prudence, "
          "ce n'est pas une observation directe.".format(pd.Timestamp(date).date(),target,saved['r2_validation']))
    return valeurs,saved['r2_validation']

In [ ]:
# ------------------------------------------------------------------
# Construction du jeu d'apprentissage : les 8 incendies (2020, 2022, Derdara 2021, Tanaqoub 2019a/b,
# 2005, 2009a, 2009bc) + des journées sans feu réparties dans le temps (pour ne pas apprendre la météo
# d'une seule date). CORRECTIF (23/09/2026) : ce commentaire datait de code103 (2 incendies seulement,
# 2020 et 2022) et n'avait jamais été mis à jour au fil des versions suivantes qui ont ajouté les 6
# autres incendies - le code juste en dessous, lui, a toujours construit les 8 (voir le concat plus bas).
# CORRECTIF (23/09/2026, v2 - "Exécuter tout") : toute la construction est maintenant rassemblée dans
# `construire_evenements_et_echantillons()` (au lieu d'un simple script de cellule), pour que la cellule
# de réparation des caches (plus bas) puisse la rappeler automatiquement juste après avoir réparé un
# cache essentiel - jusqu'ici il fallait relancer cette cellule à la main après la réparation, ce qui
# rendait "Exécuter tout" trompeur : le diagnostic de couverture affichait toujours l'état d'AVANT
# réparation, qu'on ait ou non un vrai problème de couverture.
# ------------------------------------------------------------------

def build_event_frame(event_date,label_override=None):
    """Fenêtre complète (structurel + météo/spectral du jour + PAMF causal) pour une date événement,
    avec éventuellement un label affiné (dNBR) qui remplace le label FIRMS par défaut."""
    frame=load_code89_day(event_date,include_target=True,allow_missing_download=True)
    frame['fire_today_firms']=frame['fire_today'].copy()
    if label_override is not None:
        frame['fire_today']=label_override
    return frame

def construire_evenements_et_echantillons():
    """Construit les 8 `event_*` (fenêtre complète + label) et `samples` (événements + journées
    négatives). Rassemblée dans une fonction (23/09/2026, v2) pour pouvoir être rappelée après
    réparation d'un cache essentiel (cellule plus bas) sans dépendre de l'ordre d'exécution manuel des
    cellules - `global` sur toutes les variables normalement définies par cette cellule, pour que la
    fonction les mette à jour exactement comme le ferait une ré-exécution manuelle de cette cellule."""
    global event_2020,event_2022,event_derdara2021,event_2019a,event_2019b,event_2005,event_2009a,event_2009bc
    global LABEL_2020_SOURCE,LABEL_2022_SOURCE,LABEL_DERDARA2021_SOURCE,LABEL_2019A_SOURCE,LABEL_2019B_SOURCE
    global LABEL_2005_SOURCE,LABEL_2009A_SOURCE,LABEL_2009BC_SOURCE
    global negative_dates,negative_frames,samples

    event_2020=build_event_frame(FIRE_2020_DATE,label_override=dnbr_label_2020)
    if event_2020.fire_today.sum()==0:
        print("Label {} pour 2020 ne donne aucune cellule positive à la date {} : recherche FIRMS dans une fenêtre de +/-{} jours.".format(
              LABEL_2020_SOURCE,FIRE_2020_DATE.date(),FIRE_2020_WINDOW_DAYS))
        event_2020['fire_today']=label_fire_window(event_2020,FIRE_2020_DATE)
        LABEL_2020_SOURCE='FIRMS (fenêtre)'
    event_2020['event_group']='incendie_2020'; event_2020['date_modele']=FIRE_2020_DATE

    event_2022=build_event_frame(FIRE_2022_EVENT_DATE,label_override=dnbr_label_2022)
    if event_2022.fire_today.sum()==0:
        print("Label {} pour 2022 ne donne aucune cellule positive à la date {} : recherche FIRMS dans une fenêtre de +/-2 jours.".format(
              LABEL_2022_SOURCE,FIRE_2022_EVENT_DATE.date()))
        event_2022['fire_today']=label_fire_window(event_2022,FIRE_2022_EVENT_DATE,window_days=2)
        LABEL_2022_SOURCE='FIRMS (fenêtre)'
    event_2022['event_group']='incendie_2022'; event_2022['date_modele']=FIRE_2022_EVENT_DATE

    event_derdara2021=build_event_frame(DERDARA_2021_DATE,label_override=dnbr_label_derdara2021)
    if event_derdara2021.fire_today.sum()==0:
        print("Label {} pour Derdara 2021 ne donne aucune cellule positive à la date {} : recherche FIRMS dans une fenêtre de +/-{} jours.".format(
              LABEL_DERDARA2021_SOURCE,DERDARA_2021_DATE.date(),FIRE_2020_WINDOW_DAYS))
        event_derdara2021['fire_today']=label_fire_window(event_derdara2021,DERDARA_2021_DATE)
        LABEL_DERDARA2021_SOURCE='FIRMS (fenêtre)'
    event_derdara2021['event_group']='incendie_derdara_2021'; event_derdara2021['date_modele']=DERDARA_2021_DATE

    event_2019a=build_event_frame(FIRE_2019A_DATE,label_override=dnbr_label_2019a)
    if event_2019a.fire_today.sum()==0:
        print("Label {} pour Tanaqoub août 2019 ne donne aucune cellule positive à la date {} : recherche FIRMS dans une fenêtre de +/-{} jours.".format(
              LABEL_2019A_SOURCE,FIRE_2019A_DATE.date(),FIRE_2020_WINDOW_DAYS))
        event_2019a['fire_today']=label_fire_window(event_2019a,FIRE_2019A_DATE)
        LABEL_2019A_SOURCE='FIRMS (fenêtre)'
    event_2019a['event_group']='incendie_tanaqoub_2019a'; event_2019a['date_modele']=FIRE_2019A_DATE

    event_2019b=build_event_frame(FIRE_2019B_DATE,label_override=dnbr_label_2019b)
    if event_2019b.fire_today.sum()==0:
        print("Label {} pour Tanaqoub septembre 2019 ne donne aucune cellule positive à la date {} : recherche FIRMS dans une fenêtre de +/-{} jours.".format(
              LABEL_2019B_SOURCE,FIRE_2019B_DATE.date(),FIRE_2020_WINDOW_DAYS))
        event_2019b['fire_today']=label_fire_window(event_2019b,FIRE_2019B_DATE)
        LABEL_2019B_SOURCE='FIRMS (fenêtre)'
    event_2019b['event_group']='incendie_tanaqoub_2019b'; event_2019b['date_modele']=FIRE_2019B_DATE

    event_2005=build_event_frame(FIRE_2005_DATE,label_override=dnbr_label_2005)
    if event_2005.fire_today.sum()==0:
        print("Label {} pour Tanaqoub-Bouhachem 2005 ne donne aucune cellule positive à la date {} : recherche FIRMS dans une fenêtre de +/-{} jours.".format(
              LABEL_2005_SOURCE,FIRE_2005_DATE.date(),FIRE_2020_WINDOW_DAYS))
        event_2005['fire_today']=label_fire_window(event_2005,FIRE_2005_DATE)
        LABEL_2005_SOURCE='FIRMS (fenêtre)'
    event_2005['event_group']='incendie_2005'; event_2005['date_modele']=FIRE_2005_DATE

    event_2009a=build_event_frame(FIRE_2009A_DATE,label_override=dnbr_label_2009a)
    if event_2009a.fire_today.sum()==0:
        print("Label {} pour Derdara-Jbel Sougna 2009 ne donne aucune cellule positive à la date {} : recherche FIRMS dans une fenêtre de +/-{} jours.".format(
              LABEL_2009A_SOURCE,FIRE_2009A_DATE.date(),FIRE_2020_WINDOW_DAYS))
        event_2009a['fire_today']=label_fire_window(event_2009a,FIRE_2009A_DATE)
        LABEL_2009A_SOURCE='FIRMS (fenêtre)'
    event_2009a['event_group']='incendie_2009a'; event_2009a['date_modele']=FIRE_2009A_DATE

    event_2009bc=build_event_frame(FIRE_2009BC_DATE,label_override=dnbr_label_2009bc)
    if event_2009bc.fire_today.sum()==0:
        # Fenetre de +/-2 jours (defaut FIRE_2020_WINDOW_DAYS) autour du 11/08 : capture aussi le 12/08, seule
        # date confirmee par FIRMS avant 2025 (voir cellule 2) - complexe Derdara-Bouhachem/Mont Bouhachem.
        print("Label {} pour le complexe Derdara-Bouhachem/Mont Bouhachem 2009 ne donne aucune cellule positive à la date {} : recherche FIRMS dans une fenêtre de +/-{} jours.".format(
              LABEL_2009BC_SOURCE,FIRE_2009BC_DATE.date(),FIRE_2020_WINDOW_DAYS))
        event_2009bc['fire_today']=label_fire_window(event_2009bc,FIRE_2009BC_DATE)
        LABEL_2009BC_SOURCE='FIRMS (fenêtre)'
    event_2009bc['event_group']='incendie_2009bc'; event_2009bc['date_modele']=FIRE_2009BC_DATE

    print('2020 : FIRMS jour exact',event_2020.fire_today_firms.value_counts().to_dict(),
          '| label retenu ({})'.format(LABEL_2020_SOURCE),event_2020.fire_today.value_counts().to_dict())
    print('2022 : FIRMS jour exact',event_2022.fire_today_firms.value_counts().to_dict(),
          '| label retenu ({})'.format(LABEL_2022_SOURCE),event_2022.fire_today.value_counts().to_dict())
    print('Derdara 2021 : FIRMS jour exact',event_derdara2021.fire_today_firms.value_counts().to_dict(),
          '| label retenu ({})'.format(LABEL_DERDARA2021_SOURCE),event_derdara2021.fire_today.value_counts().to_dict())
    print('Tanaqoub 2019a : FIRMS jour exact',event_2019a.fire_today_firms.value_counts().to_dict(),
          '| label retenu ({})'.format(LABEL_2019A_SOURCE),event_2019a.fire_today.value_counts().to_dict())
    print('Tanaqoub 2019b : FIRMS jour exact',event_2019b.fire_today_firms.value_counts().to_dict(),
          '| label retenu ({})'.format(LABEL_2019B_SOURCE),event_2019b.fire_today.value_counts().to_dict())
    print('2005 : FIRMS jour exact',event_2005.fire_today_firms.value_counts().to_dict(),
          '| label retenu ({})'.format(LABEL_2005_SOURCE),event_2005.fire_today.value_counts().to_dict())
    print('2009a : FIRMS jour exact',event_2009a.fire_today_firms.value_counts().to_dict(),
          '| label retenu ({})'.format(LABEL_2009A_SOURCE),event_2009a.fire_today.value_counts().to_dict())
    print('2009bc : FIRMS jour exact',event_2009bc.fire_today_firms.value_counts().to_dict(),
          '| label retenu ({})'.format(LABEL_2009BC_SOURCE),event_2009bc.fire_today.value_counts().to_dict())

    # On échoue tout de suite avec un message clair plutôt que de laisser une erreur sklearn cryptique
    # apparaître 2-3 cellules plus loin si un des deux incendies n'a aucune cellule positive.
    for _name,_ev in [('2020',event_2020),('2022',event_2022),('Derdara 2021',event_derdara2021),
                       ('Tanaqoub 2019a',event_2019a),('Tanaqoub 2019b',event_2019b),
                       ('2005',event_2005),('2009a',event_2009a),('2009bc',event_2009bc)]:
        if _ev.fire_today.sum()==0:
            raise RuntimeError(
                "Aucune cellule positive pour l'incendie {} après toutes les tentatives de label (dNBR, FIRMS jour "
                "exact, FIRMS fenêtre élargie). Vérifie manuellement les dates et les données FIRMS/Sentinel-2 pour "
                "cet événement.".format(_name)
            )

    # Journées "négatives" supplémentaires, pour varier la météo vue par le modèle.
    excluded=set()
    for center in [FIRE_2020_DATE,FIRE_2022_EVENT_DATE,DERDARA_2021_DATE,FIRE_2019A_DATE,FIRE_2019B_DATE,
                   FIRE_2005_DATE,FIRE_2009A_DATE,FIRE_2009BC_DATE]:
        excluded |= {(center+pd.Timedelta(days=d)).normalize() for d in range(-NEG_EXCLUSION_DAYS,NEG_EXCLUSION_DAYS+1)}

    cached_dates=[]
    for path in WINDOW_CACHE_SOURCE_89.glob('*.parquet'):   # code202 : candidates prises dans le cache code89
        try:
            d=pd.Timestamp(path.stem)
            if TRAIN_START<=d and d not in excluded: cached_dates.append(d)
        except ValueError: pass
    random.Random(SEED).shuffle(cached_dates)
    negative_dates=cached_dates[:N_NEG_DAYS]
    print(len(negative_dates),'journées négatives additionnelles tirées du cache code89 :',
          [d.date() for d in negative_dates])

    negative_frames=[]
    for date in negative_dates:
        try:
            frame=load_code89_day(date,include_target=True,cache_dir=WINDOW_CACHE_SOURCE_89)   # code202 : hors modèle spatial
            if frame.fire_today.sum()>0:
                print('  journée écartée (feu détecté malgré tout) :',date.date()); continue
            frame['event_group']=f'negatif_{date.date()}'; frame['date_modele']=date
            negative_frames.append(frame)
        except Exception as exc:
            print('  cache ignoré',date.date(),exc)

    samples=gpd.GeoDataFrame(pd.concat([event_2020,event_2022,event_derdara2021,event_2019a,event_2019b,
                  event_2005,event_2009a,event_2009bc]+negative_frames,ignore_index=True),crs=CRS)
    print('Total échantillons :',len(samples),'| classes :',samples.fire_today.value_counts().to_dict(),
          '| journées distinctes :',samples.date_modele.nunique())

construire_evenements_et_echantillons()

### Réparation ciblée des caches essentiels (pas les 86 jours, juste ceux utilisés par le modèle)

Le diagnostic sur le modèle proxy climatique a révélé un bug dans `_sentinel_indices` (déjà corrigé plus haut) : la scène Sentinel-2 la plus récente était choisie sans jamais vérifier qu'elle couvre réellement la forêt, produisant des `nbr_last`/`nbr_trend_day`/`nbr_today_est` presque entièrement `NaN` sur la plupart des jours en cache. Régénérer les **86** jours en cache un par un (comme tenté dans la cellule du modèle proxy avec `repair_stale=True`) est trop lent (plusieurs dizaines de minutes, voire plus d'une heure) et beaucoup de dates anciennes (2017-2018) échoueront de toute façon faute de scène disponible avec une couverture suffisante.

Ici on ne répare que les jours **réellement utilisés par le modèle de risque principal** : les 8 événements d'entraînement (2020, 2022, Derdara 2021, Tanaqoub 2019a/b, 2005, 2009a, 2009bc) et les journées négatives tirées juste au-dessus - un ensemble beaucoup plus petit que les 86 jours du cache complet, donc beaucoup plus rapide, et c'est ce qui compte vraiment pour la qualité du modèle (pas seulement pour le modèle proxy annexe). **Correctif (23/09/2026)** : le critère de "cache cassé" ne regardait que la couverture *globale* de `nbr_last`, qui reste souvent > 5% même quand une ou plusieurs parties entières (b à f, ajoutées après ces caches) sont à 0% - la cellule ci-dessous vérifie désormais aussi la couverture *par partie* avant de déclarer un cache bon. **Correctif (23/09/2026, v2 - "Exécuter tout")** : la cellule ci-dessous reconstruit maintenant automatiquement `samples` juste après une réparation, et le diagnostic de couverture par partie a été déplacé juste après elle (au lieu d'avant) - avec "Exécuter tout", ce diagnostic reflète donc toujours l'état réel, après réparation, plutôt que l'état d'avant.

In [ ]:
# ------------------------------------------------------------------
# Reparation CIBLEE (pas les 86 jours du cache, juste ceux reellement utilises par le modele) : les 8
# incendies d'entrainement + les journees negatives tirees ci-dessus. Beaucoup plus rapide qu'une reparation
# generale (voir la discussion sur build_spectral_proxy_dataset/repair_stale, desactivee par defaut car
# trop lente sur les 86 jours), et c'est ce qui compte vraiment pour la qualite du modele de risque principal
# (pas seulement le modele proxy).
# CORRECTIF (23/09/2026) : le critere de couverture ne regardait QUE nbr_last.notna().mean() sur TOUTE la
# grille - or la partie "a" (Bouhachem, majoritaire en nombre de cellules) suffit a elle seule a depasser
# seuil_cache_casse (5%) meme quand b/c/d/e/f sont ENTIEREMENT a NaN (caches herites d'avant l'ajout de
# ces parties). Consequence concrete observee : couverture satellite/meteo a 12,5% sur b-f dans le
# diagnostic de la cellule precedente, alors que cette fonction jugeait la plupart des caches "deja bons"
# et ne les reparait pas. On verifie maintenant EN PLUS la couverture PAR PARTIE (meme methode que le
# diagnostic de couverture structurelle, cellule 10) et on force une regeneration complete si une partie
# quelconque tombe sous 50% - la seule chose qui repare vraiment ce cas est de refaire la recherche
# Sentinel-2/Landsat avec l'AOI a jour (6 parties), donc le chemin rapide "repatch meteo seule" est aussi
# desactive quand ce probleme est detecte.
# ------------------------------------------------------------------
def reparer_caches_essentiels(dates,seuil_cache_casse=0.05,seuil_couverture_partie=0.5):
    reparees,echecs,deja_bonnes=[],[],[]
    for date in dates:
        date=pd.Timestamp(date).normalize()
        cache_path=WINDOW_CACHE_89/f'{date.date()}.parquet'
        if not cache_path.exists():
            continue
        try:
            raw=gpd.read_parquet(cache_path)
        except Exception as exc:
            print('  {} : cache illisible ({}) - ignore.'.format(date.date(),exc)); continue
        if 'nbr_last' not in raw.columns:
            continue
        if 'cache_version' in raw.columns and (raw['cache_version']=='code201').all():
            # code202 : cache déjà construit avec la nouvelle règle NBR - le régénérer donnerait le même résultat
            deja_bonnes.append(date); continue
        couverture=raw['nbr_last'].notna().mean()
        # CORRECTIF (23/09/2026) : voir le commentaire d'en-tete de cette cellule - une couverture globale
        # correcte peut cacher une ou plusieurs parties (b-f) entierement a NaN. On calcule ici la couverture
        # PAR PARTIE (jointure sur id_cell avec `grid`, qui a toujours la partie a jour) et on considere le
        # cache "casse" si au moins une partie tombe sous seuil_couverture_partie, meme si couverture globale
        # est bonne.
        _raw_par_partie=grid[['id_cell','partie']].merge(raw[['id_cell','nbr_last']],on='id_cell',how='left')
        _couverture_par_partie=_raw_par_partie.groupby('partie')['nbr_last'].apply(lambda d:d.notna().mean())
        _parties_manquantes=_couverture_par_partie[_couverture_par_partie<seuil_couverture_partie]
        couverture_par_partie_ok=_parties_manquantes.empty
        # Deuxieme critere de "cache casse" (inversé par rapport a l'ancienne version) : la temperature
        # varie ENCORE entre cellules (ecart-type notable) => cache genere avec l'ancien correctif de
        # spatialisation par altitude, maintenant ANNULE (voir _meteo_spatialisee : ce correctif dupliquait
        # exactement altitude_m sous 5 formes, ce qui a fait chuter le ROC-AUC 2025 de 0.80 a 0.12). Un
        # cache "bon" doit de nouveau avoir une meteo uniforme (ecart-type ~ 0).
        meteo_encore_spatialisee=('temperature_2m' in raw.columns and raw['temperature_2m'].notna().sum()>1
                                   and raw['temperature_2m'].std()>1e-9)
        if couverture>=seuil_cache_casse and not meteo_encore_spatialisee and couverture_par_partie_ok:
            deja_bonnes.append(date); continue

        # Cas rapide : le spectral (nbr_last) est deja bon, seule la meteo doit etre repatchee - pas
        # besoin de refaire toute la recherche Sentinel-2 (couteuse en reseau, c'est ça qui rendait la
        # reparation tres lente quand TOUTES les dates essentielles etaient marquees "meteo perimee").
        if couverture>=seuil_cache_casse and meteo_encore_spatialisee and couverture_par_partie_ok:
            if 'altitude_m' in raw.columns and raw['altitude_m'].notna().sum()>1:
                correlation=np.corrcoef(raw['temperature_2m'],raw['altitude_m'])[0,1]
                print('  {} : diagnostic - correlation temperature_2m / altitude_m = {:.3f} (proche de -1 = '
                      'quasi-doublon, confirme la cause de la regression du ROC-AUC).'.format(date.date(),correlation))
            print('  {} : nbr_last déjà bon, repatch rapide vers une météo de nouveau uniforme (pas de nouvelle recherche Sentinel-2)...'.format(date.date()))
            try:
                repatcher_meteo_seule(date)
                raw2=gpd.read_parquet(cache_path)
                print('    -> écart-type température = {:.3f}°C (doit être ~0 maintenant).'.format(raw2['temperature_2m'].std()))
                reparees.append(date)
            except Exception as exc:
                print('    -> echec ({}).'.format(exc)); echecs.append(date)
            continue

        raison=[]
        if couverture<seuil_cache_casse: raison.append('{:.0%} de couverture nbr_last'.format(couverture))
        if meteo_encore_spatialisee: raison.append('météo encore spatialisée par altitude (correctif annulé)')
        if not couverture_par_partie_ok:
            raison.append('partie(s) {} sous {:.0%} de couverture ({})'.format(
                list(_parties_manquantes.index),seuil_couverture_partie,
                {k:round(float(v),3) for k,v in _parties_manquantes.items()}))
        print('  {} : {} - cache généré avant un correctif, régénération complète (recherche Sentinel-2)...'.format(date.date(),' + '.join(raison)))
        try:
            cache_path.unlink()
            _create_missing_cache(date)
            raw2=gpd.read_parquet(cache_path)
            nouvelle_couverture=raw2['nbr_last'].notna().mean()
            print('    -> après régénération : {:.0%} de couverture nbr_last, écart-type température = {:.3f}°C.'.format(
                  nouvelle_couverture,raw2['temperature_2m'].std()))
            reparees.append(date)
        except Exception as exc:
            print('    -> echec ({}).'.format(exc)); echecs.append(date)
    print('Bilan : {} deja bonne(s), {} reparee(s), {} en echec (sur {} jour(s) essentiel(s)).'.format(
          len(deja_bonnes),len(reparees),len(echecs),len(dates)))
    if reparees:
        print('{} cache(s) essentiel(s) repare(s) - reconstruction automatique de samples juste apres cet '
              'appel (voir plus bas).'.format(len(reparees)))
    return reparees,echecs

_dates_essentielles=[FIRE_2020_DATE,FIRE_2022_EVENT_DATE,DERDARA_2021_DATE,FIRE_2019A_DATE,FIRE_2019B_DATE,
                     FIRE_2005_DATE,FIRE_2009A_DATE,FIRE_2009BC_DATE]+list(negative_dates)
_reparees,_echecs=reparer_caches_essentiels(_dates_essentielles)
# CORRECTIF (23/09/2026, v2 - "Exécuter tout") : si un cache a ete repare, les event_*/samples
# construits par la cellule precedente (AVANT reparation) sont perimes. Jusqu'ici il fallait relancer
# cette cellule-la a la main - avec "Executer tout", le diagnostic de couverture plus bas affichait donc
# toujours l'etat d'avant reparation, meme quand la reparation avait reussi. On rappelle maintenant
# construire_evenements_et_echantillons() automatiquement ici pour que la suite du notebook utilise
# directement les caches corriges, sans intervention manuelle.
if _reparees:
    construire_evenements_et_echantillons()


In [ ]:
# ------------------------------------------------------------------
# NOUVEAU (code105, 23/09/2026) : verification de couverture satellite/meteo par partie, sur les
# echantillons d'entrainement reellement construits ci-dessus - complement a la cellule de couverture
# structurelle (altitude_m/pente_pct/...) plus haut, qui ne teste PAS ces colonnes-la. nbr_last/
# nbr_trend_day/temperature_2m/... proviennent du cache journalier (WINDOW_CACHE_89), repare ci-dessus
# si necessaire.
# CORRECTIF (23/09/2026, v2 - "Exécuter tout") : cette cellule a ete deplacee pour tourner APRES la
# reparation ciblee des caches essentiels et sa reconstruction automatique de `samples` (voir plus
# haut) - elle sert donc maintenant de verification FINALE, a jour meme avec "Executer tout" (avant ce
# correctif, elle tournait AVANT la reparation et affichait donc toujours l'etat d'avant reparation,
# quel que soit le resultat reel de reparer_caches_essentiels).
# ------------------------------------------------------------------
_colonnes_satmeteo_a_verifier=[c for c in ['nbr_last','nbr_trend_day','ndvi_last','ndmi_last',
                                            'temperature_2m','relative_humidity_2m','wind_speed_10m',
                                            'precipitation','vpd_mean'] if c in samples.columns]
_couverture_satmeteo=samples.groupby('partie')[_colonnes_satmeteo_a_verifier].apply(lambda d:d.notna().mean())
print('Couverture satellite/météo (proportion de cellules NON manquantes) par partie, sur les échantillons construits :')
display(_couverture_satmeteo)
for _partie in _couverture_satmeteo.index:
    _pires=_couverture_satmeteo.loc[_partie][_couverture_satmeteo.loc[_partie]<0.5]
    if len(_pires):
        print('  ATTENTION : partie "{}" a encore une couverture satellite/météo < 50% pour : {} malgré la '
              'réparation automatique ci-dessus - vérifie manuellement les caches de cette zone (dates dans '
              '_dates_essentielles) avant de faire confiance aux résultats sur cette partie.'.format(_partie,', '.join(_pires.index)))


In [ ]:
# x_km/y_km volontairement exclues : les coordonnées brutes risqueraient de faire mémoriser au modèle
# "où" les feux ont eu lieu plutôt que d'apprendre des facteurs transférables.
FEATURES_TOUTES=[
 'pamf_log1p','pamf_classe_1_5','frp_historique_km2','pression_humaine_01',
 'altitude_m','pente_pct','exposition_deg','population',
 'occupation_sol_2021','inflammabilite_01',
 'topoclimat_brut','topoclimat_classe_1_5',
 'nbr_last','nbr_trend_day','image_age_days','spectral_observed_today',   # code202 : nbr_today_est retiré
 'temperature_2m','temperature_2m_max','relative_humidity_2m','relative_humidity_2m_min',
 'wind_speed_10m','wind_speed_10m_max','wind_direction_deg','precipitation','vpd_mean','vpd_max','doy_sin','doy_cos']

# ------------------------------------------------------------------
# NOUVEAU (code106) : MODÈLE SPATIAL ("où ?") séparé du DANGER DU JOUR ("quel jour ?").
# Dans code105, plus de 70 % de l'importance venait de variables IDENTIQUES pour toutes les cellules d'un
# même jour (température, saison, vent...), parce que les cellules brûlées venaient toutes de jours de feu
# et la plupart des cellules non brûlées de journées sans feu : le modèle apprenait "est-ce un jour de feu ?"
# et mettait toute la forêt en alerte (ou aucune cellule) d'un bloc.
# Ici :
#   - on n'entraîne que sur les 8 JOURS DE FEU : cellules brûlées contre cellules non brûlées du MÊME jour ;
#   - on retire les variables identiques pour toute la forêt un même jour (VARIABLES_DU_JOUR, plus un
#     contrôle automatique au cas où une autre variable serait constante dans la journée) ;
#   - on retire les cellules sans dNBR valide (nuages, bord d'image), qui étaient comptées "non brûlées".
# Le danger météo du jour est traité à part (indice_danger_jour, cellule après l'entraînement final).
# Les journées négatives (`samples`, cellule 17) restent construites et réparées comme avant : elles
# servent au diagnostic de couverture, mais plus à l'entraînement du modèle spatial.
# ------------------------------------------------------------------
VARIABLES_DU_JOUR=['temperature_2m','temperature_2m_max','relative_humidity_2m','relative_humidity_2m_min',
                   'wind_speed_10m','wind_speed_10m_max','wind_direction_deg','precipitation','vpd_mean','vpd_max',
                   'doy_sin','doy_cos','image_age_days','spectral_observed_today']

samples_spatial=samples[~samples.event_group.str.startswith('negatif_')].copy()
_valide=np.ones(len(samples_spatial),dtype=bool)
for _nom,_val in VALIDITE_DNBR.items():
    if _val is None: continue
    _masque=(samples_spatial.event_group=='incendie_'+_nom).to_numpy()
    if _masque.sum()==len(_val):
        _valide[_masque]=_val
    else:
        print('ATTENTION : {} lignes pour incendie_{} mais {} valeurs de validité dNBR - validité ignorée pour cet incendie.'.format(
              int(_masque.sum()),_nom,len(_val)))
print('Cellules retirées faute de dNBR valide : {} sur {}.'.format(int((~_valide).sum()),len(samples_spatial)))
samples_spatial=gpd.GeoDataFrame(samples_spatial[_valide].reset_index(drop=True),crs=CRS)
print('Échantillons du modèle spatial (8 jours de feu) :',len(samples_spatial),'| classes :',
      samples_spatial.fire_today.value_counts().to_dict())
display(samples_spatial.groupby('event_group').fire_today.agg(cellules='size',brulees='sum',part_brulee='mean'))

print('Image satellite utilisée pour chaque incendie (doit être AVANT la date de départ, âge <= {} j Sentinel-2 / {} j Landsat) :'.format(
      AGE_MAX_SENTINEL2,AGE_MAX_LANDSAT))
display(samples_spatial.groupby('event_group')[['date_modele','satellite_date','image_age_days']].first())

FEATURES=[x for x in FEATURES_TOUTES if x not in VARIABLES_DU_JOUR and x in samples_spatial and samples_spatial[x].notna().any()]
_ecart_dans_la_journee=samples_spatial.groupby('event_group')[FEATURES].std().median()
_constantes=list(_ecart_dans_la_journee[_ecart_dans_la_journee.fillna(0)<1e-9].index)
if _constantes:
    print('Variables retirées car constantes dans une même journée :',_constantes)
FEATURES=[x for x in FEATURES if x not in _constantes]
print(len(FEATURES),'inputs du modèle spatial :',FEATURES)

def make_model():
    # Modele volontairement plus contraint qu'au depart : avec seulement ~80-120 cellules positives
    # au total reparties sur 2 dates, un RF a 800 arbres sans limite de profondeur memorise facilement
    # le bruit propre a chaque evenement plutot qu'un signal transferable (voir les AUC proches ou en
    # dessous de 0.5 en leave-one-fire-out). On reduit donc fortement la capacite du modele.
    return Pipeline([('impute',SimpleImputer(strategy='median',keep_empty_features=True)),
      ('rf',RandomForestClassifier(n_estimators=300,max_depth=6,min_samples_leaf=10,max_features='sqrt',
       class_weight='balanced_subsample',n_jobs=-1,random_state=SEED))])

def make_model_logreg():
    # Alternative encore plus simple : une regression logistique regularisee generalise souvent mieux
    # qu'un modele complexe quand il y a tres peu d'evenements positifs independants.
    return Pipeline([('impute',SimpleImputer(strategy='median',keep_empty_features=True)),
      ('scale',StandardScaler()),
      ('logreg',LogisticRegression(penalty='l2',C=0.5,class_weight='balanced',max_iter=2000,random_state=SEED))])

def make_model_xgb():
    # Boosting regularise (peu d'arbres, tres peu profonds, forte regularisation L2, sous-echantillonnage
    # des lignes/colonnes) - comme pour le RF, l'idee n'est pas de maximiser la capacite du modele mais de
    # voir si XGBoost generalise mieux que le RF/la regression logistique avec seulement 2 evenements
    # sources. scale_pos_weight=10 correspond au ratio negatif:positif deja impose par balance_period.
    return Pipeline([('impute',SimpleImputer(strategy='median',keep_empty_features=True)),
      ('xgb',XGBClassifier(n_estimators=150,max_depth=3,learning_rate=0.05,subsample=0.8,
       colsample_bytree=0.8,reg_lambda=5.0,min_child_weight=5,scale_pos_weight=10,
       eval_metric='logloss',n_jobs=-1,random_state=SEED))])

def make_model_lgbm():
    # Meme philosophie que XGBoost : boosting fortement regularise (peu d'arbres, arbres tres peu
    # profonds/peu de feuilles, forte regularisation L2, sous-echantillonnage des lignes/colonnes) pour
    # eviter de memoriser le bruit propre a chaque evenement avec seulement 2 incendies sources.
    # class_weight='balanced' compense le desequilibre positif/negatif deja attenue par balance_period.
    return Pipeline([('impute',SimpleImputer(strategy='median',keep_empty_features=True)),
      ('lgbm',LGBMClassifier(n_estimators=150,max_depth=3,num_leaves=7,learning_rate=0.05,
       subsample=0.8,colsample_bytree=0.8,reg_lambda=5.0,min_child_samples=10,
       class_weight='balanced',n_jobs=-1,random_state=SEED,verbosity=-1))])

def classes_risque_non_supervisees(probabilities,n_classes=5):
    values=np.asarray(probabilities,dtype=float).reshape(-1,1)
    unique=np.unique(values[np.isfinite(values)])
    k=min(n_classes,len(unique))
    if k<2:return np.ones(len(values),dtype='uint8')
    km=KMeans(n_clusters=k,n_init=30,random_state=SEED).fit(values)
    centers=km.cluster_centers_.ravel()
    order=np.argsort(centers)
    mapping={cluster:rank+1 for rank,cluster in enumerate(order)}
    classes=np.asarray([mapping[x] for x in km.labels_],dtype='uint8')
    print('Centres des classes non supervisées :',{mapping[i]:round(float(centers[i]),4) for i in range(k)})
    return classes

def balance_period(frame,negative_ratio=10,seed=SEED):
    positive=frame[frame.fire_today==1]; negative=frame[frame.fire_today==0]
    if positive.empty: return frame.iloc[0:0].copy()
    n=min(len(negative),max(len(positive)*negative_ratio,100))
    sampled_negative=negative.sample(n=n,random_state=seed)
    return pd.concat([positive,sampled_negative],ignore_index=True).sample(frac=1,random_state=seed).reset_index(drop=True)

In [ ]:
# ------------------------------------------------------------------
# code204 : ÉVALUATION CORRIGÉE DU MODÈLE SPATIAL
# 1) Incendies retirés : leurs labels ne sont pas exploitables.
#      - tanaqoub_2019a (1 cellule brûlée) et tanaqoub_2019b (3 cellules) : label presque vide ;
#      - 2005 : aucune image satellite avant le feu (variables satellite remplacées par la médiane).
# 2) Découpage répété : chaque incendie exploitable sert de TEST à tour de rôle, REPETITIONS_PAR_FEU fois,
#    avec 2 incendies de VALIDATION tirés au hasard parmi les autres. On ne présente que des scores
#    d'incendies jamais vus par le modèle évalué :
#      - score de VALIDATION : modèle entraîné SANS les 2 incendies de validation ni le test ;
#      - score de TEST       : modèle final (entraînement + validation), SANS l'incendie de test.
# ------------------------------------------------------------------
warnings.filterwarnings('ignore',category=FutureWarning)
INCENDIES_EXCLUS=['tanaqoub_2019a','tanaqoub_2019b','2005']
MIN_CELLULES_EVAL=20
REPETITIONS_PAR_FEU=2
SEED_DECOUPAGE=SEED

NOMS_INCENDIES=['2020','2022','derdara_2021','tanaqoub_2019a','tanaqoub_2019b','2005','2009a','2009bc']
EVENTS_TOUS={nom:samples_spatial[samples_spatial.event_group=='incendie_'+nom].reset_index(drop=True)
             for nom in NOMS_INCENDIES if nom not in INCENDIES_EXCLUS}
print('Incendies retirés (labels non exploitables) :',', '.join(INCENDIES_EXCLUS))

CADRES_2025={}
for _ev in EVENEMENTS_2025:
    _nom='2025_'+_ev['slug']
    _frame=load_code89_day(_ev['test'],include_target=False,allow_missing_download=True,operational=False)
    _brut,_valeurs=compute_dnbr_label(_ev['pre'],_ev['after'],search_days_before=_ev.get('sd_avant'),
                                      search_days_after=_ev.get('sd_apres'))
    _frame['fire_today']=affiner_label_dnbr(_brut,_valeurs,_ev['debut'],_ev['fin'],None,nom=_nom)
    _frame['dnbr']=_valeurs
    _frame['event_group']='incendie_'+_nom; _frame['date_modele']=_ev['test']
    CADRES_2025[_nom]=gpd.GeoDataFrame(_frame,geometry='geometry',crs=CRS)
    EVENTS_TOUS[_nom]=gpd.GeoDataFrame(_frame[np.isfinite(_valeurs)].reset_index(drop=True),geometry='geometry',crs=CRS)

# ------------------------------------------------------------------
# code205 : HISTORIQUE DES FEUX PAR CELLULE -> "années depuis le dernier feu"
# Une zone qui vient de brûler a perdu son combustible pour plusieurs années. Cette variable permet au modèle
# d'apprendre cet effet (code204 plaçait le risque sur les zones déjà brûlées).
# Sources de l'historique, toujours STRICTEMENT AVANT la date étudiée (aucune fuite) :
#   - points FIRMS depuis 2005 (cellules à moins de RAYON_HISTORIQUE_M du point) ;
#   - zones brûlées dNBR des incendies connus (date de départ antérieure à la date étudiée).
# Une cellule sans feu connu reçoit ANNEES_MAX (= "pas de feu connu depuis le début des données FIRMS").
# ------------------------------------------------------------------
RAYON_HISTORIQUE_M=375      # demi-taille d'un pixel VIIRS (375 m)
ANNEES_MAX=20.0

_pts=firms[['date','geometry']].copy(); _pts['geometry']=_pts.geometry.buffer(RAYON_HISTORIQUE_M)
_hist_firms=gpd.sjoin(_pts,grid[['id_cell','geometry']],predicate='intersects',how='inner')[['id_cell','date']]
_lignes_hist=[_hist_firms]
for _nom,_cfg in EVENEMENTS_DNBR.items():
    _lab=globals().get('dnbr_label_'+_cfg['var'])
    if _lab is not None and len(_lab)==len(grid):
        _ids=grid.id_cell.to_numpy()[np.asarray(_lab)==1]
        _lignes_hist.append(pd.DataFrame({'id_cell':_ids,'date':pd.Timestamp(_cfg['debut'])}))
for _ev in EVENEMENTS_2025:
    _c=CADRES_2025.get('2025_'+_ev['slug'])
    if _c is not None:
        _lignes_hist.append(pd.DataFrame({'id_cell':_c.id_cell[_c.fire_today==1].to_numpy(),'date':pd.Timestamp(_ev['debut'])}))
HISTORIQUE_FEUX=pd.concat(_lignes_hist,ignore_index=True)
HISTORIQUE_FEUX['date']=pd.to_datetime(HISTORIQUE_FEUX['date']).dt.normalize()
print("Historique des feux : {} couples (cellule, date) issus de FIRMS et des zones brûlées dNBR.".format(len(HISTORIQUE_FEUX)))

def annees_depuis_dernier_feu(date):
    date=pd.Timestamp(date).normalize()
    h=HISTORIQUE_FEUX[HISTORIQUE_FEUX.date<date]
    dernier=h.groupby('id_cell').date.max()
    annees=((date-dernier).dt.days/365.25).clip(0,ANNEES_MAX)
    return annees.reindex(grid.id_cell).fillna(ANNEES_MAX)

def ajouter_historique(frame,date):
    a=annees_depuis_dernier_feu(date)
    frame=frame.copy()
    frame['annees_depuis_dernier_feu']=frame.id_cell.map(a).fillna(ANNEES_MAX).to_numpy()
    frame['brule_moins_5ans']=(frame['annees_depuis_dernier_feu']<5).astype(int)
    return frame

for _n in list(EVENTS_TOUS):
    EVENTS_TOUS[_n]=ajouter_historique(EVENTS_TOUS[_n],EVENTS_TOUS[_n].date_modele.iloc[0])
for _n in list(CADRES_2025):
    CADRES_2025[_n]=ajouter_historique(CADRES_2025[_n],CADRES_2025[_n].date_modele.iloc[0])
FEATURES=[f for f in FEATURES if f not in ('annees_depuis_dernier_feu','brule_moins_5ans')]+['annees_depuis_dernier_feu','brule_moins_5ans']
print('Variables ajoutées au modèle spatial : annees_depuis_dernier_feu, brule_moins_5ans ({} inputs au total).'.format(len(FEATURES)))
display(pd.DataFrame([{'incendie':n,'part_cellules_brulees_moins_5ans':ev.brule_moins_5ans.mean(),
                       'part_brulees_parmi_feu_actuel_deja_brulees_<5ans':(ev.brule_moins_5ans[ev.fire_today==1].mean() if ev.fire_today.sum() else np.nan)}
                      for n,ev in EVENTS_TOUS.items()]))


# ------------------------------------------------------------------
# code206 : 1) variables satellite retirées du modèle spatial. nbr_last, nbr_trend_day et inflammabilite_01
#              (calculée avec le NDVI/NDMI de la dernière image) sont très bas sur une zone qui vient de brûler :
#              le modèle confondait "végétation déjà brûlée" et "végétation sèche" et y mettait un risque fort.
#           2) RÈGLE DU COMBUSTIBLE, appliquée APRÈS la prédiction (règle physique, pas apprise) :
#              brûlée il y a moins de 1 an : risque multiplié par 0 (plus de combustible) ;
#              brûlée il y a 1 à 5 ans     : risque multiplié par FACTEUR_1_5_ANS.
#              Justifiée par les données (code205) : les nouveaux feux évitent les zones brûlées depuis < 5 ans.
# ------------------------------------------------------------------
VARIABLES_SATELLITE_RETIREES=[]   # code208 : variables satellite GARDÉES (les retirer faisait baisser le score : 0,576 -> 0,498 dans code206)
FEATURES=[f for f in FEATURES if f not in VARIABLES_SATELLITE_RETIREES]
print('code206 : variables retirées :',VARIABLES_SATELLITE_RETIREES,'| {} inputs du modèle spatial : {}'.format(len(FEATURES),FEATURES))
FACTEUR_1_5_ANS=0.5
def regle_combustible(prob,annees):
    prob=np.asarray(prob,float).copy(); a=np.asarray(annees,float)
    prob[a<1]=0.; prob[(a>=1)&(a<5)]*=FACTEUR_1_5_ANS
    return prob
def predire(modele,frame,appliquer_regle=True):
    p=modele.predict_proba(frame[FEATURES])[:,1]
    return regle_combustible(p,frame['annees_depuis_dernier_feu']) if appliquer_regle else p

_eligibles=sorted(n for n,ev in EVENTS_TOUS.items() if ev.fire_today.sum()>=MIN_CELLULES_EVAL)
print('Incendies utilisables en test / validation (>= {} cellules brûlées) :'.format(MIN_CELLULES_EVAL),', '.join(_eligibles))
display(pd.DataFrame([{'incendie':n,'cellules_valides':len(ev),'cellules_brulees':int(ev.fire_today.sum())}
                      for n,ev in EVENTS_TOUS.items()]))

def matrice_alerte_relative(prob,verite,fraction):
    """Alerte = les `fraction` x 100 % de cellules les plus risquées de la journée."""
    prob=np.asarray(prob,float); verite=np.asarray(verite).astype(int)
    seuil=float(np.quantile(prob,1-fraction)); alerte=prob>=seuil
    vp=int((alerte&(verite==1)).sum()); fn=int((~alerte&(verite==1)).sum())
    fp=int((alerte&(verite==0)).sum()); vn=int((~alerte&(verite==0)).sum())
    precision=vp/max(vp+fp,1); rappel=vp/max(vp+fn,1)
    f2=5*precision*rappel/(4*precision+rappel) if (4*precision+rappel)>0 else 0.
    return {'seuil_probabilite':seuil,'VP':vp,'FN':fn,'FP':fp,'VN':vn,'precision':precision,'rappel':rappel,'F2':f2}

candidates={'RandomForest (régularisé)':make_model,'Régression logistique':make_model_logreg,
            'XGBoost (régularisé)':make_model_xgb,'LightGBM (régularisé)':make_model_lgbm}

def executer_decoupage(test,validation):
    """Un découpage complet : choix du modèle et de la part d'alerte sur la validation, puis score du test."""
    entrainement=[n for n in EVENTS_TOUS if n!=test and n not in validation]
    train=balance_period(pd.concat([EVENTS_TOUS[n] for n in entrainement],ignore_index=True),negative_ratio=10,seed=SEED)
    lignes=[]; probs={}; modeles={}
    for nom_modele,fabrique in candidates.items():
        m=fabrique(); m.fit(train[FEATURES],train.fire_today); modeles[nom_modele]=m
        probs[nom_modele]={n:predire(m,EVENTS_TOUS[n]) for n in validation}
        ligne={'modèle':nom_modele}
        for n in validation:
            y=EVENTS_TOUS[n].fire_today.to_numpy(); p=probs[nom_modele][n]
            ligne[f'ROC_AUC_{n}']=roc_auc_score(y,p); ligne[f'PR_lift_{n}']=average_precision_score(y,p)/y.mean()
        ligne['ROC_AUC_moyen']=np.mean([ligne[f'ROC_AUC_{n}'] for n in validation])
        ligne['ROC_AUC_min']=np.min([ligne[f'ROC_AUC_{n}'] for n in validation])
        ligne['PR_lift_moyen']=np.mean([ligne[f'PR_lift_{n}'] for n in validation])
        lignes.append(ligne)
    table=pd.DataFrame(lignes).sort_values(['ROC_AUC_moyen','ROC_AUC_min'],ascending=False).reset_index(drop=True)
    meilleur=table.iloc[0]['modèle']
    f2={fr:np.mean([matrice_alerte_relative(probs[meilleur][n],EVENTS_TOUS[n].fire_today,fr)['F2'] for n in validation])
        for fr in FRACTIONS_ALERTE_CANDIDATES}
    fraction=max(f2,key=f2.get)
    final=balance_period(pd.concat([EVENTS_TOUS[n] for n in entrainement+list(validation)],ignore_index=True),negative_ratio=10,seed=SEED+1)
    modele_final=candidates[meilleur](); modele_final.fit(final[FEATURES],final.fire_today)
    y=EVENTS_TOUS[test].fire_today.to_numpy(); p=predire(modele_final,EVENTS_TOUS[test])
    p_sans_regle=predire(modele_final,EVENTS_TOUS[test],appliquer_regle=False)
    mat=matrice_alerte_relative(p,y,fraction)
    resultat={'test':test,'validation':' + '.join(validation),'modele':meilleur,'part_alerte':fraction,
              'ROC_AUC_validation':table.iloc[0]['ROC_AUC_moyen'],'ROC_AUC_test':roc_auc_score(y,p),'ROC_AUC_test_sans_regle':roc_auc_score(y,p_sans_regle),
              'PR_lift_test':average_precision_score(y,p)/y.mean(),'rappel_test':mat['rappel'],
              'precision_test':mat['precision'],'precision_hasard':y.mean()}
    return resultat,table,modeles[meilleur],modele_final,fraction,meilleur

# --- Découpages répétés : chaque incendie exploitable sert de test ---
_rng=random.Random(SEED_DECOUPAGE); _resultats=[]
for _test in _eligibles:
    _autres=[n for n in _eligibles if n!=_test]
    for _r in range(REPETITIONS_PAR_FEU):
        _val=_rng.sample(_autres,2)
        _res,*_=executer_decoupage(_test,_val)
        _res['repetition']=_r+1; _resultats.append(_res)
        print('  test {:<28} validation {:<45} ROC-AUC test = {:.3f}'.format(_test,_res['validation'],_res['ROC_AUC_test']))
TABLE_REPETITIONS=pd.DataFrame(_resultats)
print('Résultats de tous les découpages (seuls des incendies jamais vus sont évalués) :')
display(TABLE_REPETITIONS)
print('Moyenne par incendie de test :')
display(TABLE_REPETITIONS.groupby('test')[['ROC_AUC_test','ROC_AUC_test_sans_regle','PR_lift_test','rappel_test','precision_test','precision_hasard']].mean())
print('Effet de la règle du combustible : ROC-AUC test moyen {:.3f} sans la règle, {:.3f} avec.'.format(
      TABLE_REPETITIONS.ROC_AUC_test_sans_regle.mean(),TABLE_REPETITIONS.ROC_AUC_test.mean()))
print('BILAN GLOBAL : ROC-AUC test moyen = {:.3f} (écart-type {:.3f}) ; PR-AUC test = {:.1f} x le hasard ; '
      '{} découpage(s) sur {} au-dessus du hasard (ROC-AUC > 0,5).'.format(
      TABLE_REPETITIONS.ROC_AUC_test.mean(),TABLE_REPETITIONS.ROC_AUC_test.std(),TABLE_REPETITIONS.PR_lift_test.mean(),
      int((TABLE_REPETITIONS.ROC_AUC_test>0.5).sum()),len(TABLE_REPETITIONS)))

# --- Découpage de présentation (cartes détaillées plus bas) ---
_tirage=random.Random(SEED_DECOUPAGE).sample(_eligibles,3)
INCENDIE_TEST=_tirage[0]; INCENDIES_VALIDATION=_tirage[1:]
INCENDIES_ENTRAINEMENT=[n for n in EVENTS_TOUS if n not in _tirage]
_res_presentation,comparison_table,model_sans_validation,model,FRACTION_ALERTE,best_name=executer_decoupage(INCENDIE_TEST,INCENDIES_VALIDATION)
BEST_MODEL_FACTORY=candidates[best_name]
comparison_table['score_generalisation']=comparison_table['ROC_AUC_moyen']
print('Découpage présenté en cartes : test = {} ; validation = {} ; entraînement = {}.'.format(
      INCENDIE_TEST,', '.join(INCENDIES_VALIDATION),', '.join(INCENDIES_ENTRAINEMENT)))
print('Comparaison des modèles sur la validation (modèles entraînés SANS les incendies de validation) :')
display(comparison_table)
print('Modèle retenu : {} ; part d\'alerte retenue : top {:.0f} %.'.format(best_name,100*FRACTION_ALERTE))
EVENTS_VALIDATION={n:EVENTS_TOUS[n] for n in INCENDIES_VALIDATION}
EVENTS_ENTRAINEMENT=EVENTS_TOUS


In [ ]:
# ------------------------------------------------------------------
# NOUVEAU (code105, 24/09/2026) : diagnostic de la faible generalisation observee dans le tableau
# ci-dessus (ROC-AUC souvent proche ou en dessous de 0.5 sur plusieurs plis, meme pour le "meilleur"
# modele). Deux pistes concretes, testees ici avec ce qui est deja calcule ci-dessus (EVENTS_ENTRAINEMENT,
# comparison_table) :
# (1) desequilibre extreme des classes par evenement - un evenement avec tres peu de cellules positives
#     rend son pli de validation tres instable (quelques cellules mal classees suffisent a faire beaucoup
#     varier son ROC-AUC, sans que ca reflete une vraie qualite du modele) ;
# (2) heterogeneite de capteur - Landsat (30m, avant 2015 : 2005/2009a/2009bc) vs Sentinel-2 (10-20m,
#     apres 2015 : les 5 autres incendies) peut introduire un decalage systematique entre les deux
#     groupes d'evenements, penalisant la generalisation de l'un vers l'autre.
# Ce n'est PAS un correctif automatique : ce sont des pistes a lire et interpreter, pour decider ensuite
# si une action est utile (ex. ponderer les evenements, ou traiter Landsat/Sentinel-2 separement).
# ------------------------------------------------------------------
print('=== (1) Desequilibre des classes par evenement ===')
_desequilibre=pd.DataFrame([
    {'evenement':nom,'n_total':len(ev),'n_positif':int(ev.fire_today.sum()),
     'pct_positif':100*ev.fire_today.mean()}
    for nom,ev in EVENTS_ENTRAINEMENT.items()
]).sort_values('pct_positif')
display(_desequilibre)
print('Un evenement avec un n_positif tres petit (quelques dizaines de cellules ou moins) rend son pli de '
      'validation tres bruyant : un petit nombre de cellules mal classees y fait beaucoup varier le ROC-AUC, '
      'sans que ca reflete une vraie faiblesse du modele sur cet evenement precis.')

print()
print('=== (2) Landsat (2005/2009a/2009bc, avant Sentinel-2) vs Sentinel-2 (les 5 autres incendies) ===')
_capteur_par_evenement={nom:('landsat' if nom in ('2005','2009a','2009bc') else 'sentinel2')
                         for nom in EVENTS_ENTRAINEMENT}
_lignes_capteur=[]
for _,_row in comparison_table.iterrows():
    for _nom,_capteur in _capteur_par_evenement.items():
        _col=f'ROC_AUC_{_nom}'
        if _col in _row:
            _lignes_capteur.append({'modele':_row['modèle'],'capteur':_capteur,'ROC_AUC':_row[_col]})
_comparaison_capteur=pd.DataFrame(_lignes_capteur).groupby(['modele','capteur'])['ROC_AUC'].mean().unstack()
display(_comparaison_capteur)
print('Si la colonne "landsat" est nettement plus faible (ou plus instable) que "sentinel2" pour la plupart '
      "des modeles, ca appuie l'hypothese qu'un decalage de capteur (resolution 30m vs 10-20m, coefficients "
      "de reflectance differents) penalise la generalisation entre les deux groupes d'evenements. Si les deux "
      "colonnes sont comparables, le capteur n'explique probablement pas l'essentiel du probleme, et la piste "
      "(1) ci-dessus (ou le petit nombre total d'evenements) est plus probablement en cause.")


In [ ]:
print('Résumé : modèle {} ; ROC-AUC moyen sur les 2 incendies de validation ({}) = {:.3f}, PR-AUC moyen = {:.1f} x le hasard.'.format(
      best_name,', '.join(INCENDIES_VALIDATION),comparison_table.iloc[0]['ROC_AUC_moyen'],comparison_table.iloc[0]['PR_lift_moyen']))


In [ ]:
# ------------------------------------------------------------------
# code202 : ENTRAÎNEMENT FINAL sur les incendies d'entraînement ET de validation (le modèle et la part
# d'alerte sont déjà choisis). L'incendie de TEST n'est toujours pas utilisé.
# ------------------------------------------------------------------
THRESHOLD=None
_incendies_finaux=INCENDIES_ENTRAINEMENT+INCENDIES_VALIDATION
# code204 : `model` (entraînement + validation) et `model_sans_validation` sont déjà entraînés par executer_decoupage.
joblib.dump({'model':model,'features':FEATURES,'fraction_alerte':FRACTION_ALERTE,'model_type':best_name,
             'incendies_entrainement':INCENDIES_ENTRAINEMENT,'incendies_validation':INCENDIES_VALIDATION,
             'incendie_test':INCENDIE_TEST,'seed_decoupage':SEED_DECOUPAGE},PREP/'modele_code208.joblib')
print('Modèle final ({}) entraîné sur {} incendies : {}. Incendie de test mis de côté : {}. Classes :'.format(
      best_name,len(_incendies_finaux),', '.join(_incendies_finaux),INCENDIE_TEST),'')

# ------------------------------------------------------------------
# Importance des variables du modèle final : sur quoi s'appuie-t-il réellement ?
# Utile pour repérer s'il se raccroche à un seul facteur peu robuste (ex: uniquement le PAMF)
# plutôt qu'à un ensemble cohérent de facteurs de risque.
# ------------------------------------------------------------------
final_estimator=model.named_steps.get('rf') or model.named_steps.get('xgb') or model.named_steps.get('lgbm') or model.named_steps.get('logreg')
if hasattr(final_estimator,'feature_importances_'):
    importances=pd.Series(final_estimator.feature_importances_,index=FEATURES).sort_values(ascending=False)
    imp_title='Importance des variables ({})'.format(best_name)
elif hasattr(final_estimator,'coef_'):
    importances=pd.Series(np.abs(final_estimator.coef_[0]),index=FEATURES).sort_values(ascending=False)
    imp_title='Importance des variables (|coefficient|, régression logistique sur features standardisées)'
else:
    importances=None

if importances is not None:
    display(importances.to_frame('importance'))
    fig,ax=plt.subplots(figsize=(8,max(4,len(FEATURES)*0.3)))
    importances.sort_values().plot(kind='barh',ax=ax,color='#c0392b')
    ax.set_title("Graphique — importance des variables : sur quoi le modèle s'appuie le plus\n"+imp_title); ax.set_xlabel('importance')
    plt.tight_layout()
    fig_path=REPORT/'importance_variables_code208.png'
    plt.savefig(fig_path,dpi=200,bbox_inches='tight'); plt.show()
    print('Figure sauvegardée :',fig_path)

## Danger météo du jour (indice séparé du modèle spatial)


In [ ]:
# ------------------------------------------------------------------
# NOUVEAU (code106) : DANGER MÉTÉO DU JOUR ("quel jour ?"), séparé du modèle spatial ("où ?").
# La météo est la même pour toute la forêt un jour donné : elle ne peut pas dire OÙ le feu va prendre,
# mais elle dit SI la journée est dangereuse. On la compare aux journées d'été (MOIS_ETE) de 2017 à 2024
# présentes dans le cache code89 (2025 exclu pour rester un test aveugle) :
#   pour chaque variable, part des journées d'été moins dangereuses que le jour étudié (0 à 1) ;
#   indice = moyenne de ces parts. Classes : faible < 0.25 <= modéré < 0.50 <= élevé < 0.75 <= très élevé.
# C'est un indice simple et transparent, pas un modèle appris : avec 8 jours de feu seulement, un modèle
# "jour de feu / jour sans feu" apprendrait surtout par cœur ces 8 dates.
# ------------------------------------------------------------------
# code204 : climatologie d'été COMPLÈTE (tous les jours de juin à septembre 2017-2024) téléchargée en une fois
# sur Open-Meteo au centre de la forêt, au lieu des ~20 journées présentes dans le cache code89. Mêmes définitions
# que la météo du jour : maximum horaire de température, de vent et de VPD, minimum horaire d'humidité.
CLIMATO_CACHE=PREP/'climatologie_ete_2017_2024.parquet'
if CLIMATO_CACHE.exists():
    CLIMATO_ETE=pd.read_parquet(CLIMATO_CACHE)
else:
    _r=requete_avec_reessais('https://archive-api.open-meteo.com/v1/archive',params={
        'latitude':CENTRE_WGS84.y,'longitude':CENTRE_WGS84.x,'start_date':'2017-06-01','end_date':'2024-09-30',
        'hourly':'temperature_2m,relative_humidity_2m,wind_speed_10m','timezone':'Africa/Casablanca'},timeout=180)
    _r.raise_for_status(); _h=_r.json()['hourly']
    _df=pd.DataFrame({'time':pd.to_datetime(_h['time']),'t':_h['temperature_2m'],'hr':_h['relative_humidity_2m'],'v':_h['wind_speed_10m']}).astype({'t':float,'hr':float,'v':float})
    _df['vpd']=.6108*np.exp(17.27*_df.t/(_df.t+237.3))*(1-_df.hr/100)
    _df['date']=_df.time.dt.normalize()
    CLIMATO_ETE=_df.groupby('date').agg(temperature_2m_max=('t','max'),relative_humidity_2m_min=('hr','min'),
                                         wind_speed_10m_max=('v','max'),vpd_max=('vpd','max')).reset_index()
    CLIMATO_ETE=CLIMATO_ETE[CLIMATO_ETE.date.dt.month.isin(MOIS_ETE)].reset_index(drop=True)
    CLIMATO_ETE.to_parquet(CLIMATO_CACHE,index=False)
print("Climatologie d'été : {} journée(s) Open-Meteo (mois {}, 2017-2024).".format(len(CLIMATO_ETE),MOIS_ETE))
if len(CLIMATO_ETE)<15:
    print("  ATTENTION : moins de 15 journées d'été en cache - l'indice de danger sera peu précis.")

def indice_danger_jour(frame):
    """Indice de danger météo (0-1) et sa classe pour une journée (frame = cellules de cette journée)."""
    parts={}
    for var,sens in DANGER_VARS.items():
        if var not in frame or var not in CLIMATO_ETE or frame[var].notna().sum()==0: continue
        valeur=float(frame[var].median()); ref=CLIMATO_ETE[var].dropna()
        if ref.empty: continue
        parts[var]=float((ref<=valeur).mean()) if sens>0 else float((ref>=valeur).mean())
    if not parts: return np.nan,'inconnu',parts
    indice=float(np.mean(list(parts.values())))
    classe='faible' if indice<0.25 else 'modéré' if indice<0.50 else 'élevé' if indice<0.75 else 'très élevé'
    return indice,classe,parts

# Contrôle : les 8 jours de feu devraient en majorité sortir "élevé" ou "très élevé", et les journées
# négatives d'été plutôt moins. Si ce n'est pas le cas, l'indice ne discrimine pas et il faut revoir DANGER_VARS.
_controle=[]
for _groupe,_frame in samples.groupby('event_group'):
    _i,_c,_ = indice_danger_jour(_frame)
    _d=pd.Timestamp(_frame.date_modele.iloc[0])
    _controle.append({'journee':_groupe,'date':_d.date(),'jour_de_feu':_groupe.startswith('incendie_'),
                      'ete':_d.month in MOIS_ETE,'indice_danger':_i,'classe':_c})
_controle=pd.DataFrame(_controle).sort_values('indice_danger',ascending=False)
display(_controle)
print("Indice moyen : jours de feu = {:.2f} | journées sans feu en été = {:.2f} | journées sans feu hors été = {:.2f}".format(
      _controle[_controle.jour_de_feu].indice_danger.mean(),
      _controle[~_controle.jour_de_feu&_controle.ete].indice_danger.mean(),
      _controle[~_controle.jour_de_feu&~_controle.ete].indice_danger.mean()))


## Test sur l'incendie tiré au sort (et cartes des 2 incendies de validation)

L'incendie de TEST n'a servi ni à l'entraînement ni au choix du modèle : son résultat est le seul vraiment indépendant. Les 2 incendies de VALIDATION ont servi à choisir le modèle et ont été ajoutés à l'entraînement final, leurs cartes sont donc données à titre indicatif.


In [ ]:
# ------------------------------------------------------------------
# Confirmation informative (ne bloque plus rien) : la chronologie reçue le 23/09/2026 confirme que
# Derdara 2021 est un vrai incendie (intégré à l'entraînement, cellules précédentes) et que les 2 dates
# 2025 qui semblaient être 2 candidates pour un seul feu incertain sont en réalité 2 incendies distincts
# (voir EVENEMENTS_2025, cellule 2). On vérifie ici les détections FIRMS locales pour chaque fenêtre à
# titre indicatif seulement - aucune de ces vérifications ne modifie l'entraînement ni le calibrage du
# seuil, déjà figés dans les cellules précédentes, donc le test aveugle reste un vrai test aveugle.
# ------------------------------------------------------------------
print("Vérification FIRMS locale (indicative) des incendies désormais confirmés par chronologie externe :")
_fenetres_a_verifier=[('Derdara/Sougna 2021',DERDARA_2021_DATE-pd.Timedelta(days=1),DERDARA_2021_DATE+pd.Timedelta(days=5)),
                      ('Tanaqoub aout 2019',FIRE_2019A_DATE-pd.Timedelta(days=1),FIRE_2019A_DATE+pd.Timedelta(days=5)),
                      ('Tanaqoub septembre 2019',FIRE_2019B_DATE-pd.Timedelta(days=1),FIRE_2019B_DATE+pd.Timedelta(days=5)),
                      ('Tanaqoub-Bouhachem 2005',FIRE_2005_DATE-pd.Timedelta(days=1),FIRE_2005_DATE+pd.Timedelta(days=5)),
                      ('Derdara-Jbel Sougna 2009',FIRE_2009A_DATE-pd.Timedelta(days=1),FIRE_2009A_DATE+pd.Timedelta(days=5)),
                      ('Derdara-Bouhachem/Mont Bouhachem 2009 (fusion)',FIRE_2009BC_DATE-pd.Timedelta(days=1),FIRE_2009BC_DATE+pd.Timedelta(days=5))]
_fenetres_a_verifier+=[(e['nom']+' 2025',e['test']-pd.Timedelta(days=3),e['test']+pd.Timedelta(days=3)) for e in EVENEMENTS_2025]
for nom,debut,fin in _fenetres_a_verifier:
    dedans=firms_in_forest[(firms_in_forest.date>=debut)&(firms_in_forest.date<=fin)]
    print('  {:<30} {} -> {} : {} détection(s) FIRMS dans la forêt.'.format(nom,debut.date(),fin.date(),len(dedans)),
          sorted(dedans.date.dt.date.unique()) if len(dedans) else '(aucune - petit feu souvent manqué par FIRMS, cohérent avec la chronologie)')


In [ ]:
def cadre_complet(nom):
    """Toutes les cellules de la forêt pour cet incendie (y compris celles sans dNBR valide), avec le label
    et la valeur de dNBR, pour pouvoir dessiner les cartes sur toute la forêt."""
    if nom in CADRES_2025:
        f=CADRES_2025[nom].copy()
    else:
        f=samples[samples.event_group=='incendie_'+nom].reset_index(drop=True).copy()
        valeurs=globals().get('dnbr_values_'+EVENEMENTS_DNBR[nom]['var'])
        f['dnbr']=valeurs if valeurs is not None and len(valeurs)==len(f) else np.nan
    if 'annees_depuis_dernier_feu' not in f: f=ajouter_historique(f,f.date_modele.iloc[0])   # code205
    return gpd.GeoDataFrame(f,geometry='geometry',crs=CRS)

def evaluer_incendie(nom,role,modele=None):
    """Prédiction, chiffres et 4 cartes pour un incendie de TEST ou de VALIDATION, sur toute la forêt."""
    f=cadre_complet(nom); date=pd.Timestamp(f.date_modele.iloc[0]).date()
    print('='*100); print('{} -- incendie {} ({})'.format(role,nom,date)); print('='*100)
    print('Image satellite utilisée : {} (âge {} jours).'.format(
          f['satellite_date'].iloc[0] if 'satellite_date' in f else 'inconnue',f['image_age_days'].iloc[0] if 'image_age_days' in f else '?'))
    modele=modele if modele is not None else model
    f['probabilite_feu']=predire(modele,f)   # code206 : règle du combustible appliquée
    seuil=float(np.quantile(f.probabilite_feu,1-FRACTION_ALERTE))
    f['prediction_feu']=(f.probabilite_feu>=seuil).astype('uint8')
    f['classe_risque']=classes_risque_non_supervisees(f.probabilite_feu,5)
    danger,classe_danger,_=indice_danger_jour(f)
    print('Danger météo du jour : {:.2f} ({}) | alerte : top {:.0f} % des cellules.'.format(danger,classe_danger,100*FRACTION_ALERTE))
    if role=='TEST':
        chemin=PREP/f'prediction_test_{nom}_code208.csv'; f.drop(columns='geometry').drop(columns=['fire_today','dnbr'],errors='ignore').to_csv(chemin,index=False)
        print('Prédiction verrouillée avant comparaison avec la vérité (SHA-256) :',hashlib.sha256(chemin.read_bytes()).hexdigest())

    valide=np.isfinite(f['dnbr'].to_numpy(dtype=float)) if f['dnbr'].notna().any() else np.ones(len(f),bool)
    y=f.fire_today.to_numpy()[valide].astype(int); p=f.probabilite_feu.to_numpy()[valide]; a=f.prediction_feu.to_numpy()[valide]
    hasard=y.mean()
    print('Vérité : {} cellules brûlées (~{:.0f} ha) sur {} cellules vérifiables (sur {}).'.format(int(y.sum()),y.sum()*CELL_HA,int(valide.sum()),len(f)))
    print('ROC-AUC = {:.3f} (0,5 = hasard) | PR-AUC = {:.3f} pour un hasard à {:.3f}, soit {:.1f} x le hasard.'.format(
          roc_auc_score(y,p),average_precision_score(y,p),hasard,average_precision_score(y,p)/hasard))
    vp=int(((a==1)&(y==1)).sum()); fn=int(((a==0)&(y==1)).sum()); fp=int(((a==1)&(y==0)).sum()); vn=int(((a==0)&(y==0)).sum())
    display(pd.DataFrame([[vp,fn],[fp,vn]],index=['Réellement brûlé','Réellement non brûlé'],columns=['Prédit brûlé (alerte)','Prédit non brûlé']))
    print('précision = {:.3f} (hasard {:.3f}) | rappel = {:.3f}'.format(vp/max(vp+fp,1),hasard,vp/max(vp+fn,1)))

    f['verite_valide']=valide; brule=(f.fire_today==1).to_numpy(); alerte=(f.prediction_feu==1).to_numpy()
    f['categorie']=np.select([~valide,alerte&brule,~alerte&brule,alerte&~brule],
        ['Pas de vérité (nuages / hors image)','Vrai positif (brûlé, bien repéré)','Faux négatif (brûlé, raté)','Faux positif (fausse alerte)'],
        default='Vrai négatif (calme, bien classé)')
    couleurs={'Vrai positif (brûlé, bien repéré)':'#1a9850','Faux négatif (brûlé, raté)':'#d73027','Faux positif (fausse alerte)':'#fee08b',
              'Vrai négatif (calme, bien classé)':'#e0e0e0','Pas de vérité (nuages / hors image)':'#c6dbef'}
    etiquette='{} -- {} ({})'.format(role,nom,date)
    # code203 : couche de vérification pour QGIS, en Merchich / Nord Maroc (EPSG:26191)
    _gpkg=REPORT/f'verification_qgis_{nom}_code208.gpkg'
    if _gpkg.exists(): _gpkg.unlink()
    f[['id_cell','partie','probabilite_feu','prediction_feu','fire_today','dnbr','categorie','geometry']].to_crs(CRS).to_file(_gpkg,layer='cellules',driver='GPKG')
    print('Couche QGIS (EPSG:26191) écrite :',_gpkg,'- à superposer à l\'image Sentinel-2 / Landsat et aux points FIRMS.')

    fig,axes=plt.subplots(1,2,figsize=(18,9))
    f.plot(column='probabilite_feu',cmap='YlOrRd',legend=True,ax=axes[0]); f.plot(column='classe_risque',cmap='RdYlGn_r',vmin=1,vmax=5,legend=True,ax=axes[1])
    axes[0].set_title('Probabilité de feu par cellule'); axes[1].set_title('5 classes de risque (1 = faible, 5 = fort)')
    fig.suptitle("Carte 1/4 — PRÉDICTION : risque calculé par le modèle avant l'incendie\n"+etiquette,fontsize=13,y=1.03)
    for ax in axes: ax.set_axis_off()
    plt.tight_layout(); plt.savefig(REPORT/f'carte1_prediction_{nom}_code208.png',dpi=200,bbox_inches='tight'); plt.show()

    fig,axes=plt.subplots(1,2,figsize=(20,9))
    for ax in axes:
        for cat,coul in couleurs.items():
            s=f[f.categorie==cat]
            if len(s): s.plot(ax=ax,color=coul,label=cat)
        ax.set_axis_off()
    axes[0].legend(loc='lower left',fontsize=9); axes[0].set_title('Toute la forêt')
    if brule.any():
        xmin,ymin,xmax,ymax=f[brule].total_bounds; axes[1].set_xlim(xmin-1500,xmax+1500); axes[1].set_ylim(ymin-1500,ymax+1500)
    axes[1].set_title('Zoom sur la zone brûlée')
    fig.suptitle("Carte 2/4 — ÉVALUATION : où le modèle a eu raison ou tort (alerte = top {:.0f} %)\n".format(100*FRACTION_ALERTE)+etiquette,fontsize=13,y=1.03)
    plt.tight_layout(); plt.savefig(REPORT/f'carte2_evaluation_{nom}_code208.png',dpi=200,bbox_inches='tight'); plt.show()

    fig,axes=plt.subplots(1,2,figsize=(18,9))
    f.plot(column='probabilite_feu',cmap='YlOrRd',legend=True,ax=axes[0])
    f.plot(column='dnbr',cmap='RdYlGn_r',legend=True,ax=axes[1],missing_kwds={'color':'#c6dbef'})
    axes[0].set_title('Probabilité prédite'); axes[1].set_title('dNBR mesuré (rouge = forte brûlure, bleu clair = pas de valeur)')
    for ax in axes: ax.set_axis_off()
    fig.suptitle('Carte 3/4 — COMPARAISON DES VALEURS : risque prédit et brûlure mesurée\n'+etiquette,fontsize=13,y=1.03)
    plt.tight_layout(); plt.savefig(REPORT/f'carte3_valeurs_{nom}_code208.png',dpi=200,bbox_inches='tight'); plt.show()

    fig,axes=plt.subplots(1,2,figsize=(18,9))
    for ax,col,titre in [(axes[0],'fire_today','Zone réellement brûlée (satellite)'),(axes[1],'prediction_feu','Zone mise en alerte (modèle)')]:
        f[f[col]==0].plot(ax=ax,color='#e0e0e0'); s=f[f[col]==1]
        if len(s): s.plot(ax=ax,color='#d73027')
        if col=='fire_today' and (~valide).any(): f[~valide].plot(ax=ax,color='#c6dbef')
        ax.set_title(titre); ax.set_axis_off()
    fig.suptitle('Carte 4/4 — COMPARAISON SIMPLE : réalité et prédiction (rouge = brûlé / alerté)\n'+etiquette,fontsize=13,y=1.03)
    plt.tight_layout(); plt.savefig(REPORT/f'carte4_realite_vs_predit_{nom}_code208.png',dpi=200,bbox_inches='tight'); plt.show()
    return {'incendie':nom,'role':role,'ROC_AUC':roc_auc_score(y,p),'PR_lift':average_precision_score(y,p)/hasard,'VP':vp,'FN':fn,'FP':fp,'VN':vn}

bilan=[evaluer_incendie(INCENDIE_TEST,'TEST')]+[evaluer_incendie(n,'VALIDATION (modèle entraîné sans cet incendie)',model_sans_validation) for n in INCENDIES_VALIDATION]
print('Bilan du découpage présenté (chaque incendie est évalué par un modèle qui ne l\'a jamais vu) :')
display(pd.DataFrame(bilan))


## Carte de susceptibilité annuelle avec historique des feux

Apprise année par année (2017-2024) sur les points FIRMS, avec les années depuis le dernier feu comme variable. Validée en excluant une année entière à la fois, puis testée sur 2025.


In [ ]:
# ------------------------------------------------------------------
# code205 : CARTE DE SUSCEPTIBILITÉ ANNUELLE (FIRMS) AVEC HISTORIQUE DES FEUX
# Tableau "cellule x année" de 2017 à 2024 :
#   - label   : la cellule a eu au moins un point FIRMS pendant l'année ;
#   - entrées : facteurs fixes + années depuis le dernier feu au 1er janvier de l'année (historique antérieur).
# Validation "UNE ANNÉE EXCLUE" : chaque année est retirée à tour de rôle. Les grands feux étant presque tous
# dans des années différentes (2019, 2020, 2021, 2022), un même feu n'est jamais à la fois appris et évalué.
# Test indépendant : 2025 (historique au 1er août 2025, avant les 2 feux), points FIRMS et zones brûlées dNBR.
# ------------------------------------------------------------------
from sklearn.model_selection import LeaveOneGroupOut
ANNEES_APPRENTISSAGE=list(range(2017,2025))
base=structural.copy()
base['exposition_nord']=np.cos(np.radians(base['exposition_deg']))
base['exposition_est']=np.sin(np.radians(base['exposition_deg']))
FEATURES_SUSCEPTIBILITE=[c for c in ['altitude_m','pente_pct','exposition_nord','exposition_est','population',
    'pression_humaine_01','distance_route_m','distance_piste_m','distance_batiment_m','distance_douar_m',
    'distance_agriculture_m','occupation_sol_2021','combustibilite_01'] if c in base and base[c].notna().any()]
FEATURES_SUSCEPTIBILITE+=['annees_depuis_dernier_feu','brule_moins_5ans']
_a_idx=FEATURES_SUSCEPTIBILITE.index('annees_depuis_dernier_feu')

def cellules_avec_firms(debut,fin):
    pts=firms[(firms.date>=pd.Timestamp(debut))&(firms.date<=pd.Timestamp(fin))]
    pts=pts[pts.geometry.within(foret_unique)]
    j=gpd.sjoin(pts[['geometry']],base[['id_cell','geometry']],predicate='within',how='inner')
    return base.id_cell.isin(j.id_cell).astype(int).to_numpy(),len(pts)

_panneau=[]
for _an in ANNEES_APPRENTISSAGE:
    _f=ajouter_historique(base,f'{_an}-01-01')
    _f['feu_firms'],_n=cellules_avec_firms(f'{_an}-01-01',f'{_an}-12-31'); _f['annee']=_an
    _panneau.append(_f); print('  {} : {} points FIRMS, {} cellules avec feu'.format(_an,_n,int(_f.feu_firms.sum())))
PANNEAU=pd.concat(_panneau,ignore_index=True)
X=PANNEAU[FEATURES_SUSCEPTIBILITE]; y=PANNEAU.feu_firms; groupes=PANNEAU.annee

_cands={'Régression logistique':make_model_logreg,'RandomForest (régularisé)':make_model}
_lignes=[]
for _nom,_fab in _cands.items():
    _p=np.zeros(len(PANNEAU)); _par_an={}
    for _tr,_te in LeaveOneGroupOut().split(X,y,groupes):
        _m=_fab(); _m.fit(X.iloc[_tr],y.iloc[_tr]); _p[_te]=regle_combustible(_m.predict_proba(X.iloc[_te])[:,1],X.iloc[_te]['annees_depuis_dernier_feu'])
        _an=int(groupes.iloc[_te[0]]); _yt=y.iloc[_te]
        if 0<_yt.sum()<len(_yt): _par_an[_an]=roc_auc_score(_yt,_p[_te])
    _lignes.append({'modèle':_nom,'ROC_AUC_global':roc_auc_score(y,_p),'PR_lift':average_precision_score(y,_p)/y.mean(),
                    'ROC_AUC_moyen_par_annee':np.mean(list(_par_an.values())),
                    **{f'ROC_{a}':v for a,v in _par_an.items()}})
_table_susc=pd.DataFrame(_lignes).sort_values('ROC_AUC_moyen_par_annee',ascending=False)
print('Validation "une année exclue" (2017-2024) :'); display(_table_susc)
NOM_MODELE_SUSC=_table_susc.iloc[0]['modèle']
modele_susc=_cands[NOM_MODELE_SUSC](); modele_susc.fit(X,y)

# --- Test indépendant 2025 ---
susc=ajouter_historique(base,'2025-08-01')
susc['susceptibilite_sans_regle']=modele_susc.predict_proba(susc[FEATURES_SUSCEPTIBILITE])[:,1]
susc['susceptibilite']=regle_combustible(susc['susceptibilite_sans_regle'],susc['annees_depuis_dernier_feu'])
susc['classe_susceptibilite']=classes_risque_non_supervisees(susc.susceptibilite,5)
susc['feu_firms_2025'],_n25=cellules_avec_firms('2025-01-01','2025-12-31')
_lignes_test=[]
def _ajouter_test(libelle,y_,p_):
    if 0<y_.sum()<len(y_):
        m=matrice_alerte_relative(p_,y_,0.20)
        _lignes_test.append({'vérité 2025':libelle,'cellules_feu':int(y_.sum()),'ROC_AUC':roc_auc_score(y_,p_),
            'PR_lift':average_precision_score(y_,p_)/y_.mean(),'rappel_top20':m['rappel'],'precision_top20':m['precision'],'precision_hasard':y_.mean()})
_ajouter_test('points FIRMS 2025 ({} points) - sans règle'.format(_n25),susc.feu_firms_2025.to_numpy(),susc.susceptibilite_sans_regle.to_numpy())
_ajouter_test('points FIRMS 2025 ({} points) - avec règle'.format(_n25),susc.feu_firms_2025.to_numpy(),susc.susceptibilite.to_numpy())
for _nom,_cadre in CADRES_2025.items():
    _t=_cadre[['id_cell','fire_today','dnbr']].merge(susc[['id_cell','susceptibilite']],on='id_cell')
    _t=_t[np.isfinite(_t.dnbr.astype(float))]
    _ajouter_test('zone brûlée dNBR '+_nom,_t.fire_today.to_numpy().astype(int),_t.susceptibilite.to_numpy())
print('TEST INDÉPENDANT 2025 ({}, appris sur 2017-2024 avec historique des feux) :'.format(NOM_MODELE_SUSC))
display(pd.DataFrame(_lignes_test))

_est=modele_susc.steps[-1][1]
_imp=pd.Series(np.abs(_est.coef_[0]) if hasattr(_est,'coef_') else _est.feature_importances_,index=FEATURES_SUSCEPTIBILITE)
display(_imp.sort_values(ascending=False).to_frame('importance'))
if hasattr(_est,'coef_'):
    print("Signe de l'effet de annees_depuis_dernier_feu : {} (positif = une zone brûlée depuis longtemps est plus susceptible).".format(
          'positif' if _est.coef_[0][FEATURES_SUSCEPTIBILITE.index('annees_depuis_dernier_feu')]>0 else 'négatif'))

_pts_app=firms[(firms.date>=TRAIN_START)&(firms.date<='2024-12-31')]; _pts_app=_pts_app[_pts_app.geometry.within(foret_unique)]
_pts_25=firms[firms.date.dt.year==2025]; _pts_25=_pts_25[_pts_25.geometry.within(foret_unique)]
fig,axes=plt.subplots(1,2,figsize=(18,9))
susc.plot(column='susceptibilite',cmap='YlOrRd',legend=True,ax=axes[0])
if len(_pts_app): _pts_app.plot(ax=axes[0],color='black',markersize=4,label='FIRMS 2017-2024 (apprentissage)')
if len(_pts_25): _pts_25.plot(ax=axes[0],color='blue',markersize=8,label='FIRMS 2025 (test)')
axes[0].legend(loc='lower left',fontsize=9); axes[0].set_title('Susceptibilité au 1er août 2025 et points FIRMS')
susc.plot(column='annees_depuis_dernier_feu',cmap='viridis',legend=True,ax=axes[1])
axes[1].set_title('Années depuis le dernier feu connu (20 = aucun feu connu)')
for _ax in axes: _ax.set_axis_off()
fig.suptitle('Carte S1 — SUSCEPTIBILITÉ 2025 avec historique des feux (apprise sur FIRMS 2017-2024)\n'
             'Points noirs : feux appris. Points bleus : feux de 2025, jamais vus.',fontsize=13,y=1.03)
plt.tight_layout(); plt.savefig(REPORT/'carteS1_susceptibilite_code208.png',dpi=200,bbox_inches='tight'); plt.show()

for _nom,_cadre in CADRES_2025.items():
    _t=gpd.GeoDataFrame(_cadre[['id_cell','fire_today','dnbr','geometry']].merge(susc[['id_cell','susceptibilite']],on='id_cell'),geometry='geometry',crs=CRS)
    _valide=np.isfinite(_t.dnbr.astype(float)); _t['alerte']=(_t.susceptibilite>=np.quantile(_t.susceptibilite,0.80)).astype(int)
    fig,axes=plt.subplots(1,2,figsize=(18,9))
    _t[_t.fire_today==0].plot(ax=axes[0],color='#e0e0e0')
    if (_t.fire_today==1).any(): _t[_t.fire_today==1].plot(ax=axes[0],color='#d73027')
    if (~_valide).any(): _t[~_valide].plot(ax=axes[0],color='#c6dbef')
    _t[_t.alerte==0].plot(ax=axes[1],color='#e0e0e0'); _t[_t.alerte==1].plot(ax=axes[1],color='#d73027')
    axes[0].set_title("Zone réellement brûlée (rouge) ; bleu clair = pas d'image"); axes[1].set_title('20 % de la forêt la plus susceptible (rouge)')
    for _ax in axes: _ax.set_axis_off()
    fig.suptitle('Carte S2 — TEST 2025 de la susceptibilité : réalité et zones les plus susceptibles\n'+_nom,fontsize=13,y=1.03)
    plt.tight_layout(); plt.savefig(REPORT/f'carteS2_test_susceptibilite_{_nom}_code208.png',dpi=200,bbox_inches='tight'); plt.show()

_gpkg=REPORT/'susceptibilite_firms_code208.gpkg'
if _gpkg.exists(): _gpkg.unlink()
susc[['id_cell','partie','susceptibilite','classe_susceptibilite','annees_depuis_dernier_feu','feu_firms_2025','geometry']].to_file(_gpkg,layer='susceptibilite',driver='GPKG')
print('Couche QGIS de susceptibilité (EPSG:26191) :',_gpkg)


## Carte du risque pour aujourd'hui

Cette cellule réutilise le cache du jour créé par `code89`. Si ce cache n'existe pas, elle le crée (météo temps réel + dernière scène Sentinel-2 disponible) sans jamais toucher aux caches de `code89`/`code90` existants.

In [ ]:
# NOUVEAU (code106) : carte du jour = risque spatial ("où ?") + danger météo du jour ("quel jour ?") +
# alerte relative (top FRACTION_ALERTE des cellules). L'alerte du jour n'a de sens opérationnel que si le
# danger météo est "élevé" ou "très élevé" ; par temps calme, la carte montre seulement les zones les plus
# sensibles de la forêt.
TODAY=pd.Timestamp.now(tz='Africa/Casablanca').tz_localize(None).normalize()
today=load_code89_day(TODAY,include_target=False,allow_missing_download=True,operational=True)
today=ajouter_historique(today,TODAY)   # code205
today['probabilite_feu']=predire(model,today)   # code206 : règle du combustible
today['classe_risque']=classes_risque_non_supervisees(today.probabilite_feu,5)
_seuil_jour=float(np.quantile(today.probabilite_feu,1-FRACTION_ALERTE))
today['alerte_feu']=(today.probabilite_feu>=_seuil_jour).astype('uint8')
danger_today,classe_danger_today,_=indice_danger_jour(today)
today['danger_meteo_jour']=danger_today; today['classe_danger_jour']=classe_danger_today
print('Danger météo du {} : {:.2f} ({}) | alerte : top {:.0f} % des cellules (probabilité >= {:.4f}).'.format(
      TODAY.date(),danger_today,classe_danger_today,100*FRACTION_ALERTE,_seuil_jour))
gpkg=PREP/f'risque_feu_{TODAY.date()}_code208.gpkg'
if gpkg.exists(): gpkg.unlink()
today.to_file(gpkg,layer='risque_du_jour',driver='GPKG')
today.drop(columns='geometry').to_csv(PREP/f'risque_feu_{TODAY.date()}_code208.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(18,9))
today.plot(column='probabilite_feu',cmap='YlOrRd',legend=True,ax=axes[0])
today.plot(column='classe_risque',cmap='RdYlGn_r',vmin=1,vmax=5,legend=True,ax=axes[1])
axes[0].set_title(f'Risque spatial — {TODAY.date()}\nDanger météo du jour : {danger_today:.2f} ({classe_danger_today})')
axes[1].set_title('Classes KMeans non supervisées — 1 à 5')
for ax in axes:ax.set_axis_off()
fig.suptitle("Carte du jour — risque de feu prévu aujourd'hui ({})\nGauche : probabilité par cellule. Droite : 5 classes de risque. Le danger météo indique si la journée elle-même est dangereuse.".format(TODAY.date()),fontsize=13,y=1.03)
plt.tight_layout()
plt.savefig(REPORT/f'risque_feu_{TODAY.date()}_code208.png',dpi=220,bbox_inches='tight');plt.show();print(gpkg)


## Bilan final du modèle


In [ ]:
# ------------------------------------------------------------------
# code208 : BILAN FINAL (résumé à reporter dans le rapport)
# ------------------------------------------------------------------
print('='*90); print('BILAN FINAL — code207'); print('='*90)
print('1) MODÈLE SPATIAL ("où ?") — 12 découpages, chaque incendie évalué par un modèle qui ne l\'a jamais vu')
print('   ROC-AUC test moyen : {:.3f} sans la règle du combustible, {:.3f} avec (0,5 = hasard).'.format(
      TABLE_REPETITIONS.ROC_AUC_test_sans_regle.mean(),TABLE_REPETITIONS.ROC_AUC_test.mean()))
print('   PR-AUC test moyen : {:.1f} x le hasard.'.format(TABLE_REPETITIONS.PR_lift_test.mean()))
display(TABLE_REPETITIONS.groupby('test')[['ROC_AUC_test','PR_lift_test','rappel_test','precision_test','precision_hasard']].mean().round(3))
print('2) DANGER MÉTÉO DU JOUR ("quand ?") — comparé à {} jours d\'été 2017-2024'.format(len(CLIMATO_ETE)))
_jf=_controle[_controle.jour_de_feu]
print('   Jours de feu classés "élevé" ou "très élevé" : {} sur {} ; indice moyen des jours de feu = {:.2f} (0,50 pour un jour d\'été quelconque).'.format(
      int(_jf.classe.isin(['élevé','très élevé']).sum()),len(_jf),_jf.indice_danger.mean()))
display(_jf[['journee','date','indice_danger','classe']].reset_index(drop=True))
